In [1]:
import pandas as pd

Esse notebook utiliza o datatset aria midi unique como ponto de partida do projeto. O arquivo de metadata é analisado para recolher informações sobre os midis disponiveis para uso. Após tratativas, o datatset é separado em dados de treino, teste e validação. Cada dataset possui o nome dos arquivos midis que serão utilizados posteriormente nas etapas seguintes.

In [2]:
pip install scikit-learn

In [3]:
with open("/content/MIDI-LSTM-and-transformer-decoder/metadata.json", "r") as f:
    df = pd.read_json(f)

In [4]:
df_metadata=pd.DataFrame()

for col_name, col_data in df.items():
    raw_values = pd.DataFrame([col_data.values[0]])
    raw_values["scores"] = [col_data.values[1]]
    col_name = str(col_name)
    name = col_name.zfill(6) if len(col_name) <6 else col_name
    raw_values["audio_name"] = name
    df_metadata = pd.concat([raw_values, df_metadata], ignore_index=True)
print(df_metadata)

          composer   opus      genre performer  music_period  \
0            liszt  392.0  classical   hamelin      romantic   
1           ligeti    NaN  classical       NaN  contemporary   
2      saint-saens   56.0  classical       NaN      romantic   
3       moszkowski   68.0  classical       NaN      romantic   
4         respighi   44.0  classical       NaN     classical   
...            ...    ...        ...       ...           ...   
32517     schubert  498.0  classical       NaN      romantic   
32518       franck   10.0  classical       NaN      romantic   
32519      purcell    5.0  classical       NaN       baroque   
32520     maykapar   15.0  classical       NaN           NaN   
32521        weber   77.0  classical       NaN     classical   

                          scores audio_name  piece_number     form  \
0      {'0': 0.9963000000000001}     207047           NaN      NaN   
1      {'0': 0.9904000000000001}     207046           5.0    etude   
2      {'0': 0.992300

In [5]:
df_metadata.columns

Index(['composer', 'opus', 'genre', 'performer', 'music_period', 'scores',
       'audio_name', 'piece_number', 'form', 'key_signature', 'difficulty'],
      dtype='object')

In [6]:
df_metadata.drop(columns=['opus','performer','key_signature','piece_number','form','difficulty'], inplace=True)

In [7]:
df_metadata.columns

Index(['composer', 'genre', 'music_period', 'scores', 'audio_name'], dtype='object')

In [8]:
cleaned_df_metadata = df_metadata.dropna(subset=['genre', 'music_period'])

print(cleaned_df_metadata)

          composer      genre  music_period                     scores  \
0            liszt  classical      romantic  {'0': 0.9963000000000001}   
1           ligeti  classical  contemporary  {'0': 0.9904000000000001}   
2      saint-saens  classical      romantic  {'0': 0.9923000000000001}   
3       moszkowski  classical      romantic  {'0': 0.9902000000000001}   
4         respighi  classical     classical  {'0': 0.9981000000000001}   
...            ...        ...           ...                        ...   
32516      arensky  classical      romantic  {'0': 0.9618000000000001}   
32517     schubert  classical      romantic               {'0': 0.991}   
32518       franck  classical      romantic              {'0': 0.9846}   
32519      purcell  classical       baroque              {'0': 0.9957}   
32521        weber  classical     classical  {'0': 0.9510000000000001}   

      audio_name  
0         207047  
1         207046  
2         207042  
3         207040  
4         207039

In [9]:
print(cleaned_df_metadata.groupby(["genre", "music_period"])["genre"].count())


genre       music_period 
atonal      contemporary        8
            modern             17
blues       contemporary        1
            modern              3
classical   baroque          2291
            classical        7818
            contemporary     1990
            impressionist     854
            modern           1034
            romantic         9817
folk        classical           1
            contemporary        4
            modern              5
            romantic            2
jazz        baroque             1
            classical           2
            contemporary        5
            modern             60
            romantic            1
ragtime     classical           2
            contemporary        4
            modern              2
rock        modern              1
soundtrack  contemporary        8
Name: genre, dtype: int64


In [10]:
print(cleaned_df_metadata.groupby(["genre", "composer"])["genre"].count())


genre       composer  
atonal      bacevicius    1
            berg          3
            ligeti        2
            nancarrow     1
            noland        1
                         ..
ragtime     zerkovitz     1
rock        norton        1
soundtrack  bemani        1
            o'halloran    5
            ohalloran     2
Name: genre, Length: 1888, dtype: int64


In [11]:
most_famous_genre = cleaned_df_metadata[cleaned_df_metadata['genre'] == "classical"].copy()

print(most_famous_genre)

          composer      genre  music_period                     scores  \
0            liszt  classical      romantic  {'0': 0.9963000000000001}   
1           ligeti  classical  contemporary  {'0': 0.9904000000000001}   
2      saint-saens  classical      romantic  {'0': 0.9923000000000001}   
3       moszkowski  classical      romantic  {'0': 0.9902000000000001}   
4         respighi  classical     classical  {'0': 0.9981000000000001}   
...            ...        ...           ...                        ...   
32516      arensky  classical      romantic  {'0': 0.9618000000000001}   
32517     schubert  classical      romantic               {'0': 0.991}   
32518       franck  classical      romantic              {'0': 0.9846}   
32519      purcell  classical       baroque              {'0': 0.9957}   
32521        weber  classical     classical  {'0': 0.9510000000000001}   

      audio_name  
0         207047  
1         207046  
2         207042  
3         207040  
4         207039

In [16]:
from sklearn.model_selection import train_test_split

#Separação dos dados de treinamento, teste e validação
#treinamento = 60%, teste = 25% e validação = 15%
X_train_val, X_test = train_test_split(most_famous_genre, test_size=0.25, random_state=42)

X_train, X_val = train_test_split(X_train_val, test_size=0.15, random_state=42)

In [17]:
X_train


,composer,genre,music_period,scores,audio_name
22110,belkin,classical,contemporary,{'0': 0.9823000000000001},066383
16922,bach,classical,baroque,{'0': 0.9999},098826
31258,orona,classical,contemporary,{'0': 0.9577},007902
30601,scarlatti,classical,baroque,{'0': 0.9947},011944
28156,smith,classical,romantic,{'0': 0.9846},027268
...,...,...,...,...,...
9569,schumann,classical,romantic,{'0': 0.9994000000000001},145538
29460,berlioz,classical,romantic,{'0': 0.9761000000000001},019074
2348,bartok,classical,modern,{'0': 0.9672000000000001},191860
9435,lyadov,classical,romantic,{'0': 0.8183},146460


In [18]:
X_test

,composer,genre,music_period,scores,audio_name
17493,mozart,classical,classical,{'0': 0.9911000000000001},095174
31339,mozart,classical,classical,{'0': 0.9994000000000001},007366
3691,vladigerov,classical,classical,{'0': 0.9886},183335
20678,lemoine,classical,classical,{'0': 0.9994000000000001},075470
2652,prudent,classical,romantic,{'0': 0.9906},189873
...,...,...,...,...,...
5631,mozart,classical,classical,{'0': 0.9874},171427
10790,scarlatti,classical,baroque,{'0': 0.9908},137786
2761,schubert,classical,classical,{'0': 0.929},189232
21383,liapounov,classical,romantic,{'0': 0.9863000000000001},071103


In [19]:
X_val

,composer,genre,music_period,scores,audio_name
467,diabelli,classical,classical,{'0': 0.9837},204085
3171,czerny,classical,romantic,{'0': 0.9845},186635
15089,dussek,classical,classical,{'0': 0.9685},110689
23219,alkan,classical,romantic,{'0': 0.9923000000000001},059307
31942,waldteufel,classical,classical,{'0': 0.9589000000000001},003513
...,...,...,...,...,...
9508,brahms,classical,romantic,{'0': 0.9997},145996
16190,berger,classical,romantic,{'0': 0.9774},103604
20954,scarlatti,classical,baroque,{'0': 0.8200000000000001},073757
27219,weber,classical,romantic,{'0': 0.9262},033167


In [20]:
midi_filenames_train = (X_train["audio_name"].values).astype(str)
len(midi_filenames_train)

15175

In [21]:
midi_filenames_test = (X_test["audio_name"].values).astype(str)
len(midi_filenames_test)

5951

In [22]:
midi_filenames_val = (X_val["audio_name"].values).astype(str)
len(midi_filenames_test)

5951

In [23]:
from pathlib import Path
import shutil
import zipfile

def copy_files(origin, destination, filenames):
  with zipfile.ZipFile(destination, "w", zipfile.ZIP_DEFLATED) as zipf:
    for path_f in origin.rglob('*'):
      if path_f.is_file() and path_f.name.split("_")[0] in filenames:
        zipf.write(path_f, path_f.relative_to(origin))

copy_files(Path("/content/data2"),Path("/content/MIDI-LSTM-and-transformer-decoder/dataset_training.zip"), midi_filenames_train)
copy_files(Path("/content/data2"),Path("/content/MIDI-LSTM-and-transformer-decoder/dataset_test.zip"), midi_filenames_test)
copy_files(Path("/content/data2"),Path("/content/MIDI-LSTM-and-transformer-decoder/dataset_validation.zip"), midi_filenames_val)





In [ ]:
{'tokens': ['Bar_None', 'Position_0', 'Tempo_121.29', 'Position_13', 'Program_0', 'Pitch_43', 'Velocity_79', 'Duration_0.7.8', 'Program_0', 'Pitch_36', 'Velocity_79', 'Duration_0.7.8', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_79', 'Velocity_99', 'Duration_0.5.8', 'Program_0', 'Pitch_67', 'Velocity_99', 'Duration_0.7.8', 'Program_0', 'Pitch_76', 'Velocity_87', 'Duration_0.6.8', 'Position_18', 'Program_0', 'Pitch_79', 'Velocity_91', 'Duration_1.0.8', 'Position_20', 'Program_0', 'Pitch_76', 'Velocity_91', 'Duration_0.5.8', 'Program_0', 'Pitch_58', 'Velocity_75', 'Duration_0.7.8', 'Program_0', 'Pitch_61', 'Velocity_79', 'Duration_0.7.8', 'Program_0', 'Pitch_73', 'Velocity_79', 'Duration_0.7.8', 'Program_0', 'Pitch_67', 'Velocity_91', 'Duration_0.7.8', 'Position_23', 'Program_0', 'Pitch_70', 'Velocity_51', 'Duration_0.4.8', 'Position_24', 'Program_0', 'Pitch_76', 'Velocity_75', 'Duration_0.2.8', 'Position_26', 'Program_0', 'Pitch_57', 'Velocity_79', 'Duration_0.7.8', 'Program_0', 'Pitch_77', 'Velocity_91', 'Duration_0.5.8', 'Program_0', 'Pitch_65', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_60', 'Velocity_75', 'Duration_0.6.8', 'Position_31', 'Program_0', 'Pitch_77', 'Velocity_79', 'Duration_0.2.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_56', 'Velocity_71', 'Duration_0.7.8', 'Program_0', 'Pitch_71', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_74', 'Velocity_91', 'Duration_0.5.8', 'Program_0', 'Pitch_65', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_59', 'Velocity_75', 'Duration_0.6.8', 'Position_5', 'Program_0', 'Pitch_74', 'Velocity_79', 'Duration_0.2.8', 'Position_6', 'Program_0', 'Pitch_76', 'Velocity_91', 'Duration_0.5.8', 'Program_0', 'Pitch_58', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_64', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_55', 'Velocity_75', 'Duration_0.7.8', 'Program_0', 'Pitch_70', 'Velocity_91', 'Duration_0.7.8', 'Position_11', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_0.2.8', 'Position_12', 'Program_0', 'Pitch_57', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_72', 'Velocity_95', 'Duration_0.4.8', 'Program_0', 'Pitch_54', 'Velocity_75', 'Duration_0.6.8', 'Program_0', 'Pitch_63', 'Velocity_87', 'Duration_0.6.8', 'Position_13', 'Program_0', 'Pitch_69', 'Velocity_75', 'Duration_0.6.8', 'Program_0', 'Pitch_55', 'Velocity_47', 'Duration_0.5.8', 'Position_17', 'Program_0', 'Pitch_72', 'Velocity_79', 'Duration_0.2.8', 'Position_18', 'Program_0', 'Pitch_68', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_56', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_74', 'Velocity_91', 'Duration_0.5.8', 'Program_0', 'Pitch_62', 'Velocity_95', 'Duration_0.6.8', 'Position_19', 'Program_0', 'Pitch_53', 'Velocity_75', 'Duration_0.6.8', 'Position_23', 'Program_0', 'Pitch_74', 'Velocity_79', 'Duration_0.2.8', 'Position_24', 'Program_0', 'Pitch_67', 'Velocity_87', 'Duration_0.5.8', 'Position_25', 'Program_0', 'Pitch_62', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_55', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_59', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_65', 'Velocity_79', 'Duration_0.6.8', 'Position_29', 'Program_0', 'Pitch_67', 'Velocity_79', 'Duration_0.2.8', 'Position_31', 'Program_0', 'Pitch_43', 'Velocity_95', 'Duration_0.7.8', 'Program_0', 'Pitch_36', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_79', 'Velocity_99', 'Duration_0.5.8', 'Program_0', 'Pitch_67', 'Velocity_99', 'Duration_0.6.8', 'Program_0', 'Pitch_76', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_72', 'Velocity_95', 'Duration_0.6.8', 'Bar_None', 'Position_4', 'Program_0', 'Pitch_79', 'Velocity_91', 'Duration_1.0.8', 'Position_5', 'Program_0', 'Pitch_58', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_76', 'Velocity_95', 'Duration_0.4.8', 'Program_0', 'Pitch_67', 'Velocity_95', 'Duration_0.7.8', 'Program_0', 'Pitch_73', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_61', 'Velocity_91', 'Duration_0.6.8', 'Position_10', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_0.2.8', 'Position_11', 'Program_0', 'Pitch_77', 'Velocity_91', 'Duration_0.5.8', 'Position_12', 'Program_0', 'Pitch_60', 'Velocity_79', 'Duration_0.7.8', 'Program_0', 'Pitch_57', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_65', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.7.8', 'Position_16', 'Program_0', 'Pitch_77', 'Velocity_79', 'Duration_0.2.8', 'Position_17', 'Program_0', 'Pitch_56', 'Velocity_75', 'Duration_0.7.8', 'Program_0', 'Pitch_59', 'Velocity_87', 'Duration_0.7.8', 'Position_18', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.4.8', 'Program_0', 'Pitch_71', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_65', 'Velocity_91', 'Duration_0.7.8', 'Position_22', 'Program_0', 'Pitch_74', 'Velocity_79', 'Duration_0.2.8', 'Position_24', 'Program_0', 'Pitch_64', 'Velocity_95', 'Duration_0.7.8', 'Program_0', 'Pitch_76', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_55', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_60', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_67', 'Velocity_91', 'Duration_0.6.8', 'Position_28', 'Program_0', 'Pitch_72', 'Velocity_79', 'Duration_0.2.8', 'Position_30', 'Program_0', 'Pitch_67', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_62', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_55', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_71', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_65', 'Velocity_91', 'Duration_0.6.8', 'Bar_None', 'Position_2', 'Program_0', 'Pitch_74', 'Velocity_75', 'Duration_0.2.8', 'Position_3', 'Program_0', 'Pitch_64', 'Velocity_91', 'Duration_0.1.8', 'Position_4', 'Program_0', 'Pitch_60', 'Velocity_91', 'Duration_0.1.8', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.1.8', 'Position_7', 'Program_0', 'Pitch_55', 'Velocity_87', 'Duration_0.1.8', 'Program_0', 'Pitch_71', 'Velocity_87', 'Duration_0.1.8', 'Program_0', 'Pitch_65', 'Velocity_79', 'Duration_0.1.8', 'Position_10', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_87', 'Duration_0.1.8', 'Program_0', 'Pitch_48', 'Velocity_59', 'Duration_0.1.8', 'Position_12', 'Program_0', 'Pitch_43', 'Velocity_67', 'Duration_0.1.8', 'Position_16', 'Program_0', 'Pitch_79', 'Velocity_99', 'Duration_0.5.8', 'Program_0', 'Pitch_36', 'Velocity_87', 'Duration_1.5.8', 'Program_0', 'Pitch_43', 'Velocity_87', 'Duration_1.5.8', 'Position_17', 'Program_0', 'Pitch_67', 'Velocity_87', 'Duration_0.6.8', 'Position_19', 'Program_0', 'Pitch_72', 'Velocity_79', 'Duration_1.2.8', 'Position_21', 'Program_0', 'Pitch_79', 'Velocity_87', 'Duration_1.0.8', 'Position_22', 'Program_0', 'Pitch_58', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_61', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_76', 'Velocity_91', 'Duration_0.5.8', 'Position_23', 'Program_0', 'Pitch_67', 'Velocity_79', 'Duration_0.5.8', 'Position_25', 'Program_0', 'Pitch_73', 'Velocity_75', 'Duration_0.4.8', 'Position_27', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_0.2.8', 'Position_28', 'Program_0', 'Pitch_57', 'Velocity_87', 'Duration_1.4.8', 'Program_0', 'Pitch_60', 'Velocity_87', 'Duration_1.4.8', 'Program_0', 'Pitch_77', 'Velocity_91', 'Duration_0.5.8', 'Position_30', 'Program_0', 'Pitch_65', 'Velocity_75', 'Duration_0.6.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_72', 'Velocity_75', 'Duration_1.1.8', 'Position_1', 'Program_0', 'Pitch_77', 'Velocity_79', 'Duration_1.0.8', 'Position_2', 'Program_0', 'Pitch_59', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_56', 'Velocity_75', 'Duration_0.7.8', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.5.8', 'Position_4', 'Program_0', 'Pitch_65', 'Velocity_79', 'Duration_0.5.8', 'Position_5', 'Program_0', 'Pitch_71', 'Velocity_79', 'Duration_0.4.8', 'Position_7', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.2.8', 'Position_8', 'Program_0', 'Pitch_58', 'Velocity_95', 'Duration_0.7.8', 'Program_0', 'Pitch_55', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_76', 'Velocity_91', 'Duration_0.5.8', 'Position_10', 'Program_0', 'Pitch_64', 'Velocity_79', 'Duration_0.5.8', 'Position_11', 'Program_0', 'Pitch_70', 'Velocity_79', 'Duration_0.3.8', 'Position_13', 'Program_0', 'Pitch_76', 'Velocity_87', 'Duration_0.2.8', 'Position_14', 'Program_0', 'Pitch_54', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_72', 'Velocity_95', 'Duration_0.5.8', 'Program_0', 'Pitch_57', 'Velocity_95', 'Duration_0.6.8', 'Position_16', 'Program_0', 'Pitch_63', 'Velocity_79', 'Duration_0.5.8', 'Position_18', 'Program_0', 'Pitch_69', 'Velocity_75', 'Duration_0.3.8', 'Position_19', 'Program_0', 'Pitch_72', 'Velocity_79', 'Duration_0.2.8', 'Position_20', 'Program_0', 'Pitch_56', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.4.8', 'Position_21', 'Program_0', 'Pitch_53', 'Velocity_71', 'Duration_0.6.8', 'Program_0', 'Pitch_62', 'Velocity_87', 'Duration_0.5.8', 'Position_23', 'Program_0', 'Pitch_68', 'Velocity_79', 'Duration_0.4.8', 'Position_25', 'Program_0', 'Pitch_74', 'Velocity_79', 'Duration_0.2.8', 'Position_26', 'Program_0', 'Pitch_55', 'Velocity_71', 'Duration_0.7.8', 'Program_0', 'Pitch_67', 'Velocity_91', 'Duration_0.4.8', 'Position_27', 'Program_0', 'Pitch_59', 'Velocity_87', 'Duration_0.6.8', 'Position_28', 'Program_0', 'Pitch_62', 'Velocity_91', 'Duration_0.5.8', 'Position_29', 'Program_0', 'Pitch_65', 'Velocity_87', 'Duration_0.4.8', 'Position_30', 'Program_0', 'Pitch_67', 'Velocity_87', 'Duration_0.3.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_43', 'Velocity_99', 'Duration_0.1.8', 'Program_0', 'Pitch_36', 'Velocity_91', 'Duration_0.1.8', 'Position_1', 'Program_0', 'Pitch_79', 'Velocity_99', 'Duration_0.5.8', 'Position_2', 'Program_0', 'Pitch_67', 'Velocity_87', 'Duration_0.5.8', 'Position_4', 'Program_0', 'Pitch_72', 'Velocity_87', 'Duration_0.4.8', 'Position_5', 'Program_0', 'Pitch_79', 'Velocity_87', 'Duration_1.0.8', 'Position_7', 'Program_0', 'Pitch_58', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_76', 'Velocity_95', 'Duration_0.5.8', 'Program_0', 'Pitch_61', 'Velocity_95', 'Duration_0.7.8', 'Position_8', 'Program_0', 'Pitch_67', 'Velocity_75', 'Duration_0.5.8', 'Position_10', 'Program_0', 'Pitch_73', 'Velocity_79', 'Duration_0.4.8', 'Position_12', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_0.2.8', 'Position_13', 'Program_0', 'Pitch_60', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_57', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_77', 'Velocity_95', 'Duration_0.4.8', 'Position_14', 'Program_0', 'Pitch_65', 'Velocity_75', 'Duration_0.5.8', 'Position_16', 'Program_0', 'Pitch_72', 'Velocity_75', 'Duration_0.4.8', 'Position_17', 'Program_0', 'Pitch_77', 'Velocity_87', 'Duration_0.2.8', 'Position_19', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.5.8', 'Program_0', 'Pitch_59', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_56', 'Velocity_75', 'Duration_0.6.8', 'Position_20', 'Program_0', 'Pitch_65', 'Velocity_75', 'Duration_0.5.8', 'Position_21', 'Program_0', 'Pitch_71', 'Velocity_79', 'Duration_0.4.8', 'Position_24', 'Program_0', 'Pitch_74', 'Velocity_79', 'Duration_0.2.8', 'Position_25', 'Program_0', 'Pitch_55', 'Velocity_75', 'Duration_0.6.8', 'Program_0', 'Pitch_60', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_76', 'Velocity_87', 'Duration_0.6.8', 'Position_26', 'Program_0', 'Pitch_64', 'Velocity_75', 'Duration_0.5.8', 'Position_27', 'Program_0', 'Pitch_67', 'Velocity_87', 'Duration_0.4.8', 'Position_29', 'Program_0', 'Pitch_72', 'Velocity_87', 'Duration_0.2.8', 'Position_30', 'Program_0', 'Pitch_71', 'Velocity_87', 'Duration_0.6.8', 'Position_31', 'Program_0', 'Pitch_55', 'Velocity_75', 'Duration_0.6.8', 'Program_0', 'Pitch_62', 'Velocity_91', 'Duration_0.6.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_65', 'Velocity_79', 'Duration_0.5.8', 'Position_1', 'Program_0', 'Pitch_67', 'Velocity_87', 'Duration_0.4.8', 'Position_3', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.2.8', 'Position_4', 'Program_0', 'Pitch_72', 'Velocity_87', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_75', 'Duration_0.1.8', 'Position_5', 'Program_0', 'Pitch_64', 'Velocity_71', 'Duration_0.1.8', 'Position_7', 'Program_0', 'Pitch_55', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_65', 'Velocity_79', 'Duration_0.1.8', 'Position_8', 'Program_0', 'Pitch_71', 'Velocity_71', 'Duration_0.2.8', 'Position_10', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.2.8', 'Program_0', 'Pitch_64', 'Velocity_91', 'Duration_0.1.8', 'Program_0', 'Pitch_48', 'Velocity_79', 'Duration_0.1.8', 'Position_13', 'Program_0', 'Pitch_43', 'Velocity_67', 'Duration_0.1.8', 'Position_16', 'Program_0', 'Pitch_64', 'Velocity_99', 'Duration_0.7.8', 'Program_0', 'Pitch_76', 'Velocity_95', 'Duration_0.5.8', 'Program_0', 'Pitch_67', 'Velocity_99', 'Duration_0.7.8', 'Program_0', 'Pitch_36', 'Velocity_79', 'Duration_0.7.8', 'Program_0', 'Pitch_43', 'Velocity_79', 'Duration_0.7.8', 'Position_21', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_1.0.8', 'Position_23', 'Program_0', 'Pitch_64', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_67', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_60', 'Velocity_71', 'Duration_0.6.8', 'Program_0', 'Pitch_58', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_72', 'Velocity_87', 'Duration_0.4.8', 'Position_27', 'Program_0', 'Pitch_72', 'Velocity_87', 'Duration_0.2.8', 'Position_29', 'Program_0', 'Pitch_60', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_57', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_77', 'Velocity_95', 'Duration_0.5.8', 'Program_0', 'Pitch_65', 'Velocity_95', 'Duration_0.6.8', 'Bar_None', 'Position_1', 'Program_0', 'Pitch_77', 'Velocity_79', 'Duration_0.2.8', 'Position_3', 'Program_0', 'Pitch_71', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_56', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_59', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.5.8', 'Program_0', 'Pitch_65', 'Velocity_91', 'Duration_0.6.8', 'Position_8', 'Program_0', 'Pitch_74', 'Velocity_79', 'Duration_0.2.8', 'Position_9', 'Program_0', 'Pitch_36', 'Velocity_91', 'Duration_0.1.8', 'Program_0', 'Pitch_43', 'Velocity_95', 'Duration_0.1.8', 'Program_0', 'Pitch_76', 'Velocity_95', 'Duration_0.5.8', 'Program_0', 'Pitch_64', 'Velocity_99', 'Duration_0.7.8', 'Program_0', 'Pitch_67', 'Velocity_95', 'Duration_0.6.8', 'Position_13', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_0.2.8', 'Position_15', 'Program_0', 'Pitch_67', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_72', 'Velocity_95', 'Duration_0.4.8', 'Program_0', 'Pitch_63', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_57', 'Velocity_95', 'Duration_0.6.8', 'Position_19', 'Program_0', 'Pitch_72', 'Velocity_79', 'Duration_0.2.8', 'Position_21', 'Program_0', 'Pitch_62', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_66', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_58', 'Velocity_95', 'Duration_0.7.8', 'Program_0', 'Pitch_55', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.5.8', 'Position_26', 'Program_0', 'Pitch_74', 'Velocity_79', 'Duration_0.2.8', 'Position_27', 'Program_0', 'Pitch_67', 'Velocity_91', 'Duration_0.4.8', 'Program_0', 'Pitch_65', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_55', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_59', 'Velocity_91', 'Duration_0.1.8', 'Program_0', 'Pitch_62', 'Velocity_91', 'Duration_0.6.8', 'Position_31', 'Program_0', 'Pitch_67', 'Velocity_91', 'Duration_0.1.8', 'Bar_None', 'Position_1', 'Program_0', 'Pitch_76', 'Velocity_95', 'Duration_0.4.8', 'Program_0', 'Pitch_67', 'Velocity_95', 'Duration_0.6.8', 'Program_0', 'Pitch_43', 'Velocity_95', 'Duration_0.7.8', 'Program_0', 'Pitch_64', 'Velocity_99', 'Duration_0.6.8', 'Program_0', 'Pitch_36', 'Velocity_91', 'Duration_0.6.8', 'Position_6', 'Program_0', 'Pitch_76', 'Velocity_87', 'Duration_1.0.8', 'Position_7', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.4.8', 'Program_0', 'Pitch_67', 'Velocity_95', 'Duration_0.6.8', 'Position_8', 'Program_0', 'Pitch_60', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_58', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_64', 'Velocity_91', 'Duration_0.6.8', 'Position_11', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_1.0.8', 'Position_13', 'Program_0', 'Pitch_60', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_57', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_77', 'Velocity_95', 'Duration_0.4.8', 'Program_0', 'Pitch_65', 'Velocity_95', 'Duration_0.6.8', 'Position_18', 'Program_0', 'Pitch_77', 'Velocity_87', 'Duration_0.2.8', 'Position_19', 'Program_0', 'Pitch_71', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_59', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_56', 'Velocity_75', 'Duration_0.6.8', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.4.8', 'Program_0', 'Pitch_65', 'Velocity_91', 'Duration_0.6.8', 'Position_24', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.2.8', 'Position_25', 'Program_0', 'Pitch_55', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_87', 'Duration_0.1.8', 'Program_0', 'Pitch_76', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_67', 'Velocity_95', 'Duration_0.6.8', 'Program_0', 'Pitch_64', 'Velocity_91', 'Duration_0.6.8', 'Position_29', 'Program_0', 'Pitch_72', 'Velocity_79', 'Duration_0.7.8', 'Position_31', 'Program_0', 'Pitch_43', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_36', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_71', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_67', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_65', 'Velocity_95', 'Duration_0.6.8', 'Bar_None', 'Position_3', 'Program_0', 'Pitch_74', 'Velocity_79', 'Duration_0.2.8', 'Position_5', 'Program_0', 'Pitch_64', 'Velocity_95', 'Duration_0.1.8', 'Program_0', 'Pitch_72', 'Velocity_95', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_91', 'Duration_0.1.8', 'Position_8', 'Program_0', 'Pitch_55', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_71', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_65', 'Velocity_87', 'Duration_0.1.8', 'Position_11', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.1.8', 'Program_0', 'Pitch_48', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_87', 'Duration_0.1.8', 'Position_14', 'Program_0', 'Pitch_43', 'Velocity_67', 'Duration_0.1.8', 'Position_17', 'Program_0', 'Pitch_76', 'Velocity_95', 'Duration_0.5.8', 'Program_0', 'Pitch_43', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_36', 'Velocity_87', 'Duration_0.7.8', 'Position_18', 'Program_0', 'Pitch_64', 'Velocity_79', 'Duration_0.5.8', 'Position_20', 'Program_0', 'Pitch_67', 'Velocity_75', 'Duration_0.4.8', 'Position_22', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_1.0.8', 'Position_24', 'Program_0', 'Pitch_58', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_72', 'Velocity_95', 'Duration_0.4.8', 'Program_0', 'Pitch_60', 'Velocity_75', 'Duration_0.6.8', 'Position_25', 'Program_0', 'Pitch_64', 'Velocity_79', 'Duration_0.5.8', 'Position_26', 'Program_0', 'Pitch_67', 'Velocity_79', 'Duration_0.4.8', 'Position_28', 'Program_0', 'Pitch_72', 'Velocity_87', 'Duration_0.2.8', 'Position_29', 'Program_0', 'Pitch_57', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_77', 'Velocity_95', 'Duration_0.5.8', 'Position_30', 'Program_0', 'Pitch_60', 'Velocity_87', 'Duration_0.6.8', 'Position_31', 'Program_0', 'Pitch_65', 'Velocity_79', 'Duration_0.4.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_72', 'Velocity_75', 'Duration_0.3.8', 'Position_2', 'Program_0', 'Pitch_77', 'Velocity_87', 'Duration_1.0.8', 'Position_4', 'Program_0', 'Pitch_56', 'Velocity_75', 'Duration_0.6.8', 'Program_0', 'Pitch_59', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.4.8', 'Position_5', 'Program_0', 'Pitch_65', 'Velocity_87', 'Duration_0.5.8', 'Position_6', 'Program_0', 'Pitch_71', 'Velocity_79', 'Duration_0.4.8', 'Position_8', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.2.8', 'Position_10', 'Program_0', 'Pitch_76', 'Velocity_91', 'Duration_0.5.8', 'Program_0', 'Pitch_43', 'Velocity_95', 'Duration_0.7.8', 'Program_0', 'Pitch_36', 'Velocity_91', 'Duration_0.7.8', 'Position_11', 'Program_0', 'Pitch_64', 'Velocity_79', 'Duration_0.5.8', 'Position_13', 'Program_0', 'Pitch_67', 'Velocity_71', 'Duration_0.3.8', 'Position_15', 'Program_0', 'Pitch_76', 'Velocity_75', 'Duration_0.2.8', 'Position_16', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.4.8', 'Program_0', 'Pitch_57', 'Velocity_95', 'Duration_0.6.8', 'Position_17', 'Program_0', 'Pitch_63', 'Velocity_75', 'Duration_0.5.8', 'Position_19', 'Program_0', 'Pitch_67', 'Velocity_79', 'Duration_0.4.8', 'Position_20', 'Program_0', 'Pitch_72', 'Velocity_79', 'Duration_0.2.8', 'Position_22', 'Program_0', 'Pitch_74', 'Velocity_91', 'Duration_0.5.8', 'Program_0', 'Pitch_58', 'Velocity_91', 'Duration_1.4.8', 'Program_0', 'Pitch_55', 'Velocity_79', 'Duration_0.6.8', 'Position_23', 'Program_0', 'Pitch_62', 'Velocity_79', 'Duration_0.6.8', 'Position_24', 'Program_0', 'Pitch_66', 'Velocity_87', 'Duration_1.2.8', 'Position_26', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_1.0.8', 'Position_28', 'Program_0', 'Pitch_67', 'Velocity_91', 'Duration_0.4.8', 'Program_0', 'Pitch_55', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_59', 'Velocity_91', 'Duration_0.6.8', 'Position_29', 'Program_0', 'Pitch_62', 'Velocity_87', 'Duration_0.5.8', 'Position_31', 'Program_0', 'Pitch_65', 'Velocity_87', 'Duration_0.3.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_67', 'Velocity_79', 'Duration_0.2.8', 'Position_2', 'Program_0', 'Pitch_36', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_43', 'Velocity_99', 'Duration_0.7.8', 'Program_0', 'Pitch_76', 'Velocity_95', 'Duration_0.5.8', 'Position_3', 'Program_0', 'Pitch_64', 'Velocity_79', 'Duration_0.5.8', 'Position_5', 'Program_0', 'Pitch_67', 'Velocity_79', 'Duration_0.4.8', 'Position_6', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_1.0.8', 'Position_8', 'Program_0', 'Pitch_60', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.4.8', 'Program_0', 'Pitch_58', 'Velocity_91', 'Duration_0.6.8', 'Position_9', 'Program_0', 'Pitch_64', 'Velocity_79', 'Duration_0.5.8', 'Position_10', 'Program_0', 'Pitch_67', 'Velocity_87', 'Duration_0.4.8', 'Position_12', 'Program_0', 'Pitch_72', 'Velocity_87', 'Duration_0.2.8', 'Position_13', 'Program_0', 'Pitch_77', 'Velocity_95', 'Duration_0.5.8', 'Position_14', 'Program_0', 'Pitch_57', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_60', 'Velocity_91', 'Duration_0.6.8', 'Position_15', 'Program_0', 'Pitch_65', 'Velocity_87', 'Duration_0.5.8', 'Position_16', 'Program_0', 'Pitch_72', 'Velocity_87', 'Duration_0.3.8', 'Position_18', 'Program_0', 'Pitch_77', 'Velocity_87', 'Duration_1.5.8', 'Position_20', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.4.8', 'Program_0', 'Pitch_56', 'Velocity_79', 'Duration_1.4.8', 'Program_0', 'Pitch_59', 'Velocity_87', 'Duration_1.4.8', 'Position_21', 'Program_0', 'Pitch_65', 'Velocity_79', 'Duration_1.3.8', 'Position_22', 'Program_0', 'Pitch_71', 'Velocity_79', 'Duration_1.1.8', 'Position_24', 'Program_0', 'Pitch_74', 'Velocity_91', 'Duration_1.0.8', 'Position_26', 'Program_0', 'Pitch_55', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_76', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_60', 'Velocity_87', 'Duration_0.6.8', 'Position_27', 'Program_0', 'Pitch_64', 'Velocity_87', 'Duration_0.5.8', 'Position_28', 'Program_0', 'Pitch_67', 'Velocity_87', 'Duration_0.3.8', 'Position_30', 'Program_0', 'Pitch_72', 'Velocity_87', 'Duration_0.2.8', 'Position_31', 'Program_0', 'Pitch_71', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_43', 'Velocity_91', 'Duration_0.7.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_36', 'Velocity_79', 'Duration_0.6.8', 'Position_1', 'Program_0', 'Pitch_65', 'Velocity_87', 'Duration_0.5.8', 'Position_2', 'Program_0', 'Pitch_67', 'Velocity_79', 'Duration_0.4.8', 'Position_4', 'Program_0', 'Pitch_74', 'Velocity_75', 'Duration_0.2.8', 'Position_5', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.2.8', 'Program_0', 'Pitch_36', 'Velocity_87', 'Duration_0.1.8', 'Position_6', 'Program_0', 'Pitch_64', 'Velocity_71', 'Duration_0.1.8', 'Position_8', 'Program_0', 'Pitch_65', 'Velocity_87', 'Duration_0.1.8', 'Program_0', 'Pitch_43', 'Velocity_75', 'Duration_0.1.8', 'Position_10', 'Program_0', 'Pitch_71', 'Velocity_75', 'Duration_0.2.8', 'Position_11', 'Program_0', 'Pitch_72', 'Velocity_87', 'Duration_0.2.8', 'Program_0', 'Pitch_64', 'Velocity_91', 'Duration_0.1.8', 'Program_0', 'Pitch_48', 'Velocity_79', 'Duration_0.1.8', 'Position_14', 'Program_0', 'Pitch_55', 'Velocity_79', 'Duration_0.1.8', 'Position_17', 'Program_0', 'Pitch_79', 'Velocity_91', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_79', 'Duration_0.1.8', 'Position_19', 'Program_0', 'Pitch_81', 'Velocity_75', 'Duration_0.2.8', 'Position_20', 'Program_0', 'Pitch_67', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_72', 'Velocity_75', 'Duration_0.1.8', 'Position_22', 'Program_0', 'Pitch_77', 'Velocity_79', 'Duration_0.1.8', 'Position_23', 'Program_0', 'Pitch_76', 'Velocity_75', 'Duration_0.1.8', 'Position_24', 'Program_0', 'Pitch_67', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_73', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_71', 'Duration_0.1.8', 'Position_25', 'Program_0', 'Pitch_77', 'Velocity_71', 'Duration_0.1.8', 'Position_26', 'Program_0', 'Pitch_79', 'Velocity_79', 'Duration_0.1.8', 'Position_27', 'Program_0', 'Pitch_58', 'Velocity_75', 'Duration_0.1.8', 'Position_28', 'Program_0', 'Pitch_76', 'Velocity_71', 'Duration_0.2.8', 'Position_29', 'Program_0', 'Pitch_77', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_57', 'Velocity_87', 'Duration_0.1.8', 'Position_30', 'Program_0', 'Pitch_79', 'Velocity_75', 'Duration_0.2.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_69', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_59', 'Duration_0.1.8', 'Program_0', 'Pitch_77', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_65', 'Velocity_51', 'Duration_0.1.8', 'Position_1', 'Program_0', 'Pitch_76', 'Velocity_75', 'Duration_0.1.8', 'Position_3', 'Program_0', 'Pitch_74', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_71', 'Velocity_87', 'Duration_0.1.8', 'Program_0', 'Pitch_65', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_62', 'Velocity_67', 'Duration_0.1.8', 'Position_4', 'Program_0', 'Pitch_76', 'Velocity_75', 'Duration_0.1.8', 'Position_6', 'Program_0', 'Pitch_77', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_56', 'Velocity_67', 'Duration_0.1.8', 'Position_8', 'Program_0', 'Pitch_74', 'Velocity_59', 'Duration_0.1.8', 'Position_9', 'Program_0', 'Pitch_76', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_55', 'Velocity_79', 'Duration_0.1.8', 'Position_10', 'Program_0', 'Pitch_77', 'Velocity_71', 'Duration_0.2.8', 'Position_12', 'Program_0', 'Pitch_76', 'Velocity_75', 'Duration_0.2.8', 'Program_0', 'Pitch_67', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_51', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_51', 'Duration_0.1.8', 'Position_13', 'Program_0', 'Pitch_74', 'Velocity_71', 'Duration_0.1.8', 'Position_15', 'Program_0', 'Pitch_72', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_55', 'Duration_0.1.8', 'Program_0', 'Pitch_69', 'Velocity_75', 'Duration_0.1.8', 'Position_16', 'Program_0', 'Pitch_74', 'Velocity_71', 'Duration_0.1.8', 'Position_17', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_0.1.8', 'Position_18', 'Program_0', 'Pitch_54', 'Velocity_75', 'Duration_0.1.8', 'Position_19', 'Program_0', 'Pitch_72', 'Velocity_71', 'Duration_0.1.8', 'Position_20', 'Program_0', 'Pitch_74', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_53', 'Velocity_79', 'Duration_0.1.8', 'Position_21', 'Program_0', 'Pitch_76', 'Velocity_67', 'Duration_0.2.8', 'Position_23', 'Program_0', 'Pitch_74', 'Velocity_55', 'Duration_0.1.8', 'Program_0', 'Pitch_59', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_62', 'Velocity_75', 'Duration_0.1.8', 'Position_25', 'Program_0', 'Pitch_71', 'Velocity_67', 'Duration_0.1.8', 'Position_26', 'Program_0', 'Pitch_67', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_52', 'Velocity_79', 'Duration_0.1.8', 'Position_28', 'Program_0', 'Pitch_69', 'Velocity_67', 'Duration_0.1.8', 'Position_29', 'Program_0', 'Pitch_71', 'Velocity_59', 'Duration_0.1.8', 'Program_0', 'Pitch_50', 'Velocity_71', 'Duration_0.1.8', 'Position_31', 'Program_0', 'Pitch_74', 'Velocity_67', 'Duration_0.1.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_48', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_79', 'Velocity_87', 'Duration_0.1.8', 'Position_2', 'Program_0', 'Pitch_81', 'Velocity_79', 'Duration_0.1.8', 'Position_3', 'Program_0', 'Pitch_79', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_67', 'Velocity_67', 'Duration_0.1.8', 'Position_4', 'Program_0', 'Pitch_77', 'Velocity_75', 'Duration_0.1.8', 'Position_6', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_58', 'Velocity_71', 'Duration_0.1.8', 'Position_7', 'Program_0', 'Pitch_77', 'Velocity_71', 'Duration_0.1.8', 'Position_9', 'Program_0', 'Pitch_79', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_67', 'Velocity_59', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_67', 'Duration_0.1.8', 'Position_10', 'Program_0', 'Pitch_76', 'Velocity_67', 'Duration_0.1.8', 'Position_12', 'Program_0', 'Pitch_77', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_57', 'Velocity_79', 'Duration_0.1.8', 'Position_13', 'Program_0', 'Pitch_79', 'Velocity_67', 'Duration_0.2.8', 'Position_14', 'Program_0', 'Pitch_77', 'Velocity_71', 'Duration_0.1.8', 'Position_15', 'Program_0', 'Pitch_65', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_55', 'Duration_0.1.8', 'Position_16', 'Program_0', 'Pitch_76', 'Velocity_71', 'Duration_0.1.8', 'Position_17', 'Program_0', 'Pitch_74', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_56', 'Velocity_71', 'Duration_0.1.8', 'Position_19', 'Program_0', 'Pitch_76', 'Velocity_71', 'Duration_0.1.8', 'Position_20', 'Program_0', 'Pitch_77', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_65', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_59', 'Velocity_59', 'Duration_0.1.8', 'Position_22', 'Program_0', 'Pitch_74', 'Velocity_67', 'Duration_0.1.8', 'Position_23', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_54', 'Velocity_79', 'Duration_0.1.8', 'Position_24', 'Program_0', 'Pitch_77', 'Velocity_71', 'Duration_0.2.8', 'Position_26', 'Program_0', 'Pitch_76', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_75', 'Duration_0.1.8', 'Position_27', 'Program_0', 'Pitch_72', 'Velocity_75', 'Duration_0.1.8', 'Position_29', 'Program_0', 'Pitch_55', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_71', 'Velocity_75', 'Duration_0.1.8', 'Position_30', 'Program_0', 'Pitch_72', 'Velocity_75', 'Duration_0.1.8', 'Position_31', 'Program_0', 'Pitch_74', 'Velocity_75', 'Duration_0.1.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_62', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_65', 'Velocity_75', 'Duration_0.1.8', 'Position_1', 'Program_0', 'Pitch_71', 'Velocity_75', 'Duration_0.1.8', 'Position_2', 'Program_0', 'Pitch_72', 'Velocity_87', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_87', 'Duration_0.1.8', 'Position_3', 'Program_0', 'Pitch_60', 'Velocity_71', 'Duration_0.1.8', 'Position_4', 'Program_0', 'Pitch_74', 'Velocity_75', 'Duration_0.1.8', 'Position_5', 'Program_0', 'Pitch_72', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_55', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_62', 'Velocity_79', 'Duration_0.1.8', 'Position_6', 'Program_0', 'Pitch_71', 'Velocity_71', 'Duration_0.1.8', 'Position_8', 'Program_0', 'Pitch_72', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_48', 'Velocity_75', 'Duration_0.1.8', 'Position_11', 'Program_0', 'Pitch_55', 'Velocity_75', 'Duration_0.1.8', 'Position_14', 'Program_0', 'Pitch_76', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_75', 'Duration_0.1.8', 'Position_15', 'Program_0', 'Pitch_77', 'Velocity_71', 'Duration_0.1.8', 'Position_17', 'Program_0', 'Pitch_66', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_59', 'Duration_0.1.8', 'Position_18', 'Program_0', 'Pitch_74', 'Velocity_71', 'Duration_0.1.8', 'Position_19', 'Program_0', 'Pitch_72', 'Velocity_75', 'Duration_0.1.8', 'Position_20', 'Program_0', 'Pitch_64', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_67', 'Velocity_71', 'Duration_0.1.8', 'Position_21', 'Program_0', 'Pitch_74', 'Velocity_55', 'Duration_0.1.8', 'Position_23', 'Program_0', 'Pitch_76', 'Velocity_51', 'Duration_0.1.8', 'Program_0', 'Pitch_58', 'Velocity_55', 'Duration_0.1.8', 'Position_24', 'Program_0', 'Pitch_72', 'Velocity_59', 'Duration_0.1.8', 'Position_25', 'Program_0', 'Pitch_77', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_57', 'Velocity_87', 'Duration_0.1.8', 'Position_27', 'Program_0', 'Pitch_79', 'Velocity_71', 'Duration_0.2.8', 'Position_28', 'Program_0', 'Pitch_77', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_62', 'Velocity_75', 'Duration_0.1.8', 'Position_30', 'Program_0', 'Pitch_76', 'Velocity_71', 'Duration_0.1.8', 'Position_31', 'Program_0', 'Pitch_74', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_62', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_65', 'Velocity_71', 'Duration_0.1.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_76', 'Velocity_71', 'Duration_0.1.8', 'Position_2', 'Program_0', 'Pitch_77', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_56', 'Velocity_67', 'Duration_0.1.8', 'Position_4', 'Program_0', 'Pitch_74', 'Velocity_59', 'Duration_0.1.8', 'Position_5', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_55', 'Velocity_79', 'Duration_0.1.8', 'Position_6', 'Program_0', 'Pitch_77', 'Velocity_71', 'Duration_0.2.8', 'Position_7', 'Program_0', 'Pitch_76', 'Velocity_71', 'Duration_0.1.8', 'Position_8', 'Program_0', 'Pitch_60', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_62', 'Velocity_79', 'Duration_0.1.8', 'Position_9', 'Program_0', 'Pitch_74', 'Velocity_71', 'Duration_0.1.8', 'Position_10', 'Program_0', 'Pitch_72', 'Velocity_71', 'Duration_0.1.8', 'Position_11', 'Program_0', 'Pitch_64', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_51', 'Duration_0.1.8', 'Position_12', 'Program_0', 'Pitch_74', 'Velocity_59', 'Duration_0.1.8', 'Position_13', 'Program_0', 'Pitch_76', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_54', 'Velocity_75', 'Duration_0.1.8', 'Position_15', 'Program_0', 'Pitch_72', 'Velocity_71', 'Duration_0.1.8', 'Position_16', 'Program_0', 'Pitch_53', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_74', 'Velocity_71', 'Duration_0.1.8', 'Position_17', 'Program_0', 'Pitch_76', 'Velocity_75', 'Duration_0.2.8', 'Position_19', 'Program_0', 'Pitch_74', 'Velocity_59', 'Duration_0.1.8', 'Program_0', 'Pitch_55', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_59', 'Velocity_67', 'Duration_0.1.8', 'Position_20', 'Program_0', 'Pitch_71', 'Velocity_75', 'Duration_0.1.8', 'Position_22', 'Program_0', 'Pitch_67', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_52', 'Velocity_75', 'Duration_0.1.8', 'Position_23', 'Program_0', 'Pitch_71', 'Velocity_75', 'Duration_0.1.8', 'Position_24', 'Program_0', 'Pitch_74', 'Velocity_67', 'Duration_0.1.8', 'Position_25', 'Program_0', 'Pitch_50', 'Velocity_55', 'Duration_0.1.8', 'Program_0', 'Pitch_48', 'Velocity_39', 'Duration_0.1.8', 'Position_26', 'Program_0', 'Pitch_79', 'Velocity_71', 'Duration_0.1.8', 'Position_28', 'Program_0', 'Pitch_48', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_0.1.8', 'Position_29', 'Program_0', 'Pitch_77', 'Velocity_71', 'Duration_0.2.8', 'Position_30', 'Program_0', 'Pitch_76', 'Velocity_59', 'Duration_0.1.8', 'Position_31', 'Program_0', 'Pitch_64', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_67', 'Velocity_71', 'Duration_0.1.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_74', 'Velocity_67', 'Duration_0.1.8', 'Position_1', 'Program_0', 'Pitch_72', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_58', 'Velocity_79', 'Duration_0.1.8', 'Position_2', 'Program_0', 'Pitch_74', 'Velocity_75', 'Duration_0.1.8', 'Position_4', 'Program_0', 'Pitch_76', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_67', 'Velocity_67', 'Duration_0.1.8', 'Position_6', 'Program_0', 'Pitch_72', 'Velocity_75', 'Duration_0.1.8', 'Position_7', 'Program_0', 'Pitch_77', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_57', 'Velocity_87', 'Duration_0.1.8', 'Position_8', 'Program_0', 'Pitch_79', 'Velocity_75', 'Duration_0.2.8', 'Position_10', 'Program_0', 'Pitch_77', 'Velocity_67', 'Duration_0.2.8', 'Program_0', 'Pitch_60', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_65', 'Velocity_59', 'Duration_0.1.8', 'Position_11', 'Program_0', 'Pitch_76', 'Velocity_71', 'Duration_0.1.8', 'Position_12', 'Program_0', 'Pitch_74', 'Velocity_75', 'Duration_0.1.8', 'Position_13', 'Program_0', 'Pitch_56', 'Velocity_71', 'Duration_0.1.8', 'Position_14', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_0.1.8', 'Position_15', 'Program_0', 'Pitch_77', 'Velocity_75', 'Duration_0.1.8', 'Position_16', 'Program_0', 'Pitch_59', 'Velocity_59', 'Duration_0.1.8', 'Program_0', 'Pitch_65', 'Velocity_59', 'Duration_0.1.8', 'Position_17', 'Program_0', 'Pitch_74', 'Velocity_71', 'Duration_0.1.8', 'Position_18', 'Program_0', 'Pitch_54', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_76', 'Velocity_75', 'Duration_0.1.8', 'Position_20', 'Program_0', 'Pitch_77', 'Velocity_71', 'Duration_0.2.8', 'Position_21', 'Program_0', 'Pitch_76', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_71', 'Duration_0.1.8', 'Position_23', 'Program_0', 'Pitch_72', 'Velocity_71', 'Duration_0.1.8', 'Position_24', 'Program_0', 'Pitch_71', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_55', 'Velocity_75', 'Duration_0.1.8', 'Position_25', 'Program_0', 'Pitch_72', 'Velocity_71', 'Duration_0.1.8', 'Position_27', 'Program_0', 'Pitch_74', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_62', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_65', 'Velocity_79', 'Duration_0.1.8', 'Position_28', 'Program_0', 'Pitch_71', 'Velocity_67', 'Duration_0.1.8', 'Position_30', 'Program_0', 'Pitch_72', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_87', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_71', 'Duration_0.1.8', 'Position_31', 'Program_0', 'Pitch_74', 'Velocity_67', 'Duration_0.2.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_72', 'Velocity_75', 'Duration_0.2.8', 'Position_1', 'Program_0', 'Pitch_62', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_55', 'Velocity_71', 'Duration_0.1.8', 'Position_2', 'Program_0', 'Pitch_71', 'Velocity_75', 'Duration_0.1.8', 'Position_4', 'Program_0', 'Pitch_72', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_48', 'Velocity_79', 'Duration_0.1.8', 'Position_5', 'Program_0', 'Pitch_76', 'Velocity_91', 'Duration_0.2.8', 'Position_7', 'Program_0', 'Pitch_79', 'Velocity_87', 'Duration_0.1.8', 'Position_8', 'Program_0', 'Pitch_84', 'Velocity_79', 'Duration_0.1.8', 'Position_10', 'Program_0', 'Pitch_44', 'Velocity_95', 'Duration_1.6.8', 'Program_0', 'Pitch_87', 'Velocity_99', 'Duration_1.6.8', 'Program_0', 'Pitch_32', 'Velocity_75', 'Duration_1.6.8', 'Program_0', 'Pitch_80', 'Velocity_95', 'Duration_1.0.8', 'Position_12', 'Program_0', 'Pitch_84', 'Velocity_79', 'Duration_0.5.8', 'Program_0', 'Pitch_75', 'Velocity_79', 'Duration_0.5.8', 'Position_13', 'Program_0', 'Pitch_63', 'Velocity_79', 'Duration_1.0.8', 'Program_0', 'Pitch_72', 'Velocity_79', 'Duration_0.5.8', 'Position_15', 'Program_0', 'Pitch_68', 'Velocity_75', 'Duration_0.5.8', 'Program_0', 'Pitch_60', 'Velocity_71', 'Duration_0.5.8', 'Position_17', 'Program_0', 'Pitch_84', 'Velocity_95', 'Duration_0.7.8', 'Program_0', 'Pitch_75', 'Velocity_99', 'Duration_0.7.8', 'Position_18', 'Program_0', 'Pitch_80', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_72', 'Velocity_79', 'Duration_0.5.8', 'Position_20', 'Program_0', 'Pitch_68', 'Velocity_75', 'Duration_0.4.8', 'Program_0', 'Pitch_60', 'Velocity_59', 'Duration_0.3.8', 'Position_21', 'Program_0', 'Pitch_56', 'Velocity_71', 'Duration_0.3.8', 'Program_0', 'Pitch_63', 'Velocity_59', 'Duration_0.3.8', 'Position_23', 'Program_0', 'Pitch_85', 'Velocity_95', 'Duration_1.5.8', 'Program_0', 'Pitch_76', 'Velocity_91', 'Duration_0.7.8', 'Position_24', 'Program_0', 'Pitch_82', 'Velocity_87', 'Duration_0.5.8', 'Position_25', 'Program_0', 'Pitch_73', 'Velocity_87', 'Duration_0.5.8', 'Position_26', 'Program_0', 'Pitch_64', 'Velocity_79', 'Duration_0.7.8', 'Position_27', 'Program_0', 'Chord_min', 'Program_0', 'Pitch_58', 'Velocity_71', 'Duration_1.1.8', 'Program_0', 'Pitch_61', 'Velocity_67', 'Duration_0.5.8', 'Position_28', 'Program_0', 'Pitch_65', 'Velocity_71', 'Duration_1.0.8', 'Position_29', 'Program_0', 'Pitch_82', 'Velocity_95', 'Duration_0.7.8', 'Program_0', 'Pitch_73', 'Velocity_95', 'Duration_0.7.8', 'Position_30', 'Program_0', 'Pitch_70', 'Velocity_91', 'Duration_0.6.8', 'Position_31', 'Program_0', 'Pitch_76', 'Velocity_71', 'Duration_0.5.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_67', 'Velocity_79', 'Duration_0.4.8', 'Program_0', 'Pitch_61', 'Velocity_59', 'Duration_0.4.8', 'Position_1', 'Program_0', 'Pitch_64', 'Velocity_71', 'Duration_0.3.8', 'Position_4', 'Program_0', 'Pitch_75', 'Velocity_99', 'Duration_0.7.8', 'Program_0', 'Pitch_84', 'Velocity_99', 'Duration_1.5.8', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.5.8', 'Position_5', 'Program_0', 'Pitch_80', 'Velocity_95', 'Duration_0.5.8', 'Position_7', 'Program_0', 'Pitch_60', 'Velocity_67', 'Duration_0.7.8', 'Program_0', 'Pitch_68', 'Velocity_79', 'Duration_0.4.8', 'Program_0', 'Pitch_56', 'Velocity_67', 'Duration_0.5.8', 'Program_0', 'Pitch_63', 'Velocity_71', 'Duration_0.5.8', 'Position_10', 'Program_0', 'Pitch_80', 'Velocity_95', 'Duration_0.7.8', 'Program_0', 'Pitch_72', 'Velocity_95', 'Duration_0.7.8', 'Position_11', 'Program_0', 'Pitch_68', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_75', 'Velocity_87', 'Duration_0.6.8', 'Position_13', 'Program_0', 'Pitch_63', 'Velocity_75', 'Duration_0.4.8', 'Program_0', 'Pitch_56', 'Velocity_71', 'Duration_0.4.8', 'Position_14', 'Program_0', 'Pitch_60', 'Velocity_67', 'Duration_0.3.8', 'Program_0', 'Pitch_51', 'Velocity_67', 'Duration_0.3.8', 'Position_16', 'Program_0', 'Pitch_73', 'Velocity_99', 'Duration_0.7.8', 'Program_0', 'Pitch_82', 'Velocity_107', 'Duration_1.4.8', 'Position_17', 'Program_0', 'Pitch_75', 'Velocity_87', 'Duration_0.5.8', 'Program_0', 'Pitch_70', 'Velocity_87', 'Duration_0.5.8', 'Position_19', 'Program_0', 'Pitch_58', 'Velocity_91', 'Duration_0.7.8', 'Position_20', 'Program_0', 'Pitch_55', 'Velocity_79', 'Duration_1.0.8', 'Program_0', 'Pitch_60', 'Velocity_75', 'Duration_1.0.8', 'Program_0', 'Pitch_65', 'Velocity_59', 'Duration_1.0.8', 'Position_22', 'Program_0', 'Pitch_75', 'Velocity_95', 'Duration_0.6.8', 'Program_0', 'Pitch_70', 'Velocity_95', 'Duration_0.6.8', 'Position_23', 'Program_0', 'Pitch_63', 'Velocity_95', 'Duration_0.3.8', 'Program_0', 'Pitch_73', 'Velocity_87', 'Duration_0.5.8', 'Position_25', 'Program_0', 'Pitch_61', 'Velocity_87', 'Duration_0.3.8', 'Program_0', 'Pitch_63', 'Velocity_75', 'Duration_0.2.8', 'Position_26', 'Program_0', 'Pitch_67', 'Velocity_39', 'Duration_0.2.8', 'Program_0', 'Pitch_58', 'Velocity_51', 'Duration_0.2.8', 'Position_28', 'Program_0', 'Pitch_87', 'Velocity_99', 'Duration_1.5.8', 'Program_0', 'Pitch_80', 'Velocity_95', 'Duration_0.7.8', 'Program_0', 'Pitch_44', 'Velocity_87', 'Duration_1.4.8', 'Program_0', 'Pitch_56', 'Velocity_91', 'Duration_1.2.8', 'Position_29', 'Program_0', 'Pitch_84', 'Velocity_79', 'Duration_0.5.8', 'Program_0', 'Pitch_75', 'Velocity_91', 'Duration_0.5.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_63', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_72', 'Velocity_87', 'Duration_0.4.8', 'Position_1', 'Program_0', 'Pitch_68', 'Velocity_75', 'Duration_0.5.8', 'Program_0', 'Pitch_60', 'Velocity_79', 'Duration_0.4.8', 'Position_2', 'Program_0', 'Pitch_84', 'Velocity_95', 'Duration_0.6.8', 'Program_0', 'Pitch_75', 'Velocity_99', 'Duration_0.6.8', 'Position_3', 'Program_0', 'Pitch_80', 'Velocity_87', 'Duration_0.5.8', 'Position_4', 'Program_0', 'Pitch_72', 'Velocity_79', 'Duration_0.5.8', 'Position_5', 'Program_0', 'Pitch_60', 'Velocity_75', 'Duration_0.4.8', 'Program_0', 'Pitch_68', 'Velocity_79', 'Duration_0.3.8', 'Position_6', 'Program_0', 'Pitch_56', 'Velocity_75', 'Duration_0.2.8', 'Position_7', 'Program_0', 'Pitch_63', 'Velocity_67', 'Duration_0.2.8', 'Position_8', 'Program_0', 'Pitch_85', 'Velocity_95', 'Duration_1.5.8', 'Program_0', 'Pitch_76', 'Velocity_95', 'Duration_0.7.8', 'Position_9', 'Program_0', 'Pitch_82', 'Velocity_91', 'Duration_0.5.8', 'Position_10', 'Program_0', 'Pitch_73', 'Velocity_91', 'Duration_0.5.8', 'Position_11', 'Program_0', 'Pitch_64', 'Velocity_79', 'Duration_0.7.8', 'Position_12', 'Program_0', 'Pitch_58', 'Velocity_79', 'Duration_1.2.8', 'Program_0', 'Pitch_61', 'Velocity_75', 'Duration_0.5.8', 'Position_13', 'Program_0', 'Pitch_65', 'Velocity_75', 'Duration_1.0.8', 'Position_15', 'Program_0', 'Pitch_82', 'Velocity_99', 'Duration_0.7.8', 'Program_0', 'Pitch_73', 'Velocity_95', 'Duration_0.7.8', 'Position_16', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_70', 'Velocity_87', 'Duration_0.6.8', 'Position_17', 'Program_0', 'Pitch_67', 'Velocity_75', 'Duration_0.4.8', 'Position_18', 'Program_0', 'Pitch_61', 'Velocity_67', 'Duration_0.4.8', 'Program_0', 'Pitch_64', 'Velocity_71', 'Duration_0.3.8', 'Position_21', 'Program_0', 'Pitch_84', 'Velocity_95', 'Duration_3.1.8', 'Program_0', 'Pitch_75', 'Velocity_95', 'Duration_0.7.8', 'Position_22', 'Program_0', 'Pitch_80', 'Velocity_91', 'Duration_1.3.8', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_2.7.8', 'Position_24', 'Program_0', 'Pitch_60', 'Velocity_67', 'Duration_2.6.8', 'Program_0', 'Pitch_68', 'Velocity_71', 'Duration_1.2.8', 'Position_25', 'Program_0', 'Pitch_56', 'Velocity_67', 'Duration_1.4.8', 'Program_0', 'Pitch_63', 'Velocity_71', 'Duration_0.5.8', 'Position_27', 'Program_0', 'Pitch_73', 'Velocity_91', 'Duration_2.2.8', 'Program_0', 'Pitch_82', 'Velocity_99', 'Duration_1.7.8', 'Position_28', 'Program_0', 'Pitch_75', 'Velocity_87', 'Duration_2.2.8', 'Program_0', 'Pitch_70', 'Velocity_87', 'Duration_1.7.8', 'Position_30', 'Program_0', 'Pitch_55', 'Velocity_47', 'Duration_2.0.8', 'Program_0', 'Pitch_63', 'Velocity_71', 'Duration_2.0.8', 'Position_31', 'Program_0', 'Pitch_58', 'Velocity_67', 'Duration_1.7.8', 'Program_0', 'Pitch_51', 'Velocity_67', 'Duration_1.4.8', 'Bar_None', 'Position_1', 'Program_0', 'Pitch_80', 'Velocity_91', 'Duration_0.6.8', 'Position_3', 'Program_0', 'Pitch_68', 'Velocity_79', 'Duration_0.6.8', 'Position_4', 'Program_0', 'Pitch_56', 'Velocity_71', 'Duration_1.1.8', 'Program_0', 'Pitch_79', 'Velocity_87', 'Duration_1.1.8', 'Position_6', 'Program_0', 'Pitch_44', 'Velocity_79', 'Duration_1.0.8', 'Program_0', 'Pitch_67', 'Velocity_79', 'Duration_1.0.8', 'Position_8', 'Program_0', 'Pitch_80', 'Velocity_91', 'Duration_0.6.8', 'Position_9', 'Program_0', 'Pitch_68', 'Velocity_79', 'Duration_0.5.8', 'Position_10', 'Program_0', 'Pitch_82', 'Velocity_87', 'Duration_0.3.8', 'Position_11', 'Program_0', 'Pitch_51', 'Velocity_71', 'Duration_0.3.8', 'Program_0', 'Pitch_70', 'Velocity_87', 'Duration_0.2.8', 'Position_12', 'Program_0', 'Pitch_39', 'Velocity_67', 'Duration_0.2.8'], 'ids': [1323, 2670, 15613, 4632, 6734, 14548, 14178, 593, 1576, 662, 1014, 6866, 4922, 5815, 4496, 659, 1381, 594, 1019, 518, 1239, 5827, 6614, 5790, 6733, 552, 1519, 529, 1563, 4994, 5167, 6614, 7528, 552, 1171, 18367, 6866, 6758, 7667, 4963, 659, 1203, 529, 1060, 4765, 11938, 4467, 593, 943, 4311, 607, 1089, 529, 1455, 8444, 5167, 6061, 708, 1860, 552, 987, 19898, 578, 994, 5546, 1937, 2148, 563, 930, 529, 2664, 726, 581, 6734, 14548, 12897, 4765, 708, 4046, 662, 1775, 5647, 8086, 10855, 5823, 637, 1196, 529, 1555, 623, 903, 5016, 5702, 5790, 659, 1265, 529, 1753, 4641, 611, 980, 4532, 5702, 659, 981, 529, 881, 8764, 6808, 4350, 4232, 637, 893, 529, 918, 4878, 6808, 2238, 5394, 637, 2298, 19244, 596, 1207, 15261, 1472, 2707, 2673, 19030, 3170, 3045, 497, 2525, 14187, 821, 581, 741, 523, 741, 933, 593, 1080, 641, 1520, 613, 1523, 6425, 6676, 623, 941, 551, 1586, 554, 1199, 529, 1042, 7980, 9172, 623, 1112, 552, 895, 609, 3148, 584, 2409, 6819, 4578, 578, 1256, 551, 1353, 576, 1082, 569, 1382, 10786, 4817, 623, 1037, 551, 1375, 526, 1188, 569, 1945, 4765, 9733, 708, 1201, 551, 948, 514, 1080, 529, 1609, 4864, 617, 1840, 3803, 578, 1467, 576, 992, 529, 1366, 4342, 689, 1426, 593, 860, 623, 1117, 617, 918, 546, 2468, 10083, 596, 3180, 821, 1990, 578, 1167, 617, 1734, 613, 1674, 6676, 10784, 726, 875, 537, 1516, 576, 1005, 529, 1086, 5488, 6841, 803, 1114, 537, 891, 554, 1492, 569, 989, 5281, 6819, 552, 921, 537, 1159, 576, 865, 529, 1360, 4025, 4817, 593, 1033, 537, 924, 617, 1100, 569, 1151, 593, 1342, 3791, 637, 902, 551, 2755, 617, 1428, 569, 1167, 2490, 15073, 15798, 19071, 1046, 17228, 5807, 3870, 507, 2670, 11822, 14178, 8257, 844, 581, 9212, 583, 1194, 584, 1020, 5265, 6733, 4418, 4300, 617, 1099, 569, 1097, 5488, 4765, 6841, 9573, 708, 2115, 529, 1713, 8444, 4863, 4355, 4944, 637, 885, 529, 3905, 4102, 4387, 8753, 15258, 708, 1188, 529, 913, 5797, 9723, 5488, 708, 1080, 529, 974, 7498, 7888, 10786, 4355, 578, 1021, 529, 924, 5831, 6808, 2265, 3037, 637, 920, 596, 1836, 8086, 15016, 10208, 818, 581, 637, 1324, 613, 1187, 5326, 708, 976, 4926, 4361, 637, 1059, 662, 1086, 6969, 6841, 9068, 708, 1571, 569, 1161, 6132, 6819, 4146, 4449, 637, 865, 569, 1360, 2003, 2520, 6045, 9180, 637, 1100, 583, 2664, 15685, 5001, 4699, 6628, 708, 2804, 529, 1217, 4238, 4509, 596, 1215, 2238, 12694, 1059, 3870, 9338, 2650, 13301, 16093, 15084, 611, 1024, 551, 855, 554, 1198, 584, 1301, 4300, 8222, 552, 1028, 551, 954, 576, 893, 569, 1234, 5637, 696, 1073, 593, 1124, 576, 895, 514, 2426, 613, 1880, 4366, 4355, 617, 1326, 578, 1297, 576, 885, 569, 1196, 9853, 726, 581, 659, 1047, 551, 925, 508, 1191, 14564, 6181, 708, 1400, 537, 944, 576, 904, 17119, 6866, 11445, 563, 971, 563, 1126, 673, 1021, 613, 842, 6098, 5729, 637, 991, 578, 1124, 546, 846, 529, 5808, 11861, 14178, 696, 1524, 551, 1144, 576, 1324, 584, 976, 5790, 6747, 637, 1116, 551, 910, 617, 896, 569, 1498, 696, 1253, 6291, 637, 1111, 578, 891, 546, 1571, 741, 872, 5670, 7926, 688, 1118, 670, 1158, 620, 865, 662, 1366, 5239, 6291, 593, 1031, 578, 842, 546, 1070, 569, 1176, 10407, 659, 3499, 563, 2834, 578, 1990, 576, 1065, 18433, 12119, 534, 1172, 12253, 3353, 15179, 17170, 3997, 3870, 507, 1346, 507, 1548, 3151, 507, 1803, 14709, 1528, 6809, 1569, 507, 1186, 11823, 2109, 8815, 1583, 13929, 507, 1550, 12237, 512, 1584, 11093, 1543, 518, 854, 1502, 2477, 10830, 2971, 12805, 1977, 2673, 5772, 1259, 18282, 2292, 9451, 15324, 9562, 1587, 15531, 2901, 1528, 8530, 985, 11895, 1972, 1671, 2204, 10687, 11420, 507, 1999, 13990, 9208, 2094, 507, 1194, 15610, 2312, 6063, 1153, 11388, 2072, 15982, 11854, 2586, 10441, 491, 2016, 7818, 2608, 507, 1923, 1429, 8528, 1622, 14193, 2382, 15367, 15443, 1590, 6471, 1196, 12905, 10502, 1547, 520, 1510, 12650, 9549, 1000, 10533, 2438, 11681, 13900, 1882, 9306, 981, 12803, 2878, 18841, 17446, 1502, 10402, 1099, 16257, 8258, 1070, 10730, 496, 868, 2148, 2585, 472, 13206, 13026, 1657, 9601, 14162, 1708, 6833, 1297, 10588, 13053, 1367, 13962, 7413, 1564, 16494, 1972, 12240, 10946, 13244, 6480, 1004, 17823, 9340, 914, 19593, 11093, 1603, 19033, 1528, 6572, 1184, 10441, 1250, 1882, 490, 1008, 15628, 2292, 10631, 14846, 10928, 1712, 16765, 12474, 1276, 14637, 11127, 12608, 12362, 869, 15672, 2573, 12906, 18598, 1908, 11420, 14428, 2029, 7057, 977, 12826, 8895, 1156, 10345, 491, 1965, 3491, 592, 1570, 490, 2052, 6909, 1584, 15898, 13676, 6767, 867, 19575, 8807, 2037, 12372, 1429, 8528, 1241, 18677, 11093, 1362, 518, 1587, 2989, 10463, 1203, 9511, 496, 1756, 11884, 507, 1564, 17645, 8449, 988, 490, 1999, 2099, 15489, 15568, 1502, 6154, 1092, 11658, 8927, 1091, 9909, 1250, 9051, 934, 11853, 1429, 2490, 10441, 520, 895, 518, 2750, 1708, 13700, 12136, 13342, 1384, 658, 1669, 534, 2246, 507, 2863, 1101, 550, 1967, 675, 15201, 732, 2210, 5169, 551, 1358, 4729, 551, 1448, 3779, 536, 2404, 11188, 844, 2154, 4765, 551, 1247, 3274, 524, 1768, 3607, 524, 2831, 16245, 659, 2165, 578, 1586, 578, 1033, 583, 216, 3080, 481, 5929, 538, 915, 565, 2344, 10855, 726, 1370, 637, 1212, 536, 846, 4221, 557, 2769, 508, 1718, 19563, 1871, 469, 623, 2290, 696, 1208, 5542, 4498, 4315, 536, 2187, 8098, 726, 1463, 7206, 593, 1358, 4004, 540, 1063, 5316, 517, 1279, 19498, 2982, 1665, 5002, 578, 1557, 659, 1178, 4652, 4734, 582, 1693, 8814, 708, 1440, 10925, 578, 1601, 5276, 18121, 9641, 590, 2998, 1871, 501, 726, 532, 10337, 724, 2419, 5169, 623, 1209, 5797, 617, 3102, 3779, 576, 3057, 10050, 818, 2507, 578, 1167, 551, 1270, 3780, 526, 1914, 518, 1582, 520, 2726, 16245, 726, 2393, 623, 1516, 623, 1047, 583, 1333, 7345, 537, 1113, 570, 2356, 844, 483, 726, 1000, 4869, 593, 933, 554, 1473, 3399, 508, 2405, 2802, 487, 726, 2168, 9439, 2096, 909, 900, 482, 618, 1771, 7436, 536, 1491, 1224, 506, 2114, 1387, 16294, 773, 1345, 11650, 713, 1530, 16466, 642, 2752, 637, 1954, 563, 1880, 7922, 655, 3095, 4542, 584, 2056, 637, 1578, 551, 2335, 546, 2256, 3571, 569, 3473, 520], 'bytes': '$ÞŖëŪ;\x91¤Ū4\x91¤ŪX\x94¤Ū_\x96¢ŪS\x96¤Ū\\\x93£ðŪ_\x94¥òŪ\\\x94¢ŪJ\x90¤ŪM\x91¤ŪY\x91¤ŪS\x94¤õŪV\x8a¡öŪ\\\x90\x9føŪI\x91¤Ū]\x94¢ŪQ\x94£ŪX\x94¤ŪL\x90£ýŪ]\x91\x9f$ÞŪH\x8f¤ŪW\x93¤ŪZ\x94¢ŪQ\x94¤ŪK\x90£ãŪZ\x91\x9fäŪ\\\x94¢ŪJ\x94¤ŪP\x94¤ŪG\x90¤ŪV\x94¤éŪ\\\x91\x9fêŪI\x93£ŪX\x95¡ŪF\x90£ŪO\x93£ëŪU\x90£ŪG\x89¢ïŪX\x91\x9fðŪT\x94¤ŪH\x93¤ŪZ\x94¢ŪN\x95£ñŪE\x90£õŪZ\x91\x9föŪS\x93¢÷ŪN\x93£ŪG\x90\x9eŪK\x91\x9eŪQ\x91£ûŪS\x91\x9fýŪ;\x95¤Ū4\x94¤Ū_\x96¢ŪS\x96£Ū\\\x93£ŪX\x95£$âŪ_\x94¥ãŪJ\x93¤Ū\\\x95¡ŪS\x95¤ŪY\x93£ŪM\x94£èŪ\\\x91\x9féŪ]\x94¢êŪL\x91¤ŪI\x93¤ŪQ\x94£ŪX\x94¤îŪ]\x91\x9fïŪH\x90¤ŪK\x93¤ðŪZ\x93¡ŪW\x93¤ŪQ\x94¤ôŪZ\x91\x9föŪP\x95¤Ū\\\x94£ŪG\x91£ŪL\x91£ŪS\x94£úŪX\x91\x9füŪS\x93£ŪN\x94£ŪG\x91\x9eŪW\x93£ŪQ\x94£$àŪZ\x90\x9fáŪP\x94\x9eâŪL\x94\x9eŪX\x94\x9eåŪG\x93\x9eŪW\x93\x9eŪQ\x91\x9eèŪX\x94\x9eŪP\x93\x9eŪ@\x8c\x9eêŪ;\x8e\x9eîŪ_\x96¢Ū4\x93ªŪ;\x93ªïŪS\x93£ñŪX\x91§óŪ_\x93¥ôŪJ\x93¤ŪM\x94¤Ū\\\x94¢õŪS\x91¢÷ŪY\x90¡ùŪ\\\x91\x9fúŪI\x93©ŪL\x93©Ū]\x94¢üŪQ\x90£$ÞŪX\x90¦ßŪ]\x91¥àŪK\x93¤ŪH\x90¤ŪZ\x93¢âŪQ\x91¢ãŪW\x91¡åŪZ\x93\x9fæŪJ\x95¤ŪG\x91£Ū\\\x94¢èŪP\x91¢éŪV\x91\xa0ëŪ\\\x93\x9fìŪF\x93£ŪX\x95¢ŪI\x95£îŪO\x91¢ðŪU\x90\xa0ñŪX\x91\x9fòŪH\x93£ŪZ\x93¡óŪE\x8f£ŪN\x93¢õŪT\x91¡÷ŪZ\x91\x9føŪG\x8f¤ŪS\x94¡ùŪK\x93£úŪN\x94¢ûŪQ\x93¡üŪS\x93\xa0$ÞŪ;\x96\x9eŪ4\x94\x9eßŪ_\x96¢àŪS\x93¢âŪX\x93¡ãŪ_\x93¥åŪJ\x94¤Ū\\\x95¢ŪM\x95¤æŪS\x90¢èŪY\x91¡êŪ\\\x91\x9fëŪL\x93£ŪI\x94£Ū]\x95¡ìŪQ\x90¢îŪX\x90¡ïŪ]\x93\x9fñŪZ\x93¢ŪK\x93¤ŪH\x90£òŪQ\x90¢óŪW\x91¡öŪZ\x91\x9f÷ŪG\x90£ŪL\x91£Ū\\\x93£øŪP\x90¢ùŪS\x93¡ûŪX\x93\x9füŪW\x93£ýŪG\x90£ŪN\x94£$ÞŪQ\x91¢ßŪS\x93¡áŪZ\x93\x9fâŪX\x93\x9eŪL\x90\x9eãŪP\x8f\x9eåŪG\x8f\x9eŪQ\x91\x9eæŪW\x8f\x9fèŪX\x94\x9fŪP\x94\x9eŪ@\x91\x9eëŪ;\x8e\x9eîŪP\x96¤Ū\\\x95¢ŪS\x96¤Ū4\x91¤Ū;\x91¤óŪ\\\x91¥õŪP\x93¤ŪS\x94¤ŪL\x8f£ŪJ\x91£ŪX\x93¡ùŪX\x93\x9fûŪL\x93£ŪI\x93£ŪX\x94£Ū]\x95¢ŪQ\x95£$ßŪ]\x91\x9fáŪW\x94¤ŪH\x91£ŪK\x91£ŪZ\x93¢ŪQ\x94£æŪZ\x91\x9fçŪ4\x94\x9eŪ;\x95\x9eŪ\\\x95¢ŪP\x96¤ŪS\x95£ëŪ\\\x91\x9fíŪS\x94¤ŪX\x95¡ŪO\x93£ŪI\x95£ñŪX\x91\x9fóŪN\x94£ŪR\x94¤ŪJ\x95¤ŪG\x91£ŪZ\x93¢øŪZ\x91\x9fùŪS\x94¡ŪQ\x94£ŪG\x91\x9eŪK\x94\x9eŪN\x94£ýŪS\x94\x9e$ßŪ\\\x95¡ŪS\x95£Ū;\x95¤ŪP\x96£Ū4\x94£äŪ\\\x93¥åŪX\x94¡ŪS\x95£æŪL\x91£ŪJ\x91£ŪP\x94£éŪX\x94¥ëŪL\x94£ŪI\x94£Ū]\x95¡ŪQ\x95£ðŪ]\x93\x9fñŪW\x93¤ŪK\x93¤ŪH\x90£ŪZ\x93¡ŪQ\x94£öŪZ\x93\x9f÷ŪG\x91\x9eŪL\x93\x9eŪ\\\x94£ŪS\x95£ŪP\x94£ûŪX\x91¤ýŪ;\x93£Ū4\x91£ŪW\x93£ŪS\x94£ŪQ\x95£$áŪZ\x91\x9fãŪP\x95\x9eŪX\x95\x9eŪL\x94\x9eæŪG\x91\x9eŪW\x91\x9eŪQ\x93\x9eéŪX\x94\x9eŪ@\x91\x9eŪP\x93\x9eìŪ;\x8e\x9eïŪ\\\x95¢Ū;\x93¤Ū4\x93¤ðŪP\x91¢òŪS\x90¡ôŪ\\\x91¥öŪJ\x91£ŪX\x95¡ŪL\x90£÷ŪP\x91¢øŪS\x91¡úŪX\x93\x9fûŪI\x93£Ū]\x95¢üŪL\x93£ýŪQ\x91¡$ÞŪX\x90\xa0àŪ]\x93¥âŪH\x90£ŪK\x91£ŪZ\x93¡ãŪQ\x93¢äŪW\x91¡æŪZ\x93\x9fèŪ\\\x94¢Ū;\x95¤Ū4\x94¤éŪP\x91¢ëŪS\x8f\xa0íŪ\\\x90\x9fîŪX\x94¡ŪI\x95£ïŪO\x90¢ñŪS\x91¡òŪX\x91\x9fôŪZ\x94¢ŪJ\x94©ŪG\x91£õŪN\x91£öŪR\x93§øŪZ\x93¥úŪS\x94¡ŪG\x93£ŪK\x94£ûŪN\x93¢ýŪQ\x93\xa0$ÞŪS\x91\x9fàŪ4\x94¤Ū;\x96¤Ū\\\x95¢áŪP\x91¢ãŪS\x91¡äŪ\\\x91¥æŪL\x94£ŪX\x94¡ŪJ\x94£çŪP\x91¢èŪS\x93¡êŪX\x93\x9fëŪ]\x95¢ìŪI\x94£ŪL\x94£íŪQ\x93¢îŪX\x93\xa0ðŪ]\x93ªòŪZ\x93¡ŪH\x91©ŪK\x93©óŪQ\x91¨ôŪW\x91¦öŪZ\x94¥øŪG\x93£Ū\\\x94£ŪL\x93£ùŪP\x93¢úŪS\x93\xa0üŪX\x93\x9fýŪW\x93¤Ū;\x94¤$ÞŪ4\x91£ßŪQ\x93¢àŪS\x91¡âŪZ\x90\x9fãŪX\x94\x9fŪ4\x93\x9eäŪP\x8f\x9eæŪQ\x93\x9eŪ;\x90\x9eèŪW\x90\x9féŪX\x93\x9fŪP\x94\x9eŪ@\x91\x9eìŪG\x91\x9eïŪ_\x94\x9eŪL\x91\x9eñŪa\x90\x9fòŪS\x8e\x9eŪP\x90\x9eŪX\x90\x9eôŪ]\x91\x9eõŪ\\\x90\x9eöŪS\x8f\x9eŪY\x90\x9eŪP\x8f\x9e÷Ū]\x8f\x9eøŪ_\x91\x9eùŪJ\x90\x9eúŪ\\\x8f\x9fûŪ]\x91\x9eŪI\x93\x9eüŪ_\x90\x9f$ÞŪU\x8e\x9eŪL\x8c\x9eŪ]\x8e\x9eŪQ\x8a\x9eßŪ\\\x90\x9eáŪZ\x90\x9eŪW\x93\x9eŪQ\x8e\x9eŪN\x8e\x9eâŪ\\\x90\x9eäŪ]\x8f\x9eŪH\x8e\x9eæŪZ\x8c\x9eçŪ\\\x90\x9eŪG\x91\x9eèŪ]\x8f\x9fêŪ\\\x90\x9fŪS\x8e\x9eŪP\x8a\x9eŪL\x8a\x9eëŪZ\x8f\x9eíŪX\x91\x9eŪP\x90\x9eŪL\x8b\x9eŪU\x90\x9eîŪZ\x8f\x9eïŪ\\\x91\x9eðŪF\x90\x9eñŪX\x8f\x9eòŪZ\x8f\x9eŪE\x91\x9eóŪ\\\x8e\x9fõŪZ\x8b\x9eŪK\x8f\x9eŪN\x90\x9e÷ŪW\x8e\x9eøŪS\x8f\x9eŪD\x91\x9eúŪU\x8e\x9eûŪW\x8c\x9eŪB\x8f\x9eýŪZ\x8e\x9e$ÞŪ@\x91\x9eŪ_\x93\x9eàŪa\x91\x9eáŪ_\x8f\x9eŪP\x90\x9eŪS\x8e\x9eâŪ]\x90\x9eäŪ\\\x91\x9eŪJ\x8f\x9eåŪ]\x8f\x9eçŪ_\x8e\x9eŪS\x8c\x9eŪP\x8e\x9eèŪ\\\x8e\x9eêŪ]\x91\x9eŪI\x91\x9eëŪ_\x8e\x9fìŪ]\x8f\x9eíŪQ\x8f\x9eŪL\x8b\x9eîŪ\\\x8f\x9eïŪZ\x90\x9eŪH\x8f\x9eñŪ\\\x8f\x9eòŪ]\x90\x9eŪQ\x8e\x9eŪK\x8c\x9eôŪZ\x8e\x9eõŪ\\\x91\x9eŪF\x91\x9eöŪ]\x8f\x9føŪ\\\x8e\x9eŪL\x8e\x9eŪP\x90\x9eùŪX\x90\x9eûŪG\x90\x9eŪW\x90\x9eüŪX\x90\x9eýŪZ\x90\x9e$ÞŪN\x91\x9eŪQ\x90\x9eßŪW\x90\x9eàŪX\x93\x9eŪP\x93\x9eáŪL\x8f\x9eâŪZ\x90\x9eãŪX\x8f\x9eŪG\x90\x9eŪN\x91\x9eäŪW\x8f\x9eæŪX\x90\x9eŪ@\x90\x9eéŪG\x90\x9eìŪ\\\x90\x9eŪL\x90\x9eíŪ]\x8f\x9eïŪR\x91\x9eŪP\x8c\x9eðŪZ\x8f\x9eñŪX\x90\x9eòŪP\x8e\x9eŪS\x8f\x9eóŪZ\x8b\x9eõŪ\\\x8a\x9eŪJ\x8b\x9eöŪX\x8c\x9e÷Ū]\x91\x9eŪI\x93\x9eùŪ_\x8f\x9fúŪ]\x8e\x9eŪP\x90\x9eŪN\x90\x9eüŪ\\\x8f\x9eýŪZ\x8f\x9eŪN\x90\x9eŪQ\x8f\x9e$ÞŪ\\\x8f\x9eàŪ]\x8f\x9eŪH\x8e\x9eâŪZ\x8c\x9eãŪ\\\x91\x9eŪG\x91\x9eäŪ]\x8f\x9fåŪ\\\x8f\x9eæŪL\x8e\x9eŪN\x91\x9eçŪZ\x8f\x9eèŪX\x8f\x9eéŪP\x8f\x9eŪL\x8a\x9eêŪZ\x8c\x9eëŪ\\\x90\x9eŪF\x90\x9eíŪX\x8f\x9eîŪE\x91\x9eŪZ\x8f\x9eïŪ\\\x90\x9fñŪZ\x8c\x9eŪG\x8e\x9eŪK\x8e\x9eòŪW\x90\x9eôŪS\x8f\x9eŪD\x90\x9eõŪW\x90\x9eöŪZ\x8e\x9e÷ŪB\x8b\x9eŪ@\x87\x9eøŪ_\x8f\x9eúŪ@\x90\x9eŪ\\\x91\x9eûŪ]\x8f\x9füŪ\\\x8c\x9eýŪP\x8f\x9eŪS\x8f\x9e$ÞŪZ\x8e\x9eßŪX\x90\x9eŪJ\x91\x9eàŪZ\x90\x9eâŪ\\\x8f\x9eŪP\x90\x9eŪS\x8e\x9eäŪX\x90\x9eåŪ]\x91\x9eŪI\x93\x9eæŪ_\x90\x9fèŪ]\x8e\x9fŪL\x8e\x9eŪQ\x8c\x9eéŪ\\\x8f\x9eêŪZ\x90\x9eëŪH\x8f\x9eìŪ\\\x91\x9eíŪ]\x90\x9eîŪK\x8c\x9eŪQ\x8c\x9eïŪZ\x8f\x9eðŪF\x91\x9eŪ\\\x90\x9eòŪ]\x8f\x9fóŪ\\\x8e\x9eŪL\x8e\x9eŪP\x8f\x9eõŪX\x8f\x9eöŪW\x90\x9eŪG\x90\x9e÷ŪX\x8f\x9eùŪZ\x8f\x9eŪN\x90\x9eŪQ\x91\x9eúŪW\x8e\x9eüŪX\x8f\x9eŪP\x93\x9eŪL\x8f\x9eýŪZ\x8e\x9f$ÞŪX\x90\x9fßŪN\x8f\x9eŪG\x8f\x9eàŪW\x90\x9eâŪX\x91\x9eŪ@\x91\x9eãŪ\\\x94\x9fåŪ_\x93\x9eæŪd\x91\x9eèŪ<\x95«Ūg\x96«Ū0\x90«Ū`\x95¥êŪd\x91¢Ū[\x91¢ëŪO\x91¥ŪX\x91¢íŪT\x90¢ŪL\x8f¢ïŪd\x95¤Ū[\x96¤ðŪ`\x93£ŪX\x91¢òŪT\x90¡ŪL\x8c\xa0óŪH\x8f\xa0ŪO\x8c\xa0õŪe\x95ªŪ\\\x94¤öŪb\x93¢÷ŪY\x93¢øŪP\x91¤ùŪļŪJ\x8f¦ŪM\x8e¢úŪQ\x8f¥ûŪb\x95¤ŪY\x95¤üŪV\x94£ýŪ\\\x8f¢$ÞŪS\x91¡ŪM\x8c¡ßŪP\x8f\xa0âŪ[\x96¤Ūd\x96ªŪX\x94¢ãŪ`\x95¢åŪL\x8e¤ŪT\x91¡ŪH\x8e¢ŪO\x8f¢èŪ`\x95¤ŪX\x95¤éŪT\x94£Ū[\x93£ëŪO\x90¡ŪH\x8f¡ìŪL\x8e\xa0ŪC\x8e\xa0îŪY\x96¤Ūb\x98©ïŪ[\x93¢ŪV\x93¢ñŪJ\x94¤òŪG\x91¥ŪL\x90¥ŪQ\x8c¥ôŪ[\x95£ŪV\x95£õŪO\x95\xa0ŪY\x93¢÷ŪM\x93\xa0ŪO\x90\x9føŪS\x87\x9fŪJ\x8a\x9fúŪg\x96ªŪ`\x95¤Ū<\x93©ŪH\x94§ûŪd\x91¢Ū[\x94¢$ÞŪO\x94¤ŪX\x93¡ßŪT\x90¢ŪL\x91¡àŪd\x95£Ū[\x96£áŪ`\x93¢âŪX\x91¢ãŪL\x90¡ŪT\x91\xa0äŪH\x90\x9fåŪO\x8e\x9fæŪe\x95ªŪ\\\x95¤çŪb\x94¢èŪY\x94¢éŪP\x91¤êŪJ\x91§ŪM\x90¢ëŪQ\x90¥íŪb\x96¤ŪY\x95¤îŪ\\\x91£ŪV\x93£ïŪS\x90¡ðŪM\x8e¡ŪP\x8f\xa0óŪd\x95¶Ū[\x95¤ôŪ`\x94¨ŪX\x94´öŪL\x8e³ŪT\x8f§÷ŪH\x8e©ŪO\x8f¢ùŪY\x94¯Ūb\x96¬úŪ[\x93¯ŪV\x93¬üŪG\x89\xadŪO\x8f\xadýŪJ\x8e¬ŪC\x8e©$ßŪ`\x94£áŪT\x91£âŪH\x8f¦Ū_\x93¦äŪ<\x91¥ŪS\x91¥æŪ`\x94£çŪT\x91¢èŪb\x93\xa0éŪC\x8f\xa0ŪV\x93\x9fêŪ7\x8e\x9f', 'events': [Event(type=Bar, value=None, time=0, desc=0), Event(type=Position, value=0, time=0, desc=0), Event(type=Tempo, value=121.29, time=0, desc=121.29004087474378), Event(type=Position, value=13, time=13, desc=13), Event(type=Program, value=0, time=13, desc=20), Event(type=Pitch, value=43, time=13, desc=20), Event(type=Velocity, value=79, time=13, desc=79), Event(type=Duration, value=0.7.8, time=13, desc=7 ticks), Event(type=Program, value=0, time=13, desc=20), Event(type=Pitch, value=36, time=13, desc=20), Event(type=Velocity, value=79, time=13, desc=79), Event(type=Duration, value=0.7.8, time=13, desc=7 ticks), Event(type=Program, value=0, time=13, desc=20), Event(type=Pitch, value=72, time=13, desc=20), Event(type=Velocity, value=91, time=13, desc=91), Event(type=Duration, value=0.7.8, time=13, desc=7 ticks), Event(type=Program, value=0, time=13, desc=18), Event(type=Pitch, value=79, time=13, desc=18), Event(type=Velocity, value=99, time=13, desc=99), Event(type=Duration, value=0.5.8, time=13, desc=5 ticks), Event(type=Program, value=0, time=13, desc=20), Event(type=Pitch, value=67, time=13, desc=20), Event(type=Velocity, value=99, time=13, desc=99), Event(type=Duration, value=0.7.8, time=13, desc=7 ticks), Event(type=Program, value=0, time=13, desc=19), Event(type=Pitch, value=76, time=13, desc=19), Event(type=Velocity, value=87, time=13, desc=87), Event(type=Duration, value=0.6.8, time=13, desc=6 ticks), Event(type=Position, value=18, time=18, desc=18), Event(type=Program, value=0, time=18, desc=26), Event(type=Pitch, value=79, time=18, desc=26), Event(type=Velocity, value=91, time=18, desc=91), Event(type=Duration, value=1.0.8, time=18, desc=8 ticks), Event(type=Position, value=20, time=20, desc=20), Event(type=Program, value=0, time=20, desc=25), Event(type=Pitch, value=76, time=20, desc=25), Event(type=Velocity, value=91, time=20, desc=91), Event(type=Duration, value=0.5.8, time=20, desc=5 ticks), Event(type=Program, value=0, time=20, desc=27), Event(type=Pitch, value=58, time=20, desc=27), Event(type=Velocity, value=75, time=20, desc=75), Event(type=Duration, value=0.7.8, time=20, desc=7 ticks), Event(type=Program, value=0, time=20, desc=27), Event(type=Pitch, value=61, time=20, desc=27), Event(type=Velocity, value=79, time=20, desc=79), Event(type=Duration, value=0.7.8, time=20, desc=7 ticks), Event(type=Program, value=0, time=20, desc=27), Event(type=Pitch, value=73, time=20, desc=27), Event(type=Velocity, value=79, time=20, desc=79), Event(type=Duration, value=0.7.8, time=20, desc=7 ticks), Event(type=Program, value=0, time=20, desc=27), Event(type=Pitch, value=67, time=20, desc=27), Event(type=Velocity, value=91, time=20, desc=91), Event(type=Duration, value=0.7.8, time=20, desc=7 ticks), Event(type=Position, value=23, time=23, desc=23), Event(type=Program, value=0, time=23, desc=27), Event(type=Pitch, value=70, time=23, desc=27), Event(type=Velocity, value=51, time=23, desc=51), Event(type=Duration, value=0.4.8, time=23, desc=4 ticks), Event(type=Position, value=24, time=24, desc=24), Event(type=Program, value=0, time=24, desc=26), Event(type=Pitch, value=76, time=24, desc=26), Event(type=Velocity, value=75, time=24, desc=75), Event(type=Duration, value=0.2.8, time=24, desc=2 ticks), Event(type=Position, value=26, time=26, desc=26), Event(type=Program, value=0, time=26, desc=33), Event(type=Pitch, value=57, time=26, desc=33), Event(type=Velocity, value=79, time=26, desc=79), Event(type=Duration, value=0.7.8, time=26, desc=7 ticks), Event(type=Program, value=0, time=26, desc=31), Event(type=Pitch, value=77, time=26, desc=31), Event(type=Velocity, value=91, time=26, desc=91), Event(type=Duration, value=0.5.8, time=26, desc=5 ticks), Event(type=Program, value=0, time=26, desc=32), Event(type=Pitch, value=65, time=26, desc=32), Event(type=Velocity, value=91, time=26, desc=91), Event(type=Duration, value=0.6.8, time=26, desc=6 ticks), Event(type=Program, value=0, time=26, desc=33), Event(type=Pitch, value=72, time=26, desc=33), Event(type=Velocity, value=91, time=26, desc=91), Event(type=Duration, value=0.7.8, time=26, desc=7 ticks), Event(type=Program, value=0, time=26, desc=32), Event(type=Pitch, value=60, time=26, desc=32), Event(type=Velocity, value=75, time=26, desc=75), Event(type=Duration, value=0.6.8, time=26, desc=6 ticks), Event(type=Position, value=31, time=31, desc=31), Event(type=Program, value=0, time=31, desc=33), Event(type=Pitch, value=77, time=31, desc=33), Event(type=Velocity, value=79, time=31, desc=79), Event(type=Duration, value=0.2.8, time=31, desc=2 ticks), Event(type=Bar, value=None, time=32, desc=0), Event(type=Position, value=0, time=32, desc=32), Event(type=Program, value=0, time=32, desc=39), Event(type=Pitch, value=56, time=32, desc=39), Event(type=Velocity, value=71, time=32, desc=71), Event(type=Duration, value=0.7.8, time=32, desc=7 ticks), Event(type=Program, value=0, time=32, desc=39), Event(type=Pitch, value=71, time=32, desc=39), Event(type=Velocity, value=87, time=32, desc=87), Event(type=Duration, value=0.7.8, time=32, desc=7 ticks), Event(type=Program, value=0, time=32, desc=37), Event(type=Pitch, value=74, time=32, desc=37), Event(type=Velocity, value=91, time=32, desc=91), Event(type=Duration, value=0.5.8, time=32, desc=5 ticks), Event(type=Program, value=0, time=32, desc=39), Event(type=Pitch, value=65, time=32, desc=39), Event(type=Velocity, value=91, time=32, desc=91), Event(type=Duration, value=0.7.8, time=32, desc=7 ticks), Event(type=Program, value=0, time=32, desc=38), Event(type=Pitch, value=59, time=32, desc=38), Event(type=Velocity, value=75, time=32, desc=75), Event(type=Duration, value=0.6.8, time=32, desc=6 ticks), Event(type=Position, value=5, time=37, desc=37), Event(type=Program, value=0, time=37, desc=39), Event(type=Pitch, value=74, time=37, desc=39), Event(type=Velocity, value=79, time=37, desc=79), Event(type=Duration, value=0.2.8, time=37, desc=2 ticks), Event(type=Position, value=6, time=38, desc=38), Event(type=Program, value=0, time=38, desc=43), Event(type=Pitch, value=76, time=38, desc=43), Event(type=Velocity, value=91, time=38, desc=91), Event(type=Duration, value=0.5.8, time=38, desc=5 ticks), Event(type=Program, value=0, time=38, desc=45), Event(type=Pitch, value=58, time=38, desc=45), Event(type=Velocity, value=91, time=38, desc=91), Event(type=Duration, value=0.7.8, time=38, desc=7 ticks), Event(type=Program, value=0, time=38, desc=45), Event(type=Pitch, value=64, time=38, desc=45), Event(type=Velocity, value=91, time=38, desc=91), Event(type=Duration, value=0.7.8, time=38, desc=7 ticks), Event(type=Program, value=0, time=38, desc=45), Event(type=Pitch, value=55, time=38, desc=45), Event(type=Velocity, value=75, time=38, desc=75), Event(type=Duration, value=0.7.8, time=38, desc=7 ticks), Event(type=Program, value=0, time=38, desc=45), Event(type=Pitch, value=70, time=38, desc=45), Event(type=Velocity, value=91, time=38, desc=91), Event(type=Duration, value=0.7.8, time=38, desc=7 ticks), Event(type=Position, value=11, time=43, desc=43), Event(type=Program, value=0, time=43, desc=45), Event(type=Pitch, value=76, time=43, desc=45), Event(type=Velocity, value=79, time=43, desc=79), Event(type=Duration, value=0.2.8, time=43, desc=2 ticks), Event(type=Position, value=12, time=44, desc=44), Event(type=Program, value=0, time=44, desc=50), Event(type=Pitch, value=57, time=44, desc=50), Event(type=Velocity, value=87, time=44, desc=87), Event(type=Duration, value=0.6.8, time=44, desc=6 ticks), Event(type=Program, value=0, time=44, desc=48), Event(type=Pitch, value=72, time=44, desc=48), Event(type=Velocity, value=95, time=44, desc=95), Event(type=Duration, value=0.4.8, time=44, desc=4 ticks), Event(type=Program, value=0, time=44, desc=50), Event(type=Pitch, value=54, time=44, desc=50), Event(type=Velocity, value=75, time=44, desc=75), Event(type=Duration, value=0.6.8, time=44, desc=6 ticks), Event(type=Program, value=0, time=44, desc=50), Event(type=Pitch, value=63, time=44, desc=50), Event(type=Velocity, value=87, time=44, desc=87), Event(type=Duration, value=0.6.8, time=44, desc=6 ticks), Event(type=Position, value=13, time=45, desc=45), Event(type=Program, value=0, time=45, desc=51), Event(type=Pitch, value=69, time=45, desc=51), Event(type=Velocity, value=75, time=45, desc=75), Event(type=Duration, value=0.6.8, time=45, desc=6 ticks), Event(type=Program, value=0, time=45, desc=50), Event(type=Pitch, value=55, time=45, desc=50), Event(type=Velocity, value=47, time=45, desc=47), Event(type=Duration, value=0.5.8, time=45, desc=5 ticks), Event(type=Position, value=17, time=49, desc=49), Event(type=Program, value=0, time=49, desc=51), Event(type=Pitch, value=72, time=49, desc=51), Event(type=Velocity, value=79, time=49, desc=79), Event(type=Duration, value=0.2.8, time=49, desc=2 ticks), Event(type=Position, value=18, time=50, desc=50), Event(type=Program, value=0, time=50, desc=57), Event(type=Pitch, value=68, time=50, desc=57), Event(type=Velocity, value=91, time=50, desc=91), Event(type=Duration, value=0.7.8, time=50, desc=7 ticks), Event(type=Program, value=0, time=50, desc=57), Event(type=Pitch, value=56, time=50, desc=57), Event(type=Velocity, value=87, time=50, desc=87), Event(type=Duration, value=0.7.8, time=50, desc=7 ticks), Event(type=Program, value=0, time=50, desc=55), Event(type=Pitch, value=74, time=50, desc=55), Event(type=Velocity, value=91, time=50, desc=91), Event(type=Duration, value=0.5.8, time=50, desc=5 ticks), Event(type=Program, value=0, time=50, desc=56), Event(type=Pitch, value=62, time=50, desc=56), Event(type=Velocity, value=95, time=50, desc=95), Event(type=Duration, value=0.6.8, time=50, desc=6 ticks), Event(type=Position, value=19, time=51, desc=51), Event(type=Program, value=0, time=51, desc=57), Event(type=Pitch, value=53, time=51, desc=57), Event(type=Velocity, value=75, time=51, desc=75), Event(type=Duration, value=0.6.8, time=51, desc=6 ticks), Event(type=Position, value=23, time=55, desc=55), Event(type=Program, value=0, time=55, desc=57), Event(type=Pitch, value=74, time=55, desc=57), Event(type=Velocity, value=79, time=55, desc=79), Event(type=Duration, value=0.2.8, time=55, desc=2 ticks), Event(type=Position, value=24, time=56, desc=56), Event(type=Program, value=0, time=56, desc=61), Event(type=Pitch, value=67, time=56, desc=61), Event(type=Velocity, value=87, time=56, desc=87), Event(type=Duration, value=0.5.8, time=56, desc=5 ticks), Event(type=Position, value=25, time=57, desc=57), Event(type=Program, value=0, time=57, desc=63), Event(type=Pitch, value=62, time=57, desc=63), Event(type=Velocity, value=87, time=57, desc=87), Event(type=Duration, value=0.6.8, time=57, desc=6 ticks), Event(type=Program, value=0, time=57, desc=58), Event(type=Pitch, value=55, time=57, desc=58), Event(type=Velocity, value=75, time=57, desc=75), Event(type=Duration, value=0.1.8, time=57, desc=1 ticks), Event(type=Program, value=0, time=57, desc=58), Event(type=Pitch, value=59, time=57, desc=58), Event(type=Velocity, value=79, time=57, desc=79), Event(type=Duration, value=0.1.8, time=57, desc=1 ticks), Event(type=Program, value=0, time=57, desc=63), Event(type=Pitch, value=65, time=57, desc=63), Event(type=Velocity, value=79, time=57, desc=79), Event(type=Duration, value=0.6.8, time=57, desc=6 ticks), Event(type=Position, value=29, time=61, desc=61), Event(type=Program, value=0, time=61, desc=63), Event(type=Pitch, value=67, time=61, desc=63), Event(type=Velocity, value=79, time=61, desc=79), Event(type=Duration, value=0.2.8, time=61, desc=2 ticks), Event(type=Position, value=31, time=63, desc=63), Event(type=Program, value=0, time=63, desc=70), Event(type=Pitch, value=43, time=63, desc=70), Event(type=Velocity, value=95, time=63, desc=95), Event(type=Duration, value=0.7.8, time=63, desc=7 ticks), Event(type=Program, value=0, time=63, desc=70), Event(type=Pitch, value=36, time=63, desc=70), Event(type=Velocity, value=91, time=63, desc=91), Event(type=Duration, value=0.7.8, time=63, desc=7 ticks), Event(type=Program, value=0, time=63, desc=68), Event(type=Pitch, value=79, time=63, desc=68), Event(type=Velocity, value=99, time=63, desc=99), Event(type=Duration, value=0.5.8, time=63, desc=5 ticks), Event(type=Program, value=0, time=63, desc=69), Event(type=Pitch, value=67, time=63, desc=69), Event(type=Velocity, value=99, time=63, desc=99), Event(type=Duration, value=0.6.8, time=63, desc=6 ticks), Event(type=Program, value=0, time=63, desc=69), Event(type=Pitch, value=76, time=63, desc=69), Event(type=Velocity, value=87, time=63, desc=87), Event(type=Duration, value=0.6.8, time=63, desc=6 ticks), Event(type=Program, value=0, time=63, desc=69), Event(type=Pitch, value=72, time=63, desc=69), Event(type=Velocity, value=95, time=63, desc=95), Event(type=Duration, value=0.6.8, time=63, desc=6 ticks), Event(type=Bar, value=None, time=64, desc=0), Event(type=Position, value=4, time=68, desc=68), Event(type=Program, value=0, time=68, desc=76), Event(type=Pitch, value=79, time=68, desc=76), Event(type=Velocity, value=91, time=68, desc=91), Event(type=Duration, value=1.0.8, time=68, desc=8 ticks), Event(type=Position, value=5, time=69, desc=69), Event(type=Program, value=0, time=69, desc=76), Event(type=Pitch, value=58, time=69, desc=76), Event(type=Velocity, value=87, time=69, desc=87), Event(type=Duration, value=0.7.8, time=69, desc=7 ticks), Event(type=Program, value=0, time=69, desc=73), Event(type=Pitch, value=76, time=69, desc=73), Event(type=Velocity, value=95, time=69, desc=95), Event(type=Duration, value=0.4.8, time=69, desc=4 ticks), Event(type=Program, value=0, time=69, desc=76), Event(type=Pitch, value=67, time=69, desc=76), Event(type=Velocity, value=95, time=69, desc=95), Event(type=Duration, value=0.7.8, time=69, desc=7 ticks), Event(type=Program, value=0, time=69, desc=75), Event(type=Pitch, value=73, time=69, desc=75), Event(type=Velocity, value=87, time=69, desc=87), Event(type=Duration, value=0.6.8, time=69, desc=6 ticks), Event(type=Program, value=0, time=69, desc=75), Event(type=Pitch, value=61, time=69, desc=75), Event(type=Velocity, value=91, time=69, desc=91), Event(type=Duration, value=0.6.8, time=69, desc=6 ticks), Event(type=Position, value=10, time=74, desc=74), Event(type=Program, value=0, time=74, desc=76), Event(type=Pitch, value=76, time=74, desc=76), Event(type=Velocity, value=79, time=74, desc=79), Event(type=Duration, value=0.2.8, time=74, desc=2 ticks), Event(type=Position, value=11, time=75, desc=75), Event(type=Program, value=0, time=75, desc=80), Event(type=Pitch, value=77, time=75, desc=80), Event(type=Velocity, value=91, time=75, desc=91), Event(type=Duration, value=0.5.8, time=75, desc=5 ticks), Event(type=Position, value=12, time=76, desc=76), Event(type=Program, value=0, time=76, desc=83), Event(type=Pitch, value=60, time=76, desc=83), Event(type=Velocity, value=79, time=76, desc=79), Event(type=Duration, value=0.7.8, time=76, desc=7 ticks), Event(type=Program, value=0, time=76, desc=83), Event(type=Pitch, value=57, time=76, desc=83), Event(type=Velocity, value=87, time=76, desc=87), Event(type=Duration, value=0.7.8, time=76, desc=7 ticks), Event(type=Program, value=0, time=76, desc=82), Event(type=Pitch, value=65, time=76, desc=82), Event(type=Velocity, value=91, time=76, desc=91), Event(type=Duration, value=0.6.8, time=76, desc=6 ticks), Event(type=Program, value=0, time=76, desc=83), Event(type=Pitch, value=72, time=76, desc=83), Event(type=Velocity, value=91, time=76, desc=91), Event(type=Duration, value=0.7.8, time=76, desc=7 ticks), Event(type=Position, value=16, time=80, desc=80), Event(type=Program, value=0, time=80, desc=82), Event(type=Pitch, value=77, time=80, desc=82), Event(type=Velocity, value=79, time=80, desc=79), Event(type=Duration, value=0.2.8, time=80, desc=2 ticks), Event(type=Position, value=17, time=81, desc=81), Event(type=Program, value=0, time=81, desc=88), Event(type=Pitch, value=56, time=81, desc=88), Event(type=Velocity, value=75, time=81, desc=75), Event(type=Duration, value=0.7.8, time=81, desc=7 ticks), Event(type=Program, value=0, time=81, desc=88), Event(type=Pitch, value=59, time=81, desc=88), Event(type=Velocity, value=87, time=81, desc=87), Event(type=Duration, value=0.7.8, time=81, desc=7 ticks), Event(type=Position, value=18, time=82, desc=82), Event(type=Program, value=0, time=82, desc=86), Event(type=Pitch, value=74, time=82, desc=86), Event(type=Velocity, value=87, time=82, desc=87), Event(type=Duration, value=0.4.8, time=82, desc=4 ticks), Event(type=Program, value=0, time=82, desc=89), Event(type=Pitch, value=71, time=82, desc=89), Event(type=Velocity, value=87, time=82, desc=87), Event(type=Duration, value=0.7.8, time=82, desc=7 ticks), Event(type=Program, value=0, time=82, desc=89), Event(type=Pitch, value=65, time=82, desc=89), Event(type=Velocity, value=91, time=82, desc=91), Event(type=Duration, value=0.7.8, time=82, desc=7 ticks), Event(type=Position, value=22, time=86, desc=86), Event(type=Program, value=0, time=86, desc=88), Event(type=Pitch, value=74, time=86, desc=88), Event(type=Velocity, value=79, time=86, desc=79), Event(type=Duration, value=0.2.8, time=86, desc=2 ticks), Event(type=Position, value=24, time=88, desc=88), Event(type=Program, value=0, time=88, desc=95), Event(type=Pitch, value=64, time=88, desc=95), Event(type=Velocity, value=95, time=88, desc=95), Event(type=Duration, value=0.7.8, time=88, desc=7 ticks), Event(type=Program, value=0, time=88, desc=94), Event(type=Pitch, value=76, time=88, desc=94), Event(type=Velocity, value=91, time=88, desc=91), Event(type=Duration, value=0.6.8, time=88, desc=6 ticks), Event(type=Program, value=0, time=88, desc=94), Event(type=Pitch, value=55, time=88, desc=94), Event(type=Velocity, value=79, time=88, desc=79), Event(type=Duration, value=0.6.8, time=88, desc=6 ticks), Event(type=Program, value=0, time=88, desc=94), Event(type=Pitch, value=60, time=88, desc=94), Event(type=Velocity, value=79, time=88, desc=79), Event(type=Duration, value=0.6.8, time=88, desc=6 ticks), Event(type=Program, value=0, time=88, desc=94), Event(type=Pitch, value=67, time=88, desc=94), Event(type=Velocity, value=91, time=88, desc=91), Event(type=Duration, value=0.6.8, time=88, desc=6 ticks), Event(type=Position, value=28, time=92, desc=92), Event(type=Program, value=0, time=92, desc=94), Event(type=Pitch, value=72, time=92, desc=94), Event(type=Velocity, value=79, time=92, desc=79), Event(type=Duration, value=0.2.8, time=92, desc=2 ticks), Event(type=Position, value=30, time=94, desc=94), Event(type=Program, value=0, time=94, desc=100), Event(type=Pitch, value=67, time=94, desc=100), Event(type=Velocity, value=87, time=94, desc=87), Event(type=Duration, value=0.6.8, time=94, desc=6 ticks), Event(type=Program, value=0, time=94, desc=100), Event(type=Pitch, value=62, time=94, desc=100), Event(type=Velocity, value=91, time=94, desc=91), Event(type=Duration, value=0.6.8, time=94, desc=6 ticks), Event(type=Program, value=0, time=94, desc=95), Event(type=Pitch, value=55, time=94, desc=95), Event(type=Velocity, value=79, time=94, desc=79), Event(type=Duration, value=0.1.8, time=94, desc=1 ticks), Event(type=Program, value=0, time=94, desc=100), Event(type=Pitch, value=71, time=94, desc=100), Event(type=Velocity, value=87, time=94, desc=87), Event(type=Duration, value=0.6.8, time=94, desc=6 ticks), Event(type=Program, value=0, time=94, desc=100), Event(type=Pitch, value=65, time=94, desc=100), Event(type=Velocity, value=91, time=94, desc=91), Event(type=Duration, value=0.6.8, time=94, desc=6 ticks), Event(type=Bar, value=None, time=96, desc=0), Event(type=Position, value=2, time=98, desc=98), Event(type=Program, value=0, time=98, desc=100), Event(type=Pitch, value=74, time=98, desc=100), Event(type=Velocity, value=75, time=98, desc=75), Event(type=Duration, value=0.2.8, time=98, desc=2 ticks), Event(type=Position, value=3, time=99, desc=99), Event(type=Program, value=0, time=99, desc=100), Event(type=Pitch, value=64, time=99, desc=100), Event(type=Velocity, value=91, time=99, desc=91), Event(type=Duration, value=0.1.8, time=99, desc=1 ticks), Event(type=Position, value=4, time=100, desc=100), Event(type=Program, value=0, time=100, desc=101), Event(type=Pitch, value=60, time=100, desc=101), Event(type=Velocity, value=91, time=100, desc=91), Event(type=Duration, value=0.1.8, time=100, desc=1 ticks), Event(type=Program, value=0, time=100, desc=101), Event(type=Pitch, value=72, time=100, desc=101), Event(type=Velocity, value=91, time=100, desc=91), Event(type=Duration, value=0.1.8, time=100, desc=1 ticks), Event(type=Position, value=7, time=103, desc=103), Event(type=Program, value=0, time=103, desc=104), Event(type=Pitch, value=55, time=103, desc=104), Event(type=Velocity, value=87, time=103, desc=87), Event(type=Duration, value=0.1.8, time=103, desc=1 ticks), Event(type=Program, value=0, time=103, desc=104), Event(type=Pitch, value=71, time=103, desc=104), Event(type=Velocity, value=87, time=103, desc=87), Event(type=Duration, value=0.1.8, time=103, desc=1 ticks), Event(type=Program, value=0, time=103, desc=104), Event(type=Pitch, value=65, time=103, desc=104), Event(type=Velocity, value=79, time=103, desc=79), Event(type=Duration, value=0.1.8, time=103, desc=1 ticks), Event(type=Position, value=10, time=106, desc=106), Event(type=Program, value=0, time=106, desc=107), Event(type=Pitch, value=72, time=106, desc=107), Event(type=Velocity, value=91, time=106, desc=91), Event(type=Duration, value=0.1.8, time=106, desc=1 ticks), Event(type=Program, value=0, time=106, desc=107), Event(type=Pitch, value=64, time=106, desc=107), Event(type=Velocity, value=87, time=106, desc=87), Event(type=Duration, value=0.1.8, time=106, desc=1 ticks), Event(type=Program, value=0, time=106, desc=107), Event(type=Pitch, value=48, time=106, desc=107), Event(type=Velocity, value=59, time=106, desc=59), Event(type=Duration, value=0.1.8, time=106, desc=1 ticks), Event(type=Position, value=12, time=108, desc=108), Event(type=Program, value=0, time=108, desc=109), Event(type=Pitch, value=43, time=108, desc=109), Event(type=Velocity, value=67, time=108, desc=67), Event(type=Duration, value=0.1.8, time=108, desc=1 ticks), Event(type=Position, value=16, time=112, desc=112), Event(type=Program, value=0, time=112, desc=117), Event(type=Pitch, value=79, time=112, desc=117), Event(type=Velocity, value=99, time=112, desc=99), Event(type=Duration, value=0.5.8, time=112, desc=5 ticks), Event(type=Program, value=0, time=112, desc=125), Event(type=Pitch, value=36, time=112, desc=125), Event(type=Velocity, value=87, time=112, desc=87), Event(type=Duration, value=1.5.8, time=112, desc=13 ticks), Event(type=Program, value=0, time=112, desc=125), Event(type=Pitch, value=43, time=112, desc=125), Event(type=Velocity, value=87, time=112, desc=87), Event(type=Duration, value=1.5.8, time=112, desc=13 ticks), Event(type=Position, value=17, time=113, desc=113), Event(type=Program, value=0, time=113, desc=119), Event(type=Pitch, value=67, time=113, desc=119), Event(type=Velocity, value=87, time=113, desc=87), Event(type=Duration, value=0.6.8, time=113, desc=6 ticks), Event(type=Position, value=19, time=115, desc=115), Event(type=Program, value=0, time=115, desc=125), Event(type=Pitch, value=72, time=115, desc=125), Event(type=Velocity, value=79, time=115, desc=79), Event(type=Duration, value=1.2.8, time=115, desc=10 ticks), Event(type=Position, value=21, time=117, desc=117), Event(type=Program, value=0, time=117, desc=125), Event(type=Pitch, value=79, time=117, desc=125), Event(type=Velocity, value=87, time=117, desc=87), Event(type=Duration, value=1.0.8, time=117, desc=8 ticks), Event(type=Position, value=22, time=118, desc=118), Event(type=Program, value=0, time=118, desc=125), Event(type=Pitch, value=58, time=118, desc=125), Event(type=Velocity, value=87, time=118, desc=87), Event(type=Duration, value=0.7.8, time=118, desc=7 ticks), Event(type=Program, value=0, time=118, desc=125), Event(type=Pitch, value=61, time=118, desc=125), Event(type=Velocity, value=91, time=118, desc=91), Event(type=Duration, value=0.7.8, time=118, desc=7 ticks), Event(type=Program, value=0, time=118, desc=123), Event(type=Pitch, value=76, time=118, desc=123), Event(type=Velocity, value=91, time=118, desc=91), Event(type=Duration, value=0.5.8, time=118, desc=5 ticks), Event(type=Position, value=23, time=119, desc=119), Event(type=Program, value=0, time=119, desc=124), Event(type=Pitch, value=67, time=119, desc=124), Event(type=Velocity, value=79, time=119, desc=79), Event(type=Duration, value=0.5.8, time=119, desc=5 ticks), Event(type=Position, value=25, time=121, desc=121), Event(type=Program, value=0, time=121, desc=125), Event(type=Pitch, value=73, time=121, desc=125), Event(type=Velocity, value=75, time=121, desc=75), Event(type=Duration, value=0.4.8, time=121, desc=4 ticks), Event(type=Position, value=27, time=123, desc=123), Event(type=Program, value=0, time=123, desc=125), Event(type=Pitch, value=76, time=123, desc=125), Event(type=Velocity, value=79, time=123, desc=79), Event(type=Duration, value=0.2.8, time=123, desc=2 ticks), Event(type=Position, value=28, time=124, desc=124), Event(type=Program, value=0, time=124, desc=136), Event(type=Pitch, value=57, time=124, desc=136), Event(type=Velocity, value=87, time=124, desc=87), Event(type=Duration, value=1.4.8, time=124, desc=12 ticks), Event(type=Program, value=0, time=124, desc=136), Event(type=Pitch, value=60, time=124, desc=136), Event(type=Velocity, value=87, time=124, desc=87), Event(type=Duration, value=1.4.8, time=124, desc=12 ticks), Event(type=Program, value=0, time=124, desc=129), Event(type=Pitch, value=77, time=124, desc=129), Event(type=Velocity, value=91, time=124, desc=91), Event(type=Duration, value=0.5.8, time=124, desc=5 ticks), Event(type=Position, value=30, time=126, desc=126), Event(type=Program, value=0, time=126, desc=132), Event(type=Pitch, value=65, time=126, desc=132), Event(type=Velocity, value=75, time=126, desc=75), Event(type=Duration, value=0.6.8, time=126, desc=6 ticks), Event(type=Bar, value=None, time=128, desc=0), Event(type=Position, value=0, time=128, desc=128), Event(type=Program, value=0, time=128, desc=137), Event(type=Pitch, value=72, time=128, desc=137), Event(type=Velocity, value=75, time=128, desc=75), Event(type=Duration, value=1.1.8, time=128, desc=9 ticks), Event(type=Position, value=1, time=129, desc=129), Event(type=Program, value=0, time=129, desc=137), Event(type=Pitch, value=77, time=129, desc=137), Event(type=Velocity, value=79, time=129, desc=79), Event(type=Duration, value=1.0.8, time=129, desc=8 ticks), Event(type=Position, value=2, time=130, desc=130), Event(type=Program, value=0, time=130, desc=137), Event(type=Pitch, value=59, time=130, desc=137), Event(type=Velocity, value=87, time=130, desc=87), Event(type=Duration, value=0.7.8, time=130, desc=7 ticks), Event(type=Program, value=0, time=130, desc=137), Event(type=Pitch, value=56, time=130, desc=137), Event(type=Velocity, value=75, time=130, desc=75), Event(type=Duration, value=0.7.8, time=130, desc=7 ticks), Event(type=Program, value=0, time=130, desc=135), Event(type=Pitch, value=74, time=130, desc=135), Event(type=Velocity, value=87, time=130, desc=87), Event(type=Duration, value=0.5.8, time=130, desc=5 ticks), Event(type=Position, value=4, time=132, desc=132), Event(type=Program, value=0, time=132, desc=137), Event(type=Pitch, value=65, time=132, desc=137), Event(type=Velocity, value=79, time=132, desc=79), Event(type=Duration, value=0.5.8, time=132, desc=5 ticks), Event(type=Position, value=5, time=133, desc=133), Event(type=Program, value=0, time=133, desc=137), Event(type=Pitch, value=71, time=133, desc=137), Event(type=Velocity, value=79, time=133, desc=79), Event(type=Duration, value=0.4.8, time=133, desc=4 ticks), Event(type=Position, value=7, time=135, desc=135), Event(type=Program, value=0, time=135, desc=137), Event(type=Pitch, value=74, time=135, desc=137), Event(type=Velocity, value=87, time=135, desc=87), Event(type=Duration, value=0.2.8, time=135, desc=2 ticks), Event(type=Position, value=8, time=136, desc=136), Event(type=Program, value=0, time=136, desc=143), Event(type=Pitch, value=58, time=136, desc=143), Event(type=Velocity, value=95, time=136, desc=95), Event(type=Duration, value=0.7.8, time=136, desc=7 ticks), Event(type=Program, value=0, time=136, desc=142), Event(type=Pitch, value=55, time=136, desc=142), Event(type=Velocity, value=79, time=136, desc=79), Event(type=Duration, value=0.6.8, time=136, desc=6 ticks), Event(type=Program, value=0, time=136, desc=141), Event(type=Pitch, value=76, time=136, desc=141), Event(type=Velocity, value=91, time=136, desc=91), Event(type=Duration, value=0.5.8, time=136, desc=5 ticks), Event(type=Position, value=10, time=138, desc=138), Event(type=Program, value=0, time=138, desc=143), Event(type=Pitch, value=64, time=138, desc=143), Event(type=Velocity, value=79, time=138, desc=79), Event(type=Duration, value=0.5.8, time=138, desc=5 ticks), Event(type=Position, value=11, time=139, desc=139), Event(type=Program, value=0, time=139, desc=142), Event(type=Pitch, value=70, time=139, desc=142), Event(type=Velocity, value=79, time=139, desc=79), Event(type=Duration, value=0.3.8, time=139, desc=3 ticks), Event(type=Position, value=13, time=141, desc=141), Event(type=Program, value=0, time=141, desc=143), Event(type=Pitch, value=76, time=141, desc=143), Event(type=Velocity, value=87, time=141, desc=87), Event(type=Duration, value=0.2.8, time=141, desc=2 ticks), Event(type=Position, value=14, time=142, desc=142), Event(type=Program, value=0, time=142, desc=148), Event(type=Pitch, value=54, time=142, desc=148), Event(type=Velocity, value=87, time=142, desc=87), Event(type=Duration, value=0.6.8, time=142, desc=6 ticks), Event(type=Program, value=0, time=142, desc=147), Event(type=Pitch, value=72, time=142, desc=147), Event(type=Velocity, value=95, time=142, desc=95), Event(type=Duration, value=0.5.8, time=142, desc=5 ticks), Event(type=Program, value=0, time=142, desc=148), Event(type=Pitch, value=57, time=142, desc=148), Event(type=Velocity, value=95, time=142, desc=95), Event(type=Duration, value=0.6.8, time=142, desc=6 ticks), Event(type=Position, value=16, time=144, desc=144), Event(type=Program, value=0, time=144, desc=149), Event(type=Pitch, value=63, time=144, desc=149), Event(type=Velocity, value=79, time=144, desc=79), Event(type=Duration, value=0.5.8, time=144, desc=5 ticks), Event(type=Position, value=18, time=146, desc=146), Event(type=Program, value=0, time=146, desc=149), Event(type=Pitch, value=69, time=146, desc=149), Event(type=Velocity, value=75, time=146, desc=75), Event(type=Duration, value=0.3.8, time=146, desc=3 ticks), Event(type=Position, value=19, time=147, desc=147), Event(type=Program, value=0, time=147, desc=149), Event(type=Pitch, value=72, time=147, desc=149), Event(type=Velocity, value=79, time=147, desc=79), Event(type=Duration, value=0.2.8, time=147, desc=2 ticks), Event(type=Position, value=20, time=148, desc=148), Event(type=Program, value=0, time=148, desc=154), Event(type=Pitch, value=56, time=148, desc=154), Event(type=Velocity, value=87, time=148, desc=87), Event(type=Duration, value=0.6.8, time=148, desc=6 ticks), Event(type=Program, value=0, time=148, desc=152), Event(type=Pitch, value=74, time=148, desc=152), Event(type=Velocity, value=87, time=148, desc=87), Event(type=Duration, value=0.4.8, time=148, desc=4 ticks), Event(type=Position, value=21, time=149, desc=149), Event(type=Program, value=0, time=149, desc=155), Event(type=Pitch, value=53, time=149, desc=155), Event(type=Velocity, value=71, time=149, desc=71), Event(type=Duration, value=0.6.8, time=149, desc=6 ticks), Event(type=Program, value=0, time=149, desc=154), Event(type=Pitch, value=62, time=149, desc=154), Event(type=Velocity, value=87, time=149, desc=87), Event(type=Duration, value=0.5.8, time=149, desc=5 ticks), Event(type=Position, value=23, time=151, desc=151), Event(type=Program, value=0, time=151, desc=155), Event(type=Pitch, value=68, time=151, desc=155), Event(type=Velocity, value=79, time=151, desc=79), Event(type=Duration, value=0.4.8, time=151, desc=4 ticks), Event(type=Position, value=25, time=153, desc=153), Event(type=Program, value=0, time=153, desc=155), Event(type=Pitch, value=74, time=153, desc=155), Event(type=Velocity, value=79, time=153, desc=79), Event(type=Duration, value=0.2.8, time=153, desc=2 ticks), Event(type=Position, value=26, time=154, desc=154), Event(type=Program, value=0, time=154, desc=161), Event(type=Pitch, value=55, time=154, desc=161), Event(type=Velocity, value=71, time=154, desc=71), Event(type=Duration, value=0.7.8, time=154, desc=7 ticks), Event(type=Program, value=0, time=154, desc=158), Event(type=Pitch, value=67, time=154, desc=158), Event(type=Velocity, value=91, time=154, desc=91), Event(type=Duration, value=0.4.8, time=154, desc=4 ticks), Event(type=Position, value=27, time=155, desc=155), Event(type=Program, value=0, time=155, desc=161), Event(type=Pitch, value=59, time=155, desc=161), Event(type=Velocity, value=87, time=155, desc=87), Event(type=Duration, value=0.6.8, time=155, desc=6 ticks), Event(type=Position, value=28, time=156, desc=156), Event(type=Program, value=0, time=156, desc=161), Event(type=Pitch, value=62, time=156, desc=161), Event(type=Velocity, value=91, time=156, desc=91), Event(type=Duration, value=0.5.8, time=156, desc=5 ticks), Event(type=Position, value=29, time=157, desc=157), Event(type=Program, value=0, time=157, desc=161), Event(type=Pitch, value=65, time=157, desc=161), Event(type=Velocity, value=87, time=157, desc=87), Event(type=Duration, value=0.4.8, time=157, desc=4 ticks), Event(type=Position, value=30, time=158, desc=158), Event(type=Program, value=0, time=158, desc=161), Event(type=Pitch, value=67, time=158, desc=161), Event(type=Velocity, value=87, time=158, desc=87), Event(type=Duration, value=0.3.8, time=158, desc=3 ticks), Event(type=Bar, value=None, time=160, desc=0), Event(type=Position, value=0, time=160, desc=160), Event(type=Program, value=0, time=160, desc=161), Event(type=Pitch, value=43, time=160, desc=161), Event(type=Velocity, value=99, time=160, desc=99), Event(type=Duration, value=0.1.8, time=160, desc=1 ticks), Event(type=Program, value=0, time=160, desc=161), Event(type=Pitch, value=36, time=160, desc=161), Event(type=Velocity, value=91, time=160, desc=91), Event(type=Duration, value=0.1.8, time=160, desc=1 ticks), Event(type=Position, value=1, time=161, desc=161), Event(type=Program, value=0, time=161, desc=166), Event(type=Pitch, value=79, time=161, desc=166), Event(type=Velocity, value=99, time=161, desc=99), Event(type=Duration, value=0.5.8, time=161, desc=5 ticks), Event(type=Position, value=2, time=162, desc=162), Event(type=Program, value=0, time=162, desc=167), Event(type=Pitch, value=67, time=162, desc=167), Event(type=Velocity, value=87, time=162, desc=87), Event(type=Duration, value=0.5.8, time=162, desc=5 ticks), Event(type=Position, value=4, time=164, desc=164), Event(type=Program, value=0, time=164, desc=168), Event(type=Pitch, value=72, time=164, desc=168), Event(type=Velocity, value=87, time=164, desc=87), Event(type=Duration, value=0.4.8, time=164, desc=4 ticks), Event(type=Position, value=5, time=165, desc=165), Event(type=Program, value=0, time=165, desc=173), Event(type=Pitch, value=79, time=165, desc=173), Event(type=Velocity, value=87, time=165, desc=87), Event(type=Duration, value=1.0.8, time=165, desc=8 ticks), Event(type=Position, value=7, time=167, desc=167), Event(type=Program, value=0, time=167, desc=174), Event(type=Pitch, value=58, time=167, desc=174), Event(type=Velocity, value=91, time=167, desc=91), Event(type=Duration, value=0.7.8, time=167, desc=7 ticks), Event(type=Program, value=0, time=167, desc=172), Event(type=Pitch, value=76, time=167, desc=172), Event(type=Velocity, value=95, time=167, desc=95), Event(type=Duration, value=0.5.8, time=167, desc=5 ticks), Event(type=Program, value=0, time=167, desc=174), Event(type=Pitch, value=61, time=167, desc=174), Event(type=Velocity, value=95, time=167, desc=95), Event(type=Duration, value=0.7.8, time=167, desc=7 ticks), Event(type=Position, value=8, time=168, desc=168), Event(type=Program, value=0, time=168, desc=173), Event(type=Pitch, value=67, time=168, desc=173), Event(type=Velocity, value=75, time=168, desc=75), Event(type=Duration, value=0.5.8, time=168, desc=5 ticks), Event(type=Position, value=10, time=170, desc=170), Event(type=Program, value=0, time=170, desc=174), Event(type=Pitch, value=73, time=170, desc=174), Event(type=Velocity, value=79, time=170, desc=79), Event(type=Duration, value=0.4.8, time=170, desc=4 ticks), Event(type=Position, value=12, time=172, desc=172), Event(type=Program, value=0, time=172, desc=174), Event(type=Pitch, value=76, time=172, desc=174), Event(type=Velocity, value=79, time=172, desc=79), Event(type=Duration, value=0.2.8, time=172, desc=2 ticks), Event(type=Position, value=13, time=173, desc=173), Event(type=Program, value=0, time=173, desc=179), Event(type=Pitch, value=60, time=173, desc=179), Event(type=Velocity, value=87, time=173, desc=87), Event(type=Duration, value=0.6.8, time=173, desc=6 ticks), Event(type=Program, value=0, time=173, desc=179), Event(type=Pitch, value=57, time=173, desc=179), Event(type=Velocity, value=91, time=173, desc=91), Event(type=Duration, value=0.6.8, time=173, desc=6 ticks), Event(type=Program, value=0, time=173, desc=177), Event(type=Pitch, value=77, time=173, desc=177), Event(type=Velocity, value=95, time=173, desc=95), Event(type=Duration, value=0.4.8, time=173, desc=4 ticks), Event(type=Position, value=14, time=174, desc=174), Event(type=Program, value=0, time=174, desc=179), Event(type=Pitch, value=65, time=174, desc=179), Event(type=Velocity, value=75, time=174, desc=75), Event(type=Duration, value=0.5.8, time=174, desc=5 ticks), Event(type=Position, value=16, time=176, desc=176), Event(type=Program, value=0, time=176, desc=180), Event(type=Pitch, value=72, time=176, desc=180), Event(type=Velocity, value=75, time=176, desc=75), Event(type=Duration, value=0.4.8, time=176, desc=4 ticks), Event(type=Position, value=17, time=177, desc=177), Event(type=Program, value=0, time=177, desc=179), Event(type=Pitch, value=77, time=177, desc=179), Event(type=Velocity, value=87, time=177, desc=87), Event(type=Duration, value=0.2.8, time=177, desc=2 ticks), Event(type=Position, value=19, time=179, desc=179), Event(type=Program, value=0, time=179, desc=184), Event(type=Pitch, value=74, time=179, desc=184), Event(type=Velocity, value=87, time=179, desc=87), Event(type=Duration, value=0.5.8, time=179, desc=5 ticks), Event(type=Program, value=0, time=179, desc=186), Event(type=Pitch, value=59, time=179, desc=186), Event(type=Velocity, value=87, time=179, desc=87), Event(type=Duration, value=0.7.8, time=179, desc=7 ticks), Event(type=Program, value=0, time=179, desc=185), Event(type=Pitch, value=56, time=179, desc=185), Event(type=Velocity, value=75, time=179, desc=75), Event(type=Duration, value=0.6.8, time=179, desc=6 ticks), Event(type=Position, value=20, time=180, desc=180), Event(type=Program, value=0, time=180, desc=185), Event(type=Pitch, value=65, time=180, desc=185), Event(type=Velocity, value=75, time=180, desc=75), Event(type=Duration, value=0.5.8, time=180, desc=5 ticks), Event(type=Position, value=21, time=181, desc=181), Event(type=Program, value=0, time=181, desc=185), Event(type=Pitch, value=71, time=181, desc=185), Event(type=Velocity, value=79, time=181, desc=79), Event(type=Duration, value=0.4.8, time=181, desc=4 ticks), Event(type=Position, value=24, time=184, desc=184), Event(type=Program, value=0, time=184, desc=186), Event(type=Pitch, value=74, time=184, desc=186), Event(type=Velocity, value=79, time=184, desc=79), Event(type=Duration, value=0.2.8, time=184, desc=2 ticks), Event(type=Position, value=25, time=185, desc=185), Event(type=Program, value=0, time=185, desc=191), Event(type=Pitch, value=55, time=185, desc=191), Event(type=Velocity, value=75, time=185, desc=75), Event(type=Duration, value=0.6.8, time=185, desc=6 ticks), Event(type=Program, value=0, time=185, desc=191), Event(type=Pitch, value=60, time=185, desc=191), Event(type=Velocity, value=79, time=185, desc=79), Event(type=Duration, value=0.6.8, time=185, desc=6 ticks), Event(type=Program, value=0, time=185, desc=191), Event(type=Pitch, value=76, time=185, desc=191), Event(type=Velocity, value=87, time=185, desc=87), Event(type=Duration, value=0.6.8, time=185, desc=6 ticks), Event(type=Position, value=26, time=186, desc=186), Event(type=Program, value=0, time=186, desc=191), Event(type=Pitch, value=64, time=186, desc=191), Event(type=Velocity, value=75, time=186, desc=75), Event(type=Duration, value=0.5.8, time=186, desc=5 ticks), Event(type=Position, value=27, time=187, desc=187), Event(type=Program, value=0, time=187, desc=191), Event(type=Pitch, value=67, time=187, desc=191), Event(type=Velocity, value=87, time=187, desc=87), Event(type=Duration, value=0.4.8, time=187, desc=4 ticks), Event(type=Position, value=29, time=189, desc=189), Event(type=Program, value=0, time=189, desc=191), Event(type=Pitch, value=72, time=189, desc=191), Event(type=Velocity, value=87, time=189, desc=87), Event(type=Duration, value=0.2.8, time=189, desc=2 ticks), Event(type=Position, value=30, time=190, desc=190), Event(type=Program, value=0, time=190, desc=196), Event(type=Pitch, value=71, time=190, desc=196), Event(type=Velocity, value=87, time=190, desc=87), Event(type=Duration, value=0.6.8, time=190, desc=6 ticks), Event(type=Position, value=31, time=191, desc=191), Event(type=Program, value=0, time=191, desc=197), Event(type=Pitch, value=55, time=191, desc=197), Event(type=Velocity, value=75, time=191, desc=75), Event(type=Duration, value=0.6.8, time=191, desc=6 ticks), Event(type=Program, value=0, time=191, desc=197), Event(type=Pitch, value=62, time=191, desc=197), Event(type=Velocity, value=91, time=191, desc=91), Event(type=Duration, value=0.6.8, time=191, desc=6 ticks), Event(type=Bar, value=None, time=192, desc=0), Event(type=Position, value=0, time=192, desc=192), Event(type=Program, value=0, time=192, desc=197), Event(type=Pitch, value=65, time=192, desc=197), Event(type=Velocity, value=79, time=192, desc=79), Event(type=Duration, value=0.5.8, time=192, desc=5 ticks), Event(type=Position, value=1, time=193, desc=193), Event(type=Program, value=0, time=193, desc=197), Event(type=Pitch, value=67, time=193, desc=197), Event(type=Velocity, value=87, time=193, desc=87), Event(type=Duration, value=0.4.8, time=193, desc=4 ticks), Event(type=Position, value=3, time=195, desc=195), Event(type=Program, value=0, time=195, desc=197), Event(type=Pitch, value=74, time=195, desc=197), Event(type=Velocity, value=87, time=195, desc=87), Event(type=Duration, value=0.2.8, time=195, desc=2 ticks), Event(type=Position, value=4, time=196, desc=196), Event(type=Program, value=0, time=196, desc=197), Event(type=Pitch, value=72, time=196, desc=197), Event(type=Velocity, value=87, time=196, desc=87), Event(type=Duration, value=0.1.8, time=196, desc=1 ticks), Event(type=Program, value=0, time=196, desc=197), Event(type=Pitch, value=60, time=196, desc=197), Event(type=Velocity, value=75, time=196, desc=75), Event(type=Duration, value=0.1.8, time=196, desc=1 ticks), Event(type=Position, value=5, time=197, desc=197), Event(type=Program, value=0, time=197, desc=198), Event(type=Pitch, value=64, time=197, desc=198), Event(type=Velocity, value=71, time=197, desc=71), Event(type=Duration, value=0.1.8, time=197, desc=1 ticks), Event(type=Position, value=7, time=199, desc=199), Event(type=Program, value=0, time=199, desc=200), Event(type=Pitch, value=55, time=199, desc=200), Event(type=Velocity, value=71, time=199, desc=71), Event(type=Duration, value=0.1.8, time=199, desc=1 ticks), Event(type=Program, value=0, time=199, desc=200), Event(type=Pitch, value=65, time=199, desc=200), Event(type=Velocity, value=79, time=199, desc=79), Event(type=Duration, value=0.1.8, time=199, desc=1 ticks), Event(type=Position, value=8, time=200, desc=200), Event(type=Program, value=0, time=200, desc=202), Event(type=Pitch, value=71, time=200, desc=202), Event(type=Velocity, value=71, time=200, desc=71), Event(type=Duration, value=0.2.8, time=200, desc=2 ticks), Event(type=Position, value=10, time=202, desc=202), Event(type=Program, value=0, time=202, desc=204), Event(type=Pitch, value=72, time=202, desc=204), Event(type=Velocity, value=91, time=202, desc=91), Event(type=Duration, value=0.2.8, time=202, desc=2 ticks), Event(type=Program, value=0, time=202, desc=203), Event(type=Pitch, value=64, time=202, desc=203), Event(type=Velocity, value=91, time=202, desc=91), Event(type=Duration, value=0.1.8, time=202, desc=1 ticks), Event(type=Program, value=0, time=202, desc=203), Event(type=Pitch, value=48, time=202, desc=203), Event(type=Velocity, value=79, time=202, desc=79), Event(type=Duration, value=0.1.8, time=202, desc=1 ticks), Event(type=Position, value=13, time=205, desc=205), Event(type=Program, value=0, time=205, desc=206), Event(type=Pitch, value=43, time=205, desc=206), Event(type=Velocity, value=67, time=205, desc=67), Event(type=Duration, value=0.1.8, time=205, desc=1 ticks), Event(type=Position, value=16, time=208, desc=208), Event(type=Program, value=0, time=208, desc=215), Event(type=Pitch, value=64, time=208, desc=215), Event(type=Velocity, value=99, time=208, desc=99), Event(type=Duration, value=0.7.8, time=208, desc=7 ticks), Event(type=Program, value=0, time=208, desc=213), Event(type=Pitch, value=76, time=208, desc=213), Event(type=Velocity, value=95, time=208, desc=95), Event(type=Duration, value=0.5.8, time=208, desc=5 ticks), Event(type=Program, value=0, time=208, desc=215), Event(type=Pitch, value=67, time=208, desc=215), Event(type=Velocity, value=99, time=208, desc=99), Event(type=Duration, value=0.7.8, time=208, desc=7 ticks), Event(type=Program, value=0, time=208, desc=215), Event(type=Pitch, value=36, time=208, desc=215), Event(type=Velocity, value=79, time=208, desc=79), Event(type=Duration, value=0.7.8, time=208, desc=7 ticks), Event(type=Program, value=0, time=208, desc=215), Event(type=Pitch, value=43, time=208, desc=215), Event(type=Velocity, value=79, time=208, desc=79), Event(type=Duration, value=0.7.8, time=208, desc=7 ticks), Event(type=Position, value=21, time=213, desc=213), Event(type=Program, value=0, time=213, desc=221), Event(type=Pitch, value=76, time=213, desc=221), Event(type=Velocity, value=79, time=213, desc=79), Event(type=Duration, value=1.0.8, time=213, desc=8 ticks), Event(type=Position, value=23, time=215, desc=215), Event(type=Program, value=0, time=215, desc=222), Event(type=Pitch, value=64, time=215, desc=222), Event(type=Velocity, value=87, time=215, desc=87), Event(type=Duration, value=0.7.8, time=215, desc=7 ticks), Event(type=Program, value=0, time=215, desc=222), Event(type=Pitch, value=67, time=215, desc=222), Event(type=Velocity, value=91, time=215, desc=91), Event(type=Duration, value=0.7.8, time=215, desc=7 ticks), Event(type=Program, value=0, time=215, desc=221), Event(type=Pitch, value=60, time=215, desc=221), Event(type=Velocity, value=71, time=215, desc=71), Event(type=Duration, value=0.6.8, time=215, desc=6 ticks), Event(type=Program, value=0, time=215, desc=221), Event(type=Pitch, value=58, time=215, desc=221), Event(type=Velocity, value=79, time=215, desc=79), Event(type=Duration, value=0.6.8, time=215, desc=6 ticks), Event(type=Program, value=0, time=215, desc=219), Event(type=Pitch, value=72, time=215, desc=219), Event(type=Velocity, value=87, time=215, desc=87), Event(type=Duration, value=0.4.8, time=215, desc=4 ticks), Event(type=Position, value=27, time=219, desc=219), Event(type=Program, value=0, time=219, desc=221), Event(type=Pitch, value=72, time=219, desc=221), Event(type=Velocity, value=87, time=219, desc=87), Event(type=Duration, value=0.2.8, time=219, desc=2 ticks), Event(type=Position, value=29, time=221, desc=221), Event(type=Program, value=0, time=221, desc=227), Event(type=Pitch, value=60, time=221, desc=227), Event(type=Velocity, value=87, time=221, desc=87), Event(type=Duration, value=0.6.8, time=221, desc=6 ticks), Event(type=Program, value=0, time=221, desc=227), Event(type=Pitch, value=57, time=221, desc=227), Event(type=Velocity, value=87, time=221, desc=87), Event(type=Duration, value=0.6.8, time=221, desc=6 ticks), Event(type=Program, value=0, time=221, desc=227), Event(type=Pitch, value=72, time=221, desc=227), Event(type=Velocity, value=91, time=221, desc=91), Event(type=Duration, value=0.6.8, time=221, desc=6 ticks), Event(type=Program, value=0, time=221, desc=226), Event(type=Pitch, value=77, time=221, desc=226), Event(type=Velocity, value=95, time=221, desc=95), Event(type=Duration, value=0.5.8, time=221, desc=5 ticks), Event(type=Program, value=0, time=221, desc=227), Event(type=Pitch, value=65, time=221, desc=227), Event(type=Velocity, value=95, time=221, desc=95), Event(type=Duration, value=0.6.8, time=221, desc=6 ticks), Event(type=Bar, value=None, time=224, desc=0), Event(type=Position, value=1, time=225, desc=225), Event(type=Program, value=0, time=225, desc=227), Event(type=Pitch, value=77, time=225, desc=227), Event(type=Velocity, value=79, time=225, desc=79), Event(type=Duration, value=0.2.8, time=225, desc=2 ticks), Event(type=Position, value=3, time=227, desc=227), Event(type=Program, value=0, time=227, desc=234), Event(type=Pitch, value=71, time=227, desc=234), Event(type=Velocity, value=91, time=227, desc=91), Event(type=Duration, value=0.7.8, time=227, desc=7 ticks), Event(type=Program, value=0, time=227, desc=233), Event(type=Pitch, value=56, time=227, desc=233), Event(type=Velocity, value=79, time=227, desc=79), Event(type=Duration, value=0.6.8, time=227, desc=6 ticks), Event(type=Program, value=0, time=227, desc=233), Event(type=Pitch, value=59, time=227, desc=233), Event(type=Velocity, value=79, time=227, desc=79), Event(type=Duration, value=0.6.8, time=227, desc=6 ticks), Event(type=Program, value=0, time=227, desc=232), Event(type=Pitch, value=74, time=227, desc=232), Event(type=Velocity, value=87, time=227, desc=87), Event(type=Duration, value=0.5.8, time=227, desc=5 ticks), Event(type=Program, value=0, time=227, desc=233), Event(type=Pitch, value=65, time=227, desc=233), Event(type=Velocity, value=91, time=227, desc=91), Event(type=Duration, value=0.6.8, time=227, desc=6 ticks), Event(type=Position, value=8, time=232, desc=232), Event(type=Program, value=0, time=232, desc=234), Event(type=Pitch, value=74, time=232, desc=234), Event(type=Velocity, value=79, time=232, desc=79), Event(type=Duration, value=0.2.8, time=232, desc=2 ticks), Event(type=Position, value=9, time=233, desc=233), Event(type=Program, value=0, time=233, desc=234), Event(type=Pitch, value=36, time=233, desc=234), Event(type=Velocity, value=91, time=233, desc=91), Event(type=Duration, value=0.1.8, time=233, desc=1 ticks), Event(type=Program, value=0, time=233, desc=234), Event(type=Pitch, value=43, time=233, desc=234), Event(type=Velocity, value=95, time=233, desc=95), Event(type=Duration, value=0.1.8, time=233, desc=1 ticks), Event(type=Program, value=0, time=233, desc=238), Event(type=Pitch, value=76, time=233, desc=238), Event(type=Velocity, value=95, time=233, desc=95), Event(type=Duration, value=0.5.8, time=233, desc=5 ticks), Event(type=Program, value=0, time=233, desc=240), Event(type=Pitch, value=64, time=233, desc=240), Event(type=Velocity, value=99, time=233, desc=99), Event(type=Duration, value=0.7.8, time=233, desc=7 ticks), Event(type=Program, value=0, time=233, desc=239), Event(type=Pitch, value=67, time=233, desc=239), Event(type=Velocity, value=95, time=233, desc=95), Event(type=Duration, value=0.6.8, time=233, desc=6 ticks), Event(type=Position, value=13, time=237, desc=237), Event(type=Program, value=0, time=237, desc=239), Event(type=Pitch, value=76, time=237, desc=239), Event(type=Velocity, value=79, time=237, desc=79), Event(type=Duration, value=0.2.8, time=237, desc=2 ticks), Event(type=Position, value=15, time=239, desc=239), Event(type=Program, value=0, time=239, desc=246), Event(type=Pitch, value=67, time=239, desc=246), Event(type=Velocity, value=91, time=239, desc=91), Event(type=Duration, value=0.7.8, time=239, desc=7 ticks), Event(type=Program, value=0, time=239, desc=243), Event(type=Pitch, value=72, time=239, desc=243), Event(type=Velocity, value=95, time=239, desc=95), Event(type=Duration, value=0.4.8, time=239, desc=4 ticks), Event(type=Program, value=0, time=239, desc=245), Event(type=Pitch, value=63, time=239, desc=245), Event(type=Velocity, value=87, time=239, desc=87), Event(type=Duration, value=0.6.8, time=239, desc=6 ticks), Event(type=Program, value=0, time=239, desc=245), Event(type=Pitch, value=57, time=239, desc=245), Event(type=Velocity, value=95, time=239, desc=95), Event(type=Duration, value=0.6.8, time=239, desc=6 ticks), Event(type=Position, value=19, time=243, desc=243), Event(type=Program, value=0, time=243, desc=245), Event(type=Pitch, value=72, time=243, desc=245), Event(type=Velocity, value=79, time=243, desc=79), Event(type=Duration, value=0.2.8, time=243, desc=2 ticks), Event(type=Position, value=21, time=245, desc=245), Event(type=Program, value=0, time=245, desc=251), Event(type=Pitch, value=62, time=245, desc=251), Event(type=Velocity, value=91, time=245, desc=91), Event(type=Duration, value=0.6.8, time=245, desc=6 ticks), Event(type=Program, value=0, time=245, desc=252), Event(type=Pitch, value=66, time=245, desc=252), Event(type=Velocity, value=91, time=245, desc=91), Event(type=Duration, value=0.7.8, time=245, desc=7 ticks), Event(type=Program, value=0, time=245, desc=252), Event(type=Pitch, value=58, time=245, desc=252), Event(type=Velocity, value=95, time=245, desc=95), Event(type=Duration, value=0.7.8, time=245, desc=7 ticks), Event(type=Program, value=0, time=245, desc=251), Event(type=Pitch, value=55, time=245, desc=251), Event(type=Velocity, value=79, time=245, desc=79), Event(type=Duration, value=0.6.8, time=245, desc=6 ticks), Event(type=Program, value=0, time=245, desc=250), Event(type=Pitch, value=74, time=245, desc=250), Event(type=Velocity, value=87, time=245, desc=87), Event(type=Duration, value=0.5.8, time=245, desc=5 ticks), Event(type=Position, value=26, time=250, desc=250), Event(type=Program, value=0, time=250, desc=252), Event(type=Pitch, value=74, time=250, desc=252), Event(type=Velocity, value=79, time=250, desc=79), Event(type=Duration, value=0.2.8, time=250, desc=2 ticks), Event(type=Position, value=27, time=251, desc=251), Event(type=Program, value=0, time=251, desc=255), Event(type=Pitch, value=67, time=251, desc=255), Event(type=Velocity, value=91, time=251, desc=91), Event(type=Duration, value=0.4.8, time=251, desc=4 ticks), Event(type=Program, value=0, time=251, desc=257), Event(type=Pitch, value=65, time=251, desc=257), Event(type=Velocity, value=91, time=251, desc=91), Event(type=Duration, value=0.6.8, time=251, desc=6 ticks), Event(type=Program, value=0, time=251, desc=252), Event(type=Pitch, value=55, time=251, desc=252), Event(type=Velocity, value=79, time=251, desc=79), Event(type=Duration, value=0.1.8, time=251, desc=1 ticks), Event(type=Program, value=0, time=251, desc=252), Event(type=Pitch, value=59, time=251, desc=252), Event(type=Velocity, value=91, time=251, desc=91), Event(type=Duration, value=0.1.8, time=251, desc=1 ticks), Event(type=Program, value=0, time=251, desc=257), Event(type=Pitch, value=62, time=251, desc=257), Event(type=Velocity, value=91, time=251, desc=91), Event(type=Duration, value=0.6.8, time=251, desc=6 ticks), Event(type=Position, value=31, time=255, desc=255), Event(type=Program, value=0, time=255, desc=256), Event(type=Pitch, value=67, time=255, desc=256), Event(type=Velocity, value=91, time=255, desc=91), Event(type=Duration, value=0.1.8, time=255, desc=1 ticks), Event(type=Bar, value=None, time=256, desc=0), Event(type=Position, value=1, time=257, desc=257), Event(type=Program, value=0, time=257, desc=261), Event(type=Pitch, value=76, time=257, desc=261), Event(type=Velocity, value=95, time=257, desc=95), Event(type=Duration, value=0.4.8, time=257, desc=4 ticks), Event(type=Program, value=0, time=257, desc=263), Event(type=Pitch, value=67, time=257, desc=263), Event(type=Velocity, value=95, time=257, desc=95), Event(type=Duration, value=0.6.8, time=257, desc=6 ticks), Event(type=Program, value=0, time=257, desc=264), Event(type=Pitch, value=43, time=257, desc=264), Event(type=Velocity, value=95, time=257, desc=95), Event(type=Duration, value=0.7.8, time=257, desc=7 ticks), Event(type=Program, value=0, time=257, desc=263), Event(type=Pitch, value=64, time=257, desc=263), Event(type=Velocity, value=99, time=257, desc=99), Event(type=Duration, value=0.6.8, time=257, desc=6 ticks), Event(type=Program, value=0, time=257, desc=263), Event(type=Pitch, value=36, time=257, desc=263), Event(type=Velocity, value=91, time=257, desc=91), Event(type=Duration, value=0.6.8, time=257, desc=6 ticks), Event(type=Position, value=6, time=262, desc=262), Event(type=Program, value=0, time=262, desc=270), Event(type=Pitch, value=76, time=262, desc=270), Event(type=Velocity, value=87, time=262, desc=87), Event(type=Duration, value=1.0.8, time=262, desc=8 ticks), Event(type=Position, value=7, time=263, desc=263), Event(type=Program, value=0, time=263, desc=267), Event(type=Pitch, value=72, time=263, desc=267), Event(type=Velocity, value=91, time=263, desc=91), Event(type=Duration, value=0.4.8, time=263, desc=4 ticks), Event(type=Program, value=0, time=263, desc=269), Event(type=Pitch, value=67, time=263, desc=269), Event(type=Velocity, value=95, time=263, desc=95), Event(type=Duration, value=0.6.8, time=263, desc=6 ticks), Event(type=Position, value=8, time=264, desc=264), Event(type=Program, value=0, time=264, desc=270), Event(type=Pitch, value=60, time=264, desc=270), Event(type=Velocity, value=79, time=264, desc=79), Event(type=Duration, value=0.6.8, time=264, desc=6 ticks), Event(type=Program, value=0, time=264, desc=270), Event(type=Pitch, value=58, time=264, desc=270), Event(type=Velocity, value=79, time=264, desc=79), Event(type=Duration, value=0.6.8, time=264, desc=6 ticks), Event(type=Program, value=0, time=264, desc=270), Event(type=Pitch, value=64, time=264, desc=270), Event(type=Velocity, value=91, time=264, desc=91), Event(type=Duration, value=0.6.8, time=264, desc=6 ticks), Event(type=Position, value=11, time=267, desc=267), Event(type=Program, value=0, time=267, desc=275), Event(type=Pitch, value=72, time=267, desc=275), Event(type=Velocity, value=91, time=267, desc=91), Event(type=Duration, value=1.0.8, time=267, desc=8 ticks), Event(type=Position, value=13, time=269, desc=269), Event(type=Program, value=0, time=269, desc=275), Event(type=Pitch, value=60, time=269, desc=275), Event(type=Velocity, value=91, time=269, desc=91), Event(type=Duration, value=0.6.8, time=269, desc=6 ticks), Event(type=Program, value=0, time=269, desc=275), Event(type=Pitch, value=57, time=269, desc=275), Event(type=Velocity, value=91, time=269, desc=91), Event(type=Duration, value=0.6.8, time=269, desc=6 ticks), Event(type=Program, value=0, time=269, desc=273), Event(type=Pitch, value=77, time=269, desc=273), Event(type=Velocity, value=95, time=269, desc=95), Event(type=Duration, value=0.4.8, time=269, desc=4 ticks), Event(type=Program, value=0, time=269, desc=275), Event(type=Pitch, value=65, time=269, desc=275), Event(type=Velocity, value=95, time=269, desc=95), Event(type=Duration, value=0.6.8, time=269, desc=6 ticks), Event(type=Position, value=18, time=274, desc=274), Event(type=Program, value=0, time=274, desc=276), Event(type=Pitch, value=77, time=274, desc=276), Event(type=Velocity, value=87, time=274, desc=87), Event(type=Duration, value=0.2.8, time=274, desc=2 ticks), Event(type=Position, value=19, time=275, desc=275), Event(type=Program, value=0, time=275, desc=282), Event(type=Pitch, value=71, time=275, desc=282), Event(type=Velocity, value=87, time=275, desc=87), Event(type=Duration, value=0.7.8, time=275, desc=7 ticks), Event(type=Program, value=0, time=275, desc=282), Event(type=Pitch, value=59, time=275, desc=282), Event(type=Velocity, value=87, time=275, desc=87), Event(type=Duration, value=0.7.8, time=275, desc=7 ticks), Event(type=Program, value=0, time=275, desc=281), Event(type=Pitch, value=56, time=275, desc=281), Event(type=Velocity, value=75, time=275, desc=75), Event(type=Duration, value=0.6.8, time=275, desc=6 ticks), Event(type=Program, value=0, time=275, desc=279), Event(type=Pitch, value=74, time=275, desc=279), Event(type=Velocity, value=87, time=275, desc=87), Event(type=Duration, value=0.4.8, time=275, desc=4 ticks), Event(type=Program, value=0, time=275, desc=281), Event(type=Pitch, value=65, time=275, desc=281), Event(type=Velocity, value=91, time=275, desc=91), Event(type=Duration, value=0.6.8, time=275, desc=6 ticks), Event(type=Position, value=24, time=280, desc=280), Event(type=Program, value=0, time=280, desc=282), Event(type=Pitch, value=74, time=280, desc=282), Event(type=Velocity, value=87, time=280, desc=87), Event(type=Duration, value=0.2.8, time=280, desc=2 ticks), Event(type=Position, value=25, time=281, desc=281), Event(type=Program, value=0, time=281, desc=282), Event(type=Pitch, value=55, time=281, desc=282), Event(type=Velocity, value=79, time=281, desc=79), Event(type=Duration, value=0.1.8, time=281, desc=1 ticks), Event(type=Program, value=0, time=281, desc=282), Event(type=Pitch, value=60, time=281, desc=282), Event(type=Velocity, value=87, time=281, desc=87), Event(type=Duration, value=0.1.8, time=281, desc=1 ticks), Event(type=Program, value=0, time=281, desc=287), Event(type=Pitch, value=76, time=281, desc=287), Event(type=Velocity, value=91, time=281, desc=91), Event(type=Duration, value=0.6.8, time=281, desc=6 ticks), Event(type=Program, value=0, time=281, desc=287), Event(type=Pitch, value=67, time=281, desc=287), Event(type=Velocity, value=95, time=281, desc=95), Event(type=Duration, value=0.6.8, time=281, desc=6 ticks), Event(type=Program, value=0, time=281, desc=287), Event(type=Pitch, value=64, time=281, desc=287), Event(type=Velocity, value=91, time=281, desc=91), Event(type=Duration, value=0.6.8, time=281, desc=6 ticks), Event(type=Position, value=29, time=285, desc=285), Event(type=Program, value=0, time=285, desc=292), Event(type=Pitch, value=72, time=285, desc=292), Event(type=Velocity, value=79, time=285, desc=79), Event(type=Duration, value=0.7.8, time=285, desc=7 ticks), Event(type=Position, value=31, time=287, desc=287), Event(type=Program, value=0, time=287, desc=293), Event(type=Pitch, value=43, time=287, desc=293), Event(type=Velocity, value=87, time=287, desc=87), Event(type=Duration, value=0.6.8, time=287, desc=6 ticks), Event(type=Program, value=0, time=287, desc=293), Event(type=Pitch, value=36, time=287, desc=293), Event(type=Velocity, value=79, time=287, desc=79), Event(type=Duration, value=0.6.8, time=287, desc=6 ticks), Event(type=Program, value=0, time=287, desc=293), Event(type=Pitch, value=71, time=287, desc=293), Event(type=Velocity, value=87, time=287, desc=87), Event(type=Duration, value=0.6.8, time=287, desc=6 ticks), Event(type=Program, value=0, time=287, desc=293), Event(type=Pitch, value=67, time=287, desc=293), Event(type=Velocity, value=91, time=287, desc=91), Event(type=Duration, value=0.6.8, time=287, desc=6 ticks), Event(type=Program, value=0, time=287, desc=293), Event(type=Pitch, value=65, time=287, desc=293), Event(type=Velocity, value=95, time=287, desc=95), Event(type=Duration, value=0.6.8, time=287, desc=6 ticks), Event(type=Bar, value=None, time=288, desc=0), Event(type=Position, value=3, time=291, desc=291), Event(type=Program, value=0, time=291, desc=293), Event(type=Pitch, value=74, time=291, desc=293), Event(type=Velocity, value=79, time=291, desc=79), Event(type=Duration, value=0.2.8, time=291, desc=2 ticks), Event(type=Position, value=5, time=293, desc=293), Event(type=Program, value=0, time=293, desc=294), Event(type=Pitch, value=64, time=293, desc=294), Event(type=Velocity, value=95, time=293, desc=95), Event(type=Duration, value=0.1.8, time=293, desc=1 ticks), Event(type=Program, value=0, time=293, desc=294), Event(type=Pitch, value=72, time=293, desc=294), Event(type=Velocity, value=95, time=293, desc=95), Event(type=Duration, value=0.1.8, time=293, desc=1 ticks), Event(type=Program, value=0, time=293, desc=294), Event(type=Pitch, value=60, time=293, desc=294), Event(type=Velocity, value=91, time=293, desc=91), Event(type=Duration, value=0.1.8, time=293, desc=1 ticks), Event(type=Position, value=8, time=296, desc=296), Event(type=Program, value=0, time=296, desc=297), Event(type=Pitch, value=55, time=296, desc=297), Event(type=Velocity, value=79, time=296, desc=79), Event(type=Duration, value=0.1.8, time=296, desc=1 ticks), Event(type=Program, value=0, time=296, desc=297), Event(type=Pitch, value=71, time=296, desc=297), Event(type=Velocity, value=79, time=296, desc=79), Event(type=Duration, value=0.1.8, time=296, desc=1 ticks), Event(type=Program, value=0, time=296, desc=297), Event(type=Pitch, value=65, time=296, desc=297), Event(type=Velocity, value=87, time=296, desc=87), Event(type=Duration, value=0.1.8, time=296, desc=1 ticks), Event(type=Position, value=11, time=299, desc=299), Event(type=Program, value=0, time=299, desc=300), Event(type=Pitch, value=72, time=299, desc=300), Event(type=Velocity, value=91, time=299, desc=91), Event(type=Duration, value=0.1.8, time=299, desc=1 ticks), Event(type=Program, value=0, time=299, desc=300), Event(type=Pitch, value=48, time=299, desc=300), Event(type=Velocity, value=79, time=299, desc=79), Event(type=Duration, value=0.1.8, time=299, desc=1 ticks), Event(type=Program, value=0, time=299, desc=300), Event(type=Pitch, value=64, time=299, desc=300), Event(type=Velocity, value=87, time=299, desc=87), Event(type=Duration, value=0.1.8, time=299, desc=1 ticks), Event(type=Position, value=14, time=302, desc=302), Event(type=Program, value=0, time=302, desc=303), Event(type=Pitch, value=43, time=302, desc=303), Event(type=Velocity, value=67, time=302, desc=67), Event(type=Duration, value=0.1.8, time=302, desc=1 ticks), Event(type=Position, value=17, time=305, desc=305), Event(type=Program, value=0, time=305, desc=310), Event(type=Pitch, value=76, time=305, desc=310), Event(type=Velocity, value=95, time=305, desc=95), Event(type=Duration, value=0.5.8, time=305, desc=5 ticks), Event(type=Program, value=0, time=305, desc=312), Event(type=Pitch, value=43, time=305, desc=312), Event(type=Velocity, value=87, time=305, desc=87), Event(type=Duration, value=0.7.8, time=305, desc=7 ticks), Event(type=Program, value=0, time=305, desc=312), Event(type=Pitch, value=36, time=305, desc=312), Event(type=Velocity, value=87, time=305, desc=87), Event(type=Duration, value=0.7.8, time=305, desc=7 ticks), Event(type=Position, value=18, time=306, desc=306), Event(type=Program, value=0, time=306, desc=311), Event(type=Pitch, value=64, time=306, desc=311), Event(type=Velocity, value=79, time=306, desc=79), Event(type=Duration, value=0.5.8, time=306, desc=5 ticks), Event(type=Position, value=20, time=308, desc=308), Event(type=Program, value=0, time=308, desc=312), Event(type=Pitch, value=67, time=308, desc=312), Event(type=Velocity, value=75, time=308, desc=75), Event(type=Duration, value=0.4.8, time=308, desc=4 ticks), Event(type=Position, value=22, time=310, desc=310), Event(type=Program, value=0, time=310, desc=318), Event(type=Pitch, value=76, time=310, desc=318), Event(type=Velocity, value=79, time=310, desc=79), Event(type=Duration, value=1.0.8, time=310, desc=8 ticks), Event(type=Position, value=24, time=312, desc=312), Event(type=Program, value=0, time=312, desc=318), Event(type=Pitch, value=58, time=312, desc=318), Event(type=Velocity, value=79, time=312, desc=79), Event(type=Duration, value=0.6.8, time=312, desc=6 ticks), Event(type=Program, value=0, time=312, desc=316), Event(type=Pitch, value=72, time=312, desc=316), Event(type=Velocity, value=95, time=312, desc=95), Event(type=Duration, value=0.4.8, time=312, desc=4 ticks), Event(type=Program, value=0, time=312, desc=318), Event(type=Pitch, value=60, time=312, desc=318), Event(type=Velocity, value=75, time=312, desc=75), Event(type=Duration, value=0.6.8, time=312, desc=6 ticks), Event(type=Position, value=25, time=313, desc=313), Event(type=Program, value=0, time=313, desc=318), Event(type=Pitch, value=64, time=313, desc=318), Event(type=Velocity, value=79, time=313, desc=79), Event(type=Duration, value=0.5.8, time=313, desc=5 ticks), Event(type=Position, value=26, time=314, desc=314), Event(type=Program, value=0, time=314, desc=318), Event(type=Pitch, value=67, time=314, desc=318), Event(type=Velocity, value=79, time=314, desc=79), Event(type=Duration, value=0.4.8, time=314, desc=4 ticks), Event(type=Position, value=28, time=316, desc=316), Event(type=Program, value=0, time=316, desc=318), Event(type=Pitch, value=72, time=316, desc=318), Event(type=Velocity, value=87, time=316, desc=87), Event(type=Duration, value=0.2.8, time=316, desc=2 ticks), Event(type=Position, value=29, time=317, desc=317), Event(type=Program, value=0, time=317, desc=323), Event(type=Pitch, value=57, time=317, desc=323), Event(type=Velocity, value=87, time=317, desc=87), Event(type=Duration, value=0.6.8, time=317, desc=6 ticks), Event(type=Program, value=0, time=317, desc=322), Event(type=Pitch, value=77, time=317, desc=322), Event(type=Velocity, value=95, time=317, desc=95), Event(type=Duration, value=0.5.8, time=317, desc=5 ticks), Event(type=Position, value=30, time=318, desc=318), Event(type=Program, value=0, time=318, desc=324), Event(type=Pitch, value=60, time=318, desc=324), Event(type=Velocity, value=87, time=318, desc=87), Event(type=Duration, value=0.6.8, time=318, desc=6 ticks), Event(type=Position, value=31, time=319, desc=319), Event(type=Program, value=0, time=319, desc=323), Event(type=Pitch, value=65, time=319, desc=323), Event(type=Velocity, value=79, time=319, desc=79), Event(type=Duration, value=0.4.8, time=319, desc=4 ticks), Event(type=Bar, value=None, time=320, desc=0), Event(type=Position, value=0, time=320, desc=320), Event(type=Program, value=0, time=320, desc=323), Event(type=Pitch, value=72, time=320, desc=323), Event(type=Velocity, value=75, time=320, desc=75), Event(type=Duration, value=0.3.8, time=320, desc=3 ticks), Event(type=Position, value=2, time=322, desc=322), Event(type=Program, value=0, time=322, desc=330), Event(type=Pitch, value=77, time=322, desc=330), Event(type=Velocity, value=87, time=322, desc=87), Event(type=Duration, value=1.0.8, time=322, desc=8 ticks), Event(type=Position, value=4, time=324, desc=324), Event(type=Program, value=0, time=324, desc=330), Event(type=Pitch, value=56, time=324, desc=330), Event(type=Velocity, value=75, time=324, desc=75), Event(type=Duration, value=0.6.8, time=324, desc=6 ticks), Event(type=Program, value=0, time=324, desc=330), Event(type=Pitch, value=59, time=324, desc=330), Event(type=Velocity, value=79, time=324, desc=79), Event(type=Duration, value=0.6.8, time=324, desc=6 ticks), Event(type=Program, value=0, time=324, desc=328), Event(type=Pitch, value=74, time=324, desc=328), Event(type=Velocity, value=87, time=324, desc=87), Event(type=Duration, value=0.4.8, time=324, desc=4 ticks), Event(type=Position, value=5, time=325, desc=325), Event(type=Program, value=0, time=325, desc=330), Event(type=Pitch, value=65, time=325, desc=330), Event(type=Velocity, value=87, time=325, desc=87), Event(type=Duration, value=0.5.8, time=325, desc=5 ticks), Event(type=Position, value=6, time=326, desc=326), Event(type=Program, value=0, time=326, desc=330), Event(type=Pitch, value=71, time=326, desc=330), Event(type=Velocity, value=79, time=326, desc=79), Event(type=Duration, value=0.4.8, time=326, desc=4 ticks), Event(type=Position, value=8, time=328, desc=328), Event(type=Program, value=0, time=328, desc=330), Event(type=Pitch, value=74, time=328, desc=330), Event(type=Velocity, value=87, time=328, desc=87), Event(type=Duration, value=0.2.8, time=328, desc=2 ticks), Event(type=Position, value=10, time=330, desc=330), Event(type=Program, value=0, time=330, desc=335), Event(type=Pitch, value=76, time=330, desc=335), Event(type=Velocity, value=91, time=330, desc=91), Event(type=Duration, value=0.5.8, time=330, desc=5 ticks), Event(type=Program, value=0, time=330, desc=337), Event(type=Pitch, value=43, time=330, desc=337), Event(type=Velocity, value=95, time=330, desc=95), Event(type=Duration, value=0.7.8, time=330, desc=7 ticks), Event(type=Program, value=0, time=330, desc=337), Event(type=Pitch, value=36, time=330, desc=337), Event(type=Velocity, value=91, time=330, desc=91), Event(type=Duration, value=0.7.8, time=330, desc=7 ticks), Event(type=Position, value=11, time=331, desc=331), Event(type=Program, value=0, time=331, desc=336), Event(type=Pitch, value=64, time=331, desc=336), Event(type=Velocity, value=79, time=331, desc=79), Event(type=Duration, value=0.5.8, time=331, desc=5 ticks), Event(type=Position, value=13, time=333, desc=333), Event(type=Program, value=0, time=333, desc=336), Event(type=Pitch, value=67, time=333, desc=336), Event(type=Velocity, value=71, time=333, desc=71), Event(type=Duration, value=0.3.8, time=333, desc=3 ticks), Event(type=Position, value=15, time=335, desc=335), Event(type=Program, value=0, time=335, desc=337), Event(type=Pitch, value=76, time=335, desc=337), Event(type=Velocity, value=75, time=335, desc=75), Event(type=Duration, value=0.2.8, time=335, desc=2 ticks), Event(type=Position, value=16, time=336, desc=336), Event(type=Program, value=0, time=336, desc=340), Event(type=Pitch, value=72, time=336, desc=340), Event(type=Velocity, value=91, time=336, desc=91), Event(type=Duration, value=0.4.8, time=336, desc=4 ticks), Event(type=Program, value=0, time=336, desc=342), Event(type=Pitch, value=57, time=336, desc=342), Event(type=Velocity, value=95, time=336, desc=95), Event(type=Duration, value=0.6.8, time=336, desc=6 ticks), Event(type=Position, value=17, time=337, desc=337), Event(type=Program, value=0, time=337, desc=342), Event(type=Pitch, value=63, time=337, desc=342), Event(type=Velocity, value=75, time=337, desc=75), Event(type=Duration, value=0.5.8, time=337, desc=5 ticks), Event(type=Position, value=19, time=339, desc=339), Event(type=Program, value=0, time=339, desc=343), Event(type=Pitch, value=67, time=339, desc=343), Event(type=Velocity, value=79, time=339, desc=79), Event(type=Duration, value=0.4.8, time=339, desc=4 ticks), Event(type=Position, value=20, time=340, desc=340), Event(type=Program, value=0, time=340, desc=342), Event(type=Pitch, value=72, time=340, desc=342), Event(type=Velocity, value=79, time=340, desc=79), Event(type=Duration, value=0.2.8, time=340, desc=2 ticks), Event(type=Position, value=22, time=342, desc=342), Event(type=Program, value=0, time=342, desc=347), Event(type=Pitch, value=74, time=342, desc=347), Event(type=Velocity, value=91, time=342, desc=91), Event(type=Duration, value=0.5.8, time=342, desc=5 ticks), Event(type=Program, value=0, time=342, desc=354), Event(type=Pitch, value=58, time=342, desc=354), Event(type=Velocity, value=91, time=342, desc=91), Event(type=Duration, value=1.4.8, time=342, desc=12 ticks), Event(type=Program, value=0, time=342, desc=348), Event(type=Pitch, value=55, time=342, desc=348), Event(type=Velocity, value=79, time=342, desc=79), Event(type=Duration, value=0.6.8, time=342, desc=6 ticks), Event(type=Position, value=23, time=343, desc=343), Event(type=Program, value=0, time=343, desc=349), Event(type=Pitch, value=62, time=343, desc=349), Event(type=Velocity, value=79, time=343, desc=79), Event(type=Duration, value=0.6.8, time=343, desc=6 ticks), Event(type=Position, value=24, time=344, desc=344), Event(type=Program, value=0, time=344, desc=354), Event(type=Pitch, value=66, time=344, desc=354), Event(type=Velocity, value=87, time=344, desc=87), Event(type=Duration, value=1.2.8, time=344, desc=10 ticks), Event(type=Position, value=26, time=346, desc=346), Event(type=Program, value=0, time=346, desc=354), Event(type=Pitch, value=74, time=346, desc=354), Event(type=Velocity, value=87, time=346, desc=87), Event(type=Duration, value=1.0.8, time=346, desc=8 ticks), Event(type=Position, value=28, time=348, desc=348), Event(type=Program, value=0, time=348, desc=352), Event(type=Pitch, value=67, time=348, desc=352), Event(type=Velocity, value=91, time=348, desc=91), Event(type=Duration, value=0.4.8, time=348, desc=4 ticks), Event(type=Program, value=0, time=348, desc=354), Event(type=Pitch, value=55, time=348, desc=354), Event(type=Velocity, value=87, time=348, desc=87), Event(type=Duration, value=0.6.8, time=348, desc=6 ticks), Event(type=Program, value=0, time=348, desc=354), Event(type=Pitch, value=59, time=348, desc=354), Event(type=Velocity, value=91, time=348, desc=91), Event(type=Duration, value=0.6.8, time=348, desc=6 ticks), Event(type=Position, value=29, time=349, desc=349), Event(type=Program, value=0, time=349, desc=354), Event(type=Pitch, value=62, time=349, desc=354), Event(type=Velocity, value=87, time=349, desc=87), Event(type=Duration, value=0.5.8, time=349, desc=5 ticks), Event(type=Position, value=31, time=351, desc=351), Event(type=Program, value=0, time=351, desc=354), Event(type=Pitch, value=65, time=351, desc=354), Event(type=Velocity, value=87, time=351, desc=87), Event(type=Duration, value=0.3.8, time=351, desc=3 ticks), Event(type=Bar, value=None, time=352, desc=0), Event(type=Position, value=0, time=352, desc=352), Event(type=Program, value=0, time=352, desc=354), Event(type=Pitch, value=67, time=352, desc=354), Event(type=Velocity, value=79, time=352, desc=79), Event(type=Duration, value=0.2.8, time=352, desc=2 ticks), Event(type=Position, value=2, time=354, desc=354), Event(type=Program, value=0, time=354, desc=361), Event(type=Pitch, value=36, time=354, desc=361), Event(type=Velocity, value=91, time=354, desc=91), Event(type=Duration, value=0.7.8, time=354, desc=7 ticks), Event(type=Program, value=0, time=354, desc=361), Event(type=Pitch, value=43, time=354, desc=361), Event(type=Velocity, value=99, time=354, desc=99), Event(type=Duration, value=0.7.8, time=354, desc=7 ticks), Event(type=Program, value=0, time=354, desc=359), Event(type=Pitch, value=76, time=354, desc=359), Event(type=Velocity, value=95, time=354, desc=95), Event(type=Duration, value=0.5.8, time=354, desc=5 ticks), Event(type=Position, value=3, time=355, desc=355), Event(type=Program, value=0, time=355, desc=360), Event(type=Pitch, value=64, time=355, desc=360), Event(type=Velocity, value=79, time=355, desc=79), Event(type=Duration, value=0.5.8, time=355, desc=5 ticks), Event(type=Position, value=5, time=357, desc=357), Event(type=Program, value=0, time=357, desc=361), Event(type=Pitch, value=67, time=357, desc=361), Event(type=Velocity, value=79, time=357, desc=79), Event(type=Duration, value=0.4.8, time=357, desc=4 ticks), Event(type=Position, value=6, time=358, desc=358), Event(type=Program, value=0, time=358, desc=366), Event(type=Pitch, value=76, time=358, desc=366), Event(type=Velocity, value=79, time=358, desc=79), Event(type=Duration, value=1.0.8, time=358, desc=8 ticks), Event(type=Position, value=8, time=360, desc=360), Event(type=Program, value=0, time=360, desc=366), Event(type=Pitch, value=60, time=360, desc=366), Event(type=Velocity, value=91, time=360, desc=91), Event(type=Duration, value=0.6.8, time=360, desc=6 ticks), Event(type=Program, value=0, time=360, desc=364), Event(type=Pitch, value=72, time=360, desc=364), Event(type=Velocity, value=91, time=360, desc=91), Event(type=Duration, value=0.4.8, time=360, desc=4 ticks), Event(type=Program, value=0, time=360, desc=366), Event(type=Pitch, value=58, time=360, desc=366), Event(type=Velocity, value=91, time=360, desc=91), Event(type=Duration, value=0.6.8, time=360, desc=6 ticks), Event(type=Position, value=9, time=361, desc=361), Event(type=Program, value=0, time=361, desc=366), Event(type=Pitch, value=64, time=361, desc=366), Event(type=Velocity, value=79, time=361, desc=79), Event(type=Duration, value=0.5.8, time=361, desc=5 ticks), Event(type=Position, value=10, time=362, desc=362), Event(type=Program, value=0, time=362, desc=366), Event(type=Pitch, value=67, time=362, desc=366), Event(type=Velocity, value=87, time=362, desc=87), Event(type=Duration, value=0.4.8, time=362, desc=4 ticks), Event(type=Position, value=12, time=364, desc=364), Event(type=Program, value=0, time=364, desc=366), Event(type=Pitch, value=72, time=364, desc=366), Event(type=Velocity, value=87, time=364, desc=87), Event(type=Duration, value=0.2.8, time=364, desc=2 ticks), Event(type=Position, value=13, time=365, desc=365), Event(type=Program, value=0, time=365, desc=370), Event(type=Pitch, value=77, time=365, desc=370), Event(type=Velocity, value=95, time=365, desc=95), Event(type=Duration, value=0.5.8, time=365, desc=5 ticks), Event(type=Position, value=14, time=366, desc=366), Event(type=Program, value=0, time=366, desc=372), Event(type=Pitch, value=57, time=366, desc=372), Event(type=Velocity, value=91, time=366, desc=91), Event(type=Duration, value=0.6.8, time=366, desc=6 ticks), Event(type=Program, value=0, time=366, desc=372), Event(type=Pitch, value=60, time=366, desc=372), Event(type=Velocity, value=91, time=366, desc=91), Event(type=Duration, value=0.6.8, time=366, desc=6 ticks), Event(type=Position, value=15, time=367, desc=367), Event(type=Program, value=0, time=367, desc=372), Event(type=Pitch, value=65, time=367, desc=372), Event(type=Velocity, value=87, time=367, desc=87), Event(type=Duration, value=0.5.8, time=367, desc=5 ticks), Event(type=Position, value=16, time=368, desc=368), Event(type=Program, value=0, time=368, desc=371), Event(type=Pitch, value=72, time=368, desc=371), Event(type=Velocity, value=87, time=368, desc=87), Event(type=Duration, value=0.3.8, time=368, desc=3 ticks), Event(type=Position, value=18, time=370, desc=370), Event(type=Program, value=0, time=370, desc=383), Event(type=Pitch, value=77, time=370, desc=383), Event(type=Velocity, value=87, time=370, desc=87), Event(type=Duration, value=1.5.8, time=370, desc=13 ticks), Event(type=Position, value=20, time=372, desc=372), Event(type=Program, value=0, time=372, desc=376), Event(type=Pitch, value=74, time=372, desc=376), Event(type=Velocity, value=87, time=372, desc=87), Event(type=Duration, value=0.4.8, time=372, desc=4 ticks), Event(type=Program, value=0, time=372, desc=384), Event(type=Pitch, value=56, time=372, desc=384), Event(type=Velocity, value=79, time=372, desc=79), Event(type=Duration, value=1.4.8, time=372, desc=12 ticks), Event(type=Program, value=0, time=372, desc=384), Event(type=Pitch, value=59, time=372, desc=384), Event(type=Velocity, value=87, time=372, desc=87), Event(type=Duration, value=1.4.8, time=372, desc=12 ticks), Event(type=Position, value=21, time=373, desc=373), Event(type=Program, value=0, time=373, desc=384), Event(type=Pitch, value=65, time=373, desc=384), Event(type=Velocity, value=79, time=373, desc=79), Event(type=Duration, value=1.3.8, time=373, desc=11 ticks), Event(type=Position, value=22, time=374, desc=374), Event(type=Program, value=0, time=374, desc=383), Event(type=Pitch, value=71, time=374, desc=383), Event(type=Velocity, value=79, time=374, desc=79), Event(type=Duration, value=1.1.8, time=374, desc=9 ticks), Event(type=Position, value=24, time=376, desc=376), Event(type=Program, value=0, time=376, desc=384), Event(type=Pitch, value=74, time=376, desc=384), Event(type=Velocity, value=91, time=376, desc=91), Event(type=Duration, value=1.0.8, time=376, desc=8 ticks), Event(type=Position, value=26, time=378, desc=378), Event(type=Program, value=0, time=378, desc=384), Event(type=Pitch, value=55, time=378, desc=384), Event(type=Velocity, value=87, time=378, desc=87), Event(type=Duration, value=0.6.8, time=378, desc=6 ticks), Event(type=Program, value=0, time=378, desc=384), Event(type=Pitch, value=76, time=378, desc=384), Event(type=Velocity, value=91, time=378, desc=91), Event(type=Duration, value=0.6.8, time=378, desc=6 ticks), Event(type=Program, value=0, time=378, desc=384), Event(type=Pitch, value=60, time=378, desc=384), Event(type=Velocity, value=87, time=378, desc=87), Event(type=Duration, value=0.6.8, time=378, desc=6 ticks), Event(type=Position, value=27, time=379, desc=379), Event(type=Program, value=0, time=379, desc=384), Event(type=Pitch, value=64, time=379, desc=384), Event(type=Velocity, value=87, time=379, desc=87), Event(type=Duration, value=0.5.8, time=379, desc=5 ticks), Event(type=Position, value=28, time=380, desc=380), Event(type=Program, value=0, time=380, desc=383), Event(type=Pitch, value=67, time=380, desc=383), Event(type=Velocity, value=87, time=380, desc=87), Event(type=Duration, value=0.3.8, time=380, desc=3 ticks), Event(type=Position, value=30, time=382, desc=382), Event(type=Program, value=0, time=382, desc=384), Event(type=Pitch, value=72, time=382, desc=384), Event(type=Velocity, value=87, time=382, desc=87), Event(type=Duration, value=0.2.8, time=382, desc=2 ticks), Event(type=Position, value=31, time=383, desc=383), Event(type=Program, value=0, time=383, desc=390), Event(type=Pitch, value=71, time=383, desc=390), Event(type=Velocity, value=87, time=383, desc=87), Event(type=Duration, value=0.7.8, time=383, desc=7 ticks), Event(type=Program, value=0, time=383, desc=390), Event(type=Pitch, value=43, time=383, desc=390), Event(type=Velocity, value=91, time=383, desc=91), Event(type=Duration, value=0.7.8, time=383, desc=7 ticks), Event(type=Bar, value=None, time=384, desc=0), Event(type=Position, value=0, time=384, desc=384), Event(type=Program, value=0, time=384, desc=390), Event(type=Pitch, value=36, time=384, desc=390), Event(type=Velocity, value=79, time=384, desc=79), Event(type=Duration, value=0.6.8, time=384, desc=6 ticks), Event(type=Position, value=1, time=385, desc=385), Event(type=Program, value=0, time=385, desc=390), Event(type=Pitch, value=65, time=385, desc=390), Event(type=Velocity, value=87, time=385, desc=87), Event(type=Duration, value=0.5.8, time=385, desc=5 ticks), Event(type=Position, value=2, time=386, desc=386), Event(type=Program, value=0, time=386, desc=390), Event(type=Pitch, value=67, time=386, desc=390), Event(type=Velocity, value=79, time=386, desc=79), Event(type=Duration, value=0.4.8, time=386, desc=4 ticks), Event(type=Position, value=4, time=388, desc=388), Event(type=Program, value=0, time=388, desc=390), Event(type=Pitch, value=74, time=388, desc=390), Event(type=Velocity, value=75, time=388, desc=75), Event(type=Duration, value=0.2.8, time=388, desc=2 ticks), Event(type=Position, value=5, time=389, desc=389), Event(type=Program, value=0, time=389, desc=391), Event(type=Pitch, value=72, time=389, desc=391), Event(type=Velocity, value=91, time=389, desc=91), Event(type=Duration, value=0.2.8, time=389, desc=2 ticks), Event(type=Program, value=0, time=389, desc=390), Event(type=Pitch, value=36, time=389, desc=390), Event(type=Velocity, value=87, time=389, desc=87), Event(type=Duration, value=0.1.8, time=389, desc=1 ticks), Event(type=Position, value=6, time=390, desc=390), Event(type=Program, value=0, time=390, desc=391), Event(type=Pitch, value=64, time=390, desc=391), Event(type=Velocity, value=71, time=390, desc=71), Event(type=Duration, value=0.1.8, time=390, desc=1 ticks), Event(type=Position, value=8, time=392, desc=392), Event(type=Program, value=0, time=392, desc=393), Event(type=Pitch, value=65, time=392, desc=393), Event(type=Velocity, value=87, time=392, desc=87), Event(type=Duration, value=0.1.8, time=392, desc=1 ticks), Event(type=Program, value=0, time=392, desc=393), Event(type=Pitch, value=43, time=392, desc=393), Event(type=Velocity, value=75, time=392, desc=75), Event(type=Duration, value=0.1.8, time=392, desc=1 ticks), Event(type=Position, value=10, time=394, desc=394), Event(type=Program, value=0, time=394, desc=396), Event(type=Pitch, value=71, time=394, desc=396), Event(type=Velocity, value=75, time=394, desc=75), Event(type=Duration, value=0.2.8, time=394, desc=2 ticks), Event(type=Position, value=11, time=395, desc=395), Event(type=Program, value=0, time=395, desc=397), Event(type=Pitch, value=72, time=395, desc=397), Event(type=Velocity, value=87, time=395, desc=87), Event(type=Duration, value=0.2.8, time=395, desc=2 ticks), Event(type=Program, value=0, time=395, desc=396), Event(type=Pitch, value=64, time=395, desc=396), Event(type=Velocity, value=91, time=395, desc=91), Event(type=Duration, value=0.1.8, time=395, desc=1 ticks), Event(type=Program, value=0, time=395, desc=396), Event(type=Pitch, value=48, time=395, desc=396), Event(type=Velocity, value=79, time=395, desc=79), Event(type=Duration, value=0.1.8, time=395, desc=1 ticks), Event(type=Position, value=14, time=398, desc=398), Event(type=Program, value=0, time=398, desc=399), Event(type=Pitch, value=55, time=398, desc=399), Event(type=Velocity, value=79, time=398, desc=79), Event(type=Duration, value=0.1.8, time=398, desc=1 ticks), Event(type=Position, value=17, time=401, desc=401), Event(type=Program, value=0, time=401, desc=402), Event(type=Pitch, value=79, time=401, desc=402), Event(type=Velocity, value=91, time=401, desc=91), Event(type=Duration, value=0.1.8, time=401, desc=1 ticks), Event(type=Program, value=0, time=401, desc=402), Event(type=Pitch, value=60, time=401, desc=402), Event(type=Velocity, value=79, time=401, desc=79), Event(type=Duration, value=0.1.8, time=401, desc=1 ticks), Event(type=Position, value=19, time=403, desc=403), Event(type=Program, value=0, time=403, desc=405), Event(type=Pitch, value=81, time=403, desc=405), Event(type=Velocity, value=75, time=403, desc=75), Event(type=Duration, value=0.2.8, time=403, desc=2 ticks), Event(type=Position, value=20, time=404, desc=404), Event(type=Program, value=0, time=404, desc=405), Event(type=Pitch, value=67, time=404, desc=405), Event(type=Velocity, value=67, time=404, desc=67), Event(type=Duration, value=0.1.8, time=404, desc=1 ticks), Event(type=Program, value=0, time=404, desc=405), Event(type=Pitch, value=64, time=404, desc=405), Event(type=Velocity, value=75, time=404, desc=75), Event(type=Duration, value=0.1.8, time=404, desc=1 ticks), Event(type=Program, value=0, time=404, desc=405), Event(type=Pitch, value=72, time=404, desc=405), Event(type=Velocity, value=75, time=404, desc=75), Event(type=Duration, value=0.1.8, time=404, desc=1 ticks), Event(type=Position, value=22, time=406, desc=406), Event(type=Program, value=0, time=406, desc=407), Event(type=Pitch, value=77, time=406, desc=407), Event(type=Velocity, value=79, time=406, desc=79), Event(type=Duration, value=0.1.8, time=406, desc=1 ticks), Event(type=Position, value=23, time=407, desc=407), Event(type=Program, value=0, time=407, desc=408), Event(type=Pitch, value=76, time=407, desc=408), Event(type=Velocity, value=75, time=407, desc=75), Event(type=Duration, value=0.1.8, time=407, desc=1 ticks), Event(type=Position, value=24, time=408, desc=408), Event(type=Program, value=0, time=408, desc=409), Event(type=Pitch, value=67, time=408, desc=409), Event(type=Velocity, value=71, time=408, desc=71), Event(type=Duration, value=0.1.8, time=408, desc=1 ticks), Event(type=Program, value=0, time=408, desc=409), Event(type=Pitch, value=73, time=408, desc=409), Event(type=Velocity, value=75, time=408, desc=75), Event(type=Duration, value=0.1.8, time=408, desc=1 ticks), Event(type=Program, value=0, time=408, desc=409), Event(type=Pitch, value=64, time=408, desc=409), Event(type=Velocity, value=71, time=408, desc=71), Event(type=Duration, value=0.1.8, time=408, desc=1 ticks), Event(type=Position, value=25, time=409, desc=409), Event(type=Program, value=0, time=409, desc=410), Event(type=Pitch, value=77, time=409, desc=410), Event(type=Velocity, value=71, time=409, desc=71), Event(type=Duration, value=0.1.8, time=409, desc=1 ticks), Event(type=Position, value=26, time=410, desc=410), Event(type=Program, value=0, time=410, desc=411), Event(type=Pitch, value=79, time=410, desc=411), Event(type=Velocity, value=79, time=410, desc=79), Event(type=Duration, value=0.1.8, time=410, desc=1 ticks), Event(type=Position, value=27, time=411, desc=411), Event(type=Program, value=0, time=411, desc=412), Event(type=Pitch, value=58, time=411, desc=412), Event(type=Velocity, value=75, time=411, desc=75), Event(type=Duration, value=0.1.8, time=411, desc=1 ticks), Event(type=Position, value=28, time=412, desc=412), Event(type=Program, value=0, time=412, desc=414), Event(type=Pitch, value=76, time=412, desc=414), Event(type=Velocity, value=71, time=412, desc=71), Event(type=Duration, value=0.2.8, time=412, desc=2 ticks), Event(type=Position, value=29, time=413, desc=413), Event(type=Program, value=0, time=413, desc=414), Event(type=Pitch, value=77, time=413, desc=414), Event(type=Velocity, value=79, time=413, desc=79), Event(type=Duration, value=0.1.8, time=413, desc=1 ticks), Event(type=Program, value=0, time=413, desc=414), Event(type=Pitch, value=57, time=413, desc=414), Event(type=Velocity, value=87, time=413, desc=87), Event(type=Duration, value=0.1.8, time=413, desc=1 ticks), Event(type=Position, value=30, time=414, desc=414), Event(type=Program, value=0, time=414, desc=416), Event(type=Pitch, value=79, time=414, desc=416), Event(type=Velocity, value=75, time=414, desc=75), Event(type=Duration, value=0.2.8, time=414, desc=2 ticks), Event(type=Bar, value=None, time=416, desc=0), Event(type=Position, value=0, time=416, desc=416), Event(type=Program, value=0, time=416, desc=417), Event(type=Pitch, value=69, time=416, desc=417), Event(type=Velocity, value=67, time=416, desc=67), Event(type=Duration, value=0.1.8, time=416, desc=1 ticks), Event(type=Program, value=0, time=416, desc=417), Event(type=Pitch, value=60, time=416, desc=417), Event(type=Velocity, value=59, time=416, desc=59), Event(type=Duration, value=0.1.8, time=416, desc=1 ticks), Event(type=Program, value=0, time=416, desc=417), Event(type=Pitch, value=77, time=416, desc=417), Event(type=Velocity, value=67, time=416, desc=67), Event(type=Duration, value=0.1.8, time=416, desc=1 ticks), Event(type=Program, value=0, time=416, desc=417), Event(type=Pitch, value=65, time=416, desc=417), Event(type=Velocity, value=51, time=416, desc=51), Event(type=Duration, value=0.1.8, time=416, desc=1 ticks), Event(type=Position, value=1, time=417, desc=417), Event(type=Program, value=0, time=417, desc=418), Event(type=Pitch, value=76, time=417, desc=418), Event(type=Velocity, value=75, time=417, desc=75), Event(type=Duration, value=0.1.8, time=417, desc=1 ticks), Event(type=Position, value=3, time=419, desc=419), Event(type=Program, value=0, time=419, desc=420), Event(type=Pitch, value=74, time=419, desc=420), Event(type=Velocity, value=75, time=419, desc=75), Event(type=Duration, value=0.1.8, time=419, desc=1 ticks), Event(type=Program, value=0, time=419, desc=420), Event(type=Pitch, value=71, time=419, desc=420), Event(type=Velocity, value=87, time=419, desc=87), Event(type=Duration, value=0.1.8, time=419, desc=1 ticks), Event(type=Program, value=0, time=419, desc=420), Event(type=Pitch, value=65, time=419, desc=420), Event(type=Velocity, value=67, time=419, desc=67), Event(type=Duration, value=0.1.8, time=419, desc=1 ticks), Event(type=Program, value=0, time=419, desc=420), Event(type=Pitch, value=62, time=419, desc=420), Event(type=Velocity, value=67, time=419, desc=67), Event(type=Duration, value=0.1.8, time=419, desc=1 ticks), Event(type=Position, value=4, time=420, desc=420), Event(type=Program, value=0, time=420, desc=421), Event(type=Pitch, value=76, time=420, desc=421), Event(type=Velocity, value=75, time=420, desc=75), Event(type=Duration, value=0.1.8, time=420, desc=1 ticks), Event(type=Position, value=6, time=422, desc=422), Event(type=Program, value=0, time=422, desc=423), Event(type=Pitch, value=77, time=422, desc=423), Event(type=Velocity, value=71, time=422, desc=71), Event(type=Duration, value=0.1.8, time=422, desc=1 ticks), Event(type=Program, value=0, time=422, desc=423), Event(type=Pitch, value=56, time=422, desc=423), Event(type=Velocity, value=67, time=422, desc=67), Event(type=Duration, value=0.1.8, time=422, desc=1 ticks), Event(type=Position, value=8, time=424, desc=424), Event(type=Program, value=0, time=424, desc=425), Event(type=Pitch, value=74, time=424, desc=425), Event(type=Velocity, value=59, time=424, desc=59), Event(type=Duration, value=0.1.8, time=424, desc=1 ticks), Event(type=Position, value=9, time=425, desc=425), Event(type=Program, value=0, time=425, desc=426), Event(type=Pitch, value=76, time=425, desc=426), Event(type=Velocity, value=75, time=425, desc=75), Event(type=Duration, value=0.1.8, time=425, desc=1 ticks), Event(type=Program, value=0, time=425, desc=426), Event(type=Pitch, value=55, time=425, desc=426), Event(type=Velocity, value=79, time=425, desc=79), Event(type=Duration, value=0.1.8, time=425, desc=1 ticks), Event(type=Position, value=10, time=426, desc=426), Event(type=Program, value=0, time=426, desc=428), Event(type=Pitch, value=77, time=426, desc=428), Event(type=Velocity, value=71, time=426, desc=71), Event(type=Duration, value=0.2.8, time=426, desc=2 ticks), Event(type=Position, value=12, time=428, desc=428), Event(type=Program, value=0, time=428, desc=430), Event(type=Pitch, value=76, time=428, desc=430), Event(type=Velocity, value=75, time=428, desc=75), Event(type=Duration, value=0.2.8, time=428, desc=2 ticks), Event(type=Program, value=0, time=428, desc=429), Event(type=Pitch, value=67, time=428, desc=429), Event(type=Velocity, value=67, time=428, desc=67), Event(type=Duration, value=0.1.8, time=428, desc=1 ticks), Event(type=Program, value=0, time=428, desc=429), Event(type=Pitch, value=64, time=428, desc=429), Event(type=Velocity, value=51, time=428, desc=51), Event(type=Duration, value=0.1.8, time=428, desc=1 ticks), Event(type=Program, value=0, time=428, desc=429), Event(type=Pitch, value=60, time=428, desc=429), Event(type=Velocity, value=51, time=428, desc=51), Event(type=Duration, value=0.1.8, time=428, desc=1 ticks), Event(type=Position, value=13, time=429, desc=429), Event(type=Program, value=0, time=429, desc=430), Event(type=Pitch, value=74, time=429, desc=430), Event(type=Velocity, value=71, time=429, desc=71), Event(type=Duration, value=0.1.8, time=429, desc=1 ticks), Event(type=Position, value=15, time=431, desc=431), Event(type=Program, value=0, time=431, desc=432), Event(type=Pitch, value=72, time=431, desc=432), Event(type=Velocity, value=79, time=431, desc=79), Event(type=Duration, value=0.1.8, time=431, desc=1 ticks), Event(type=Program, value=0, time=431, desc=432), Event(type=Pitch, value=64, time=431, desc=432), Event(type=Velocity, value=75, time=431, desc=75), Event(type=Duration, value=0.1.8, time=431, desc=1 ticks), Event(type=Program, value=0, time=431, desc=432), Event(type=Pitch, value=60, time=431, desc=432), Event(type=Velocity, value=55, time=431, desc=55), Event(type=Duration, value=0.1.8, time=431, desc=1 ticks), Event(type=Program, value=0, time=431, desc=432), Event(type=Pitch, value=69, time=431, desc=432), Event(type=Velocity, value=75, time=431, desc=75), Event(type=Duration, value=0.1.8, time=431, desc=1 ticks), Event(type=Position, value=16, time=432, desc=432), Event(type=Program, value=0, time=432, desc=433), Event(type=Pitch, value=74, time=432, desc=433), Event(type=Velocity, value=71, time=432, desc=71), Event(type=Duration, value=0.1.8, time=432, desc=1 ticks), Event(type=Position, value=17, time=433, desc=433), Event(type=Program, value=0, time=433, desc=434), Event(type=Pitch, value=76, time=433, desc=434), Event(type=Velocity, value=79, time=433, desc=79), Event(type=Duration, value=0.1.8, time=433, desc=1 ticks), Event(type=Position, value=18, time=434, desc=434), Event(type=Program, value=0, time=434, desc=435), Event(type=Pitch, value=54, time=434, desc=435), Event(type=Velocity, value=75, time=434, desc=75), Event(type=Duration, value=0.1.8, time=434, desc=1 ticks), Event(type=Position, value=19, time=435, desc=435), Event(type=Program, value=0, time=435, desc=436), Event(type=Pitch, value=72, time=435, desc=436), Event(type=Velocity, value=71, time=435, desc=71), Event(type=Duration, value=0.1.8, time=435, desc=1 ticks), Event(type=Position, value=20, time=436, desc=436), Event(type=Program, value=0, time=436, desc=437), Event(type=Pitch, value=74, time=436, desc=437), Event(type=Velocity, value=71, time=436, desc=71), Event(type=Duration, value=0.1.8, time=436, desc=1 ticks), Event(type=Program, value=0, time=436, desc=437), Event(type=Pitch, value=53, time=436, desc=437), Event(type=Velocity, value=79, time=436, desc=79), Event(type=Duration, value=0.1.8, time=436, desc=1 ticks), Event(type=Position, value=21, time=437, desc=437), Event(type=Program, value=0, time=437, desc=439), Event(type=Pitch, value=76, time=437, desc=439), Event(type=Velocity, value=67, time=437, desc=67), Event(type=Duration, value=0.2.8, time=437, desc=2 ticks), Event(type=Position, value=23, time=439, desc=439), Event(type=Program, value=0, time=439, desc=440), Event(type=Pitch, value=74, time=439, desc=440), Event(type=Velocity, value=55, time=439, desc=55), Event(type=Duration, value=0.1.8, time=439, desc=1 ticks), Event(type=Program, value=0, time=439, desc=440), Event(type=Pitch, value=59, time=439, desc=440), Event(type=Velocity, value=71, time=439, desc=71), Event(type=Duration, value=0.1.8, time=439, desc=1 ticks), Event(type=Program, value=0, time=439, desc=440), Event(type=Pitch, value=62, time=439, desc=440), Event(type=Velocity, value=75, time=439, desc=75), Event(type=Duration, value=0.1.8, time=439, desc=1 ticks), Event(type=Position, value=25, time=441, desc=441), Event(type=Program, value=0, time=441, desc=442), Event(type=Pitch, value=71, time=441, desc=442), Event(type=Velocity, value=67, time=441, desc=67), Event(type=Duration, value=0.1.8, time=441, desc=1 ticks), Event(type=Position, value=26, time=442, desc=442), Event(type=Program, value=0, time=442, desc=443), Event(type=Pitch, value=67, time=442, desc=443), Event(type=Velocity, value=71, time=442, desc=71), Event(type=Duration, value=0.1.8, time=442, desc=1 ticks), Event(type=Program, value=0, time=442, desc=443), Event(type=Pitch, value=52, time=442, desc=443), Event(type=Velocity, value=79, time=442, desc=79), Event(type=Duration, value=0.1.8, time=442, desc=1 ticks), Event(type=Position, value=28, time=444, desc=444), Event(type=Program, value=0, time=444, desc=445), Event(type=Pitch, value=69, time=444, desc=445), Event(type=Velocity, value=67, time=444, desc=67), Event(type=Duration, value=0.1.8, time=444, desc=1 ticks), Event(type=Position, value=29, time=445, desc=445), Event(type=Program, value=0, time=445, desc=446), Event(type=Pitch, value=71, time=445, desc=446), Event(type=Velocity, value=59, time=445, desc=59), Event(type=Duration, value=0.1.8, time=445, desc=1 ticks), Event(type=Program, value=0, time=445, desc=446), Event(type=Pitch, value=50, time=445, desc=446), Event(type=Velocity, value=71, time=445, desc=71), Event(type=Duration, value=0.1.8, time=445, desc=1 ticks), Event(type=Position, value=31, time=447, desc=447), Event(type=Program, value=0, time=447, desc=448), Event(type=Pitch, value=74, time=447, desc=448), Event(type=Velocity, value=67, time=447, desc=67), Event(type=Duration, value=0.1.8, time=447, desc=1 ticks), Event(type=Bar, value=None, time=448, desc=0), Event(type=Position, value=0, time=448, desc=448), Event(type=Program, value=0, time=448, desc=449), Event(type=Pitch, value=48, time=448, desc=449), Event(type=Velocity, value=79, time=448, desc=79), Event(type=Duration, value=0.1.8, time=448, desc=1 ticks), Event(type=Program, value=0, time=448, desc=449), Event(type=Pitch, value=79, time=448, desc=449), Event(type=Velocity, value=87, time=448, desc=87), Event(type=Duration, value=0.1.8, time=448, desc=1 ticks), Event(type=Position, value=2, time=450, desc=450), Event(type=Program, value=0, time=450, desc=451), Event(type=Pitch, value=81, time=450, desc=451), Event(type=Velocity, value=79, time=450, desc=79), Event(type=Duration, value=0.1.8, time=450, desc=1 ticks), Event(type=Position, value=3, time=451, desc=451), Event(type=Program, value=0, time=451, desc=452), Event(type=Pitch, value=79, time=451, desc=452), Event(type=Velocity, value=71, time=451, desc=71), Event(type=Duration, value=0.1.8, time=451, desc=1 ticks), Event(type=Program, value=0, time=451, desc=452), Event(type=Pitch, value=64, time=451, desc=452), Event(type=Velocity, value=75, time=451, desc=75), Event(type=Duration, value=0.1.8, time=451, desc=1 ticks), Event(type=Program, value=0, time=451, desc=452), Event(type=Pitch, value=67, time=451, desc=452), Event(type=Velocity, value=67, time=451, desc=67), Event(type=Duration, value=0.1.8, time=451, desc=1 ticks), Event(type=Position, value=4, time=452, desc=452), Event(type=Program, value=0, time=452, desc=453), Event(type=Pitch, value=77, time=452, desc=453), Event(type=Velocity, value=75, time=452, desc=75), Event(type=Duration, value=0.1.8, time=452, desc=1 ticks), Event(type=Position, value=6, time=454, desc=454), Event(type=Program, value=0, time=454, desc=455), Event(type=Pitch, value=76, time=454, desc=455), Event(type=Velocity, value=79, time=454, desc=79), Event(type=Duration, value=0.1.8, time=454, desc=1 ticks), Event(type=Program, value=0, time=454, desc=455), Event(type=Pitch, value=58, time=454, desc=455), Event(type=Velocity, value=71, time=454, desc=71), Event(type=Duration, value=0.1.8, time=454, desc=1 ticks), Event(type=Position, value=7, time=455, desc=455), Event(type=Program, value=0, time=455, desc=456), Event(type=Pitch, value=77, time=455, desc=456), Event(type=Velocity, value=71, time=455, desc=71), Event(type=Duration, value=0.1.8, time=455, desc=1 ticks), Event(type=Position, value=9, time=457, desc=457), Event(type=Program, value=0, time=457, desc=458), Event(type=Pitch, value=79, time=457, desc=458), Event(type=Velocity, value=67, time=457, desc=67), Event(type=Duration, value=0.1.8, time=457, desc=1 ticks), Event(type=Program, value=0, time=457, desc=458), Event(type=Pitch, value=67, time=457, desc=458), Event(type=Velocity, value=59, time=457, desc=59), Event(type=Duration, value=0.1.8, time=457, desc=1 ticks), Event(type=Program, value=0, time=457, desc=458), Event(type=Pitch, value=64, time=457, desc=458), Event(type=Velocity, value=67, time=457, desc=67), Event(type=Duration, value=0.1.8, time=457, desc=1 ticks), Event(type=Position, value=10, time=458, desc=458), Event(type=Program, value=0, time=458, desc=459), Event(type=Pitch, value=76, time=458, desc=459), Event(type=Velocity, value=67, time=458, desc=67), Event(type=Duration, value=0.1.8, time=458, desc=1 ticks), Event(type=Position, value=12, time=460, desc=460), Event(type=Program, value=0, time=460, desc=461), Event(type=Pitch, value=77, time=460, desc=461), Event(type=Velocity, value=79, time=460, desc=79), Event(type=Duration, value=0.1.8, time=460, desc=1 ticks), Event(type=Program, value=0, time=460, desc=461), Event(type=Pitch, value=57, time=460, desc=461), Event(type=Velocity, value=79, time=460, desc=79), Event(type=Duration, value=0.1.8, time=460, desc=1 ticks), Event(type=Position, value=13, time=461, desc=461), Event(type=Program, value=0, time=461, desc=463), Event(type=Pitch, value=79, time=461, desc=463), Event(type=Velocity, value=67, time=461, desc=67), Event(type=Duration, value=0.2.8, time=461, desc=2 ticks), Event(type=Position, value=14, time=462, desc=462), Event(type=Program, value=0, time=462, desc=463), Event(type=Pitch, value=77, time=462, desc=463), Event(type=Velocity, value=71, time=462, desc=71), Event(type=Duration, value=0.1.8, time=462, desc=1 ticks), Event(type=Position, value=15, time=463, desc=463), Event(type=Program, value=0, time=463, desc=464), Event(type=Pitch, value=65, time=463, desc=464), Event(type=Velocity, value=71, time=463, desc=71), Event(type=Duration, value=0.1.8, time=463, desc=1 ticks), Event(type=Program, value=0, time=463, desc=464), Event(type=Pitch, value=60, time=463, desc=464), Event(type=Velocity, value=55, time=463, desc=55), Event(type=Duration, value=0.1.8, time=463, desc=1 ticks), Event(type=Position, value=16, time=464, desc=464), Event(type=Program, value=0, time=464, desc=465), Event(type=Pitch, value=76, time=464, desc=465), Event(type=Velocity, value=71, time=464, desc=71), Event(type=Duration, value=0.1.8, time=464, desc=1 ticks), Event(type=Position, value=17, time=465, desc=465), Event(type=Program, value=0, time=465, desc=466), Event(type=Pitch, value=74, time=465, desc=466), Event(type=Velocity, value=75, time=465, desc=75), Event(type=Duration, value=0.1.8, time=465, desc=1 ticks), Event(type=Program, value=0, time=465, desc=466), Event(type=Pitch, value=56, time=465, desc=466), Event(type=Velocity, value=71, time=465, desc=71), Event(type=Duration, value=0.1.8, time=465, desc=1 ticks), Event(type=Position, value=19, time=467, desc=467), Event(type=Program, value=0, time=467, desc=468), Event(type=Pitch, value=76, time=467, desc=468), Event(type=Velocity, value=71, time=467, desc=71), Event(type=Duration, value=0.1.8, time=467, desc=1 ticks), Event(type=Position, value=20, time=468, desc=468), Event(type=Program, value=0, time=468, desc=469), Event(type=Pitch, value=77, time=468, desc=469), Event(type=Velocity, value=75, time=468, desc=75), Event(type=Duration, value=0.1.8, time=468, desc=1 ticks), Event(type=Program, value=0, time=468, desc=469), Event(type=Pitch, value=65, time=468, desc=469), Event(type=Velocity, value=67, time=468, desc=67), Event(type=Duration, value=0.1.8, time=468, desc=1 ticks), Event(type=Program, value=0, time=468, desc=469), Event(type=Pitch, value=59, time=468, desc=469), Event(type=Velocity, value=59, time=468, desc=59), Event(type=Duration, value=0.1.8, time=468, desc=1 ticks), Event(type=Position, value=22, time=470, desc=470), Event(type=Program, value=0, time=470, desc=471), Event(type=Pitch, value=74, time=470, desc=471), Event(type=Velocity, value=67, time=470, desc=67), Event(type=Duration, value=0.1.8, time=470, desc=1 ticks), Event(type=Position, value=23, time=471, desc=471), Event(type=Program, value=0, time=471, desc=472), Event(type=Pitch, value=76, time=471, desc=472), Event(type=Velocity, value=79, time=471, desc=79), Event(type=Duration, value=0.1.8, time=471, desc=1 ticks), Event(type=Program, value=0, time=471, desc=472), Event(type=Pitch, value=54, time=471, desc=472), Event(type=Velocity, value=79, time=471, desc=79), Event(type=Duration, value=0.1.8, time=471, desc=1 ticks), Event(type=Position, value=24, time=472, desc=472), Event(type=Program, value=0, time=472, desc=474), Event(type=Pitch, value=77, time=472, desc=474), Event(type=Velocity, value=71, time=472, desc=71), Event(type=Duration, value=0.2.8, time=472, desc=2 ticks), Event(type=Position, value=26, time=474, desc=474), Event(type=Program, value=0, time=474, desc=475), Event(type=Pitch, value=76, time=474, desc=475), Event(type=Velocity, value=67, time=474, desc=67), Event(type=Duration, value=0.1.8, time=474, desc=1 ticks), Event(type=Program, value=0, time=474, desc=475), Event(type=Pitch, value=60, time=474, desc=475), Event(type=Velocity, value=67, time=474, desc=67), Event(type=Duration, value=0.1.8, time=474, desc=1 ticks), Event(type=Program, value=0, time=474, desc=475), Event(type=Pitch, value=64, time=474, desc=475), Event(type=Velocity, value=75, time=474, desc=75), Event(type=Duration, value=0.1.8, time=474, desc=1 ticks), Event(type=Position, value=27, time=475, desc=475), Event(type=Program, value=0, time=475, desc=476), Event(type=Pitch, value=72, time=475, desc=476), Event(type=Velocity, value=75, time=475, desc=75), Event(type=Duration, value=0.1.8, time=475, desc=1 ticks), Event(type=Position, value=29, time=477, desc=477), Event(type=Program, value=0, time=477, desc=478), Event(type=Pitch, value=55, time=477, desc=478), Event(type=Velocity, value=75, time=477, desc=75), Event(type=Duration, value=0.1.8, time=477, desc=1 ticks), Event(type=Program, value=0, time=477, desc=478), Event(type=Pitch, value=71, time=477, desc=478), Event(type=Velocity, value=75, time=477, desc=75), Event(type=Duration, value=0.1.8, time=477, desc=1 ticks), Event(type=Position, value=30, time=478, desc=478), Event(type=Program, value=0, time=478, desc=479), Event(type=Pitch, value=72, time=478, desc=479), Event(type=Velocity, value=75, time=478, desc=75), Event(type=Duration, value=0.1.8, time=478, desc=1 ticks), Event(type=Position, value=31, time=479, desc=479), Event(type=Program, value=0, time=479, desc=480), Event(type=Pitch, value=74, time=479, desc=480), Event(type=Velocity, value=75, time=479, desc=75), Event(type=Duration, value=0.1.8, time=479, desc=1 ticks), Event(type=Bar, value=None, time=480, desc=0), Event(type=Position, value=0, time=480, desc=480), Event(type=Program, value=0, time=480, desc=481), Event(type=Pitch, value=62, time=480, desc=481), Event(type=Velocity, value=79, time=480, desc=79), Event(type=Duration, value=0.1.8, time=480, desc=1 ticks), Event(type=Program, value=0, time=480, desc=481), Event(type=Pitch, value=65, time=480, desc=481), Event(type=Velocity, value=75, time=480, desc=75), Event(type=Duration, value=0.1.8, time=480, desc=1 ticks), Event(type=Position, value=1, time=481, desc=481), Event(type=Program, value=0, time=481, desc=482), Event(type=Pitch, value=71, time=481, desc=482), Event(type=Velocity, value=75, time=481, desc=75), Event(type=Duration, value=0.1.8, time=481, desc=1 ticks), Event(type=Position, value=2, time=482, desc=482), Event(type=Program, value=0, time=482, desc=483), Event(type=Pitch, value=72, time=482, desc=483), Event(type=Velocity, value=87, time=482, desc=87), Event(type=Duration, value=0.1.8, time=482, desc=1 ticks), Event(type=Program, value=0, time=482, desc=483), Event(type=Pitch, value=64, time=482, desc=483), Event(type=Velocity, value=87, time=482, desc=87), Event(type=Duration, value=0.1.8, time=482, desc=1 ticks), Event(type=Position, value=3, time=483, desc=483), Event(type=Program, value=0, time=483, desc=484), Event(type=Pitch, value=60, time=483, desc=484), Event(type=Velocity, value=71, time=483, desc=71), Event(type=Duration, value=0.1.8, time=483, desc=1 ticks), Event(type=Position, value=4, time=484, desc=484), Event(type=Program, value=0, time=484, desc=485), Event(type=Pitch, value=74, time=484, desc=485), Event(type=Velocity, value=75, time=484, desc=75), Event(type=Duration, value=0.1.8, time=484, desc=1 ticks), Event(type=Position, value=5, time=485, desc=485), Event(type=Program, value=0, time=485, desc=486), Event(type=Pitch, value=72, time=485, desc=486), Event(type=Velocity, value=71, time=485, desc=71), Event(type=Duration, value=0.1.8, time=485, desc=1 ticks), Event(type=Program, value=0, time=485, desc=486), Event(type=Pitch, value=55, time=485, desc=486), Event(type=Velocity, value=75, time=485, desc=75), Event(type=Duration, value=0.1.8, time=485, desc=1 ticks), Event(type=Program, value=0, time=485, desc=486), Event(type=Pitch, value=62, time=485, desc=486), Event(type=Velocity, value=79, time=485, desc=79), Event(type=Duration, value=0.1.8, time=485, desc=1 ticks), Event(type=Position, value=6, time=486, desc=486), Event(type=Program, value=0, time=486, desc=487), Event(type=Pitch, value=71, time=486, desc=487), Event(type=Velocity, value=71, time=486, desc=71), Event(type=Duration, value=0.1.8, time=486, desc=1 ticks), Event(type=Position, value=8, time=488, desc=488), Event(type=Program, value=0, time=488, desc=489), Event(type=Pitch, value=72, time=488, desc=489), Event(type=Velocity, value=75, time=488, desc=75), Event(type=Duration, value=0.1.8, time=488, desc=1 ticks), Event(type=Program, value=0, time=488, desc=489), Event(type=Pitch, value=48, time=488, desc=489), Event(type=Velocity, value=75, time=488, desc=75), Event(type=Duration, value=0.1.8, time=488, desc=1 ticks), Event(type=Position, value=11, time=491, desc=491), Event(type=Program, value=0, time=491, desc=492), Event(type=Pitch, value=55, time=491, desc=492), Event(type=Velocity, value=75, time=491, desc=75), Event(type=Duration, value=0.1.8, time=491, desc=1 ticks), Event(type=Position, value=14, time=494, desc=494), Event(type=Program, value=0, time=494, desc=495), Event(type=Pitch, value=76, time=494, desc=495), Event(type=Velocity, value=75, time=494, desc=75), Event(type=Duration, value=0.1.8, time=494, desc=1 ticks), Event(type=Program, value=0, time=494, desc=495), Event(type=Pitch, value=60, time=494, desc=495), Event(type=Velocity, value=75, time=494, desc=75), Event(type=Duration, value=0.1.8, time=494, desc=1 ticks), Event(type=Position, value=15, time=495, desc=495), Event(type=Program, value=0, time=495, desc=496), Event(type=Pitch, value=77, time=495, desc=496), Event(type=Velocity, value=71, time=495, desc=71), Event(type=Duration, value=0.1.8, time=495, desc=1 ticks), Event(type=Position, value=17, time=497, desc=497), Event(type=Program, value=0, time=497, desc=498), Event(type=Pitch, value=66, time=497, desc=498), Event(type=Velocity, value=79, time=497, desc=79), Event(type=Duration, value=0.1.8, time=497, desc=1 ticks), Event(type=Program, value=0, time=497, desc=498), Event(type=Pitch, value=64, time=497, desc=498), Event(type=Velocity, value=59, time=497, desc=59), Event(type=Duration, value=0.1.8, time=497, desc=1 ticks), Event(type=Position, value=18, time=498, desc=498), Event(type=Program, value=0, time=498, desc=499), Event(type=Pitch, value=74, time=498, desc=499), Event(type=Velocity, value=71, time=498, desc=71), Event(type=Duration, value=0.1.8, time=498, desc=1 ticks), Event(type=Position, value=19, time=499, desc=499), Event(type=Program, value=0, time=499, desc=500), Event(type=Pitch, value=72, time=499, desc=500), Event(type=Velocity, value=75, time=499, desc=75), Event(type=Duration, value=0.1.8, time=499, desc=1 ticks), Event(type=Position, value=20, time=500, desc=500), Event(type=Program, value=0, time=500, desc=501), Event(type=Pitch, value=64, time=500, desc=501), Event(type=Velocity, value=67, time=500, desc=67), Event(type=Duration, value=0.1.8, time=500, desc=1 ticks), Event(type=Program, value=0, time=500, desc=501), Event(type=Pitch, value=67, time=500, desc=501), Event(type=Velocity, value=71, time=500, desc=71), Event(type=Duration, value=0.1.8, time=500, desc=1 ticks), Event(type=Position, value=21, time=501, desc=501), Event(type=Program, value=0, time=501, desc=502), Event(type=Pitch, value=74, time=501, desc=502), Event(type=Velocity, value=55, time=501, desc=55), Event(type=Duration, value=0.1.8, time=501, desc=1 ticks), Event(type=Position, value=23, time=503, desc=503), Event(type=Program, value=0, time=503, desc=504), Event(type=Pitch, value=76, time=503, desc=504), Event(type=Velocity, value=51, time=503, desc=51), Event(type=Duration, value=0.1.8, time=503, desc=1 ticks), Event(type=Program, value=0, time=503, desc=504), Event(type=Pitch, value=58, time=503, desc=504), Event(type=Velocity, value=55, time=503, desc=55), Event(type=Duration, value=0.1.8, time=503, desc=1 ticks), Event(type=Position, value=24, time=504, desc=504), Event(type=Program, value=0, time=504, desc=505), Event(type=Pitch, value=72, time=504, desc=505), Event(type=Velocity, value=59, time=504, desc=59), Event(type=Duration, value=0.1.8, time=504, desc=1 ticks), Event(type=Position, value=25, time=505, desc=505), Event(type=Program, value=0, time=505, desc=506), Event(type=Pitch, value=77, time=505, desc=506), Event(type=Velocity, value=79, time=505, desc=79), Event(type=Duration, value=0.1.8, time=505, desc=1 ticks), Event(type=Program, value=0, time=505, desc=506), Event(type=Pitch, value=57, time=505, desc=506), Event(type=Velocity, value=87, time=505, desc=87), Event(type=Duration, value=0.1.8, time=505, desc=1 ticks), Event(type=Position, value=27, time=507, desc=507), Event(type=Program, value=0, time=507, desc=509), Event(type=Pitch, value=79, time=507, desc=509), Event(type=Velocity, value=71, time=507, desc=71), Event(type=Duration, value=0.2.8, time=507, desc=2 ticks), Event(type=Position, value=28, time=508, desc=508), Event(type=Program, value=0, time=508, desc=509), Event(type=Pitch, value=77, time=508, desc=509), Event(type=Velocity, value=67, time=508, desc=67), Event(type=Duration, value=0.1.8, time=508, desc=1 ticks), Event(type=Program, value=0, time=508, desc=509), Event(type=Pitch, value=64, time=508, desc=509), Event(type=Velocity, value=75, time=508, desc=75), Event(type=Duration, value=0.1.8, time=508, desc=1 ticks), Event(type=Program, value=0, time=508, desc=509), Event(type=Pitch, value=62, time=508, desc=509), Event(type=Velocity, value=75, time=508, desc=75), Event(type=Duration, value=0.1.8, time=508, desc=1 ticks), Event(type=Position, value=30, time=510, desc=510), Event(type=Program, value=0, time=510, desc=511), Event(type=Pitch, value=76, time=510, desc=511), Event(type=Velocity, value=71, time=510, desc=71), Event(type=Duration, value=0.1.8, time=510, desc=1 ticks), Event(type=Position, value=31, time=511, desc=511), Event(type=Program, value=0, time=511, desc=512), Event(type=Pitch, value=74, time=511, desc=512), Event(type=Velocity, value=71, time=511, desc=71), Event(type=Duration, value=0.1.8, time=511, desc=1 ticks), Event(type=Program, value=0, time=511, desc=512), Event(type=Pitch, value=62, time=511, desc=512), Event(type=Velocity, value=75, time=511, desc=75), Event(type=Duration, value=0.1.8, time=511, desc=1 ticks), Event(type=Program, value=0, time=511, desc=512), Event(type=Pitch, value=65, time=511, desc=512), Event(type=Velocity, value=71, time=511, desc=71), Event(type=Duration, value=0.1.8, time=511, desc=1 ticks), Event(type=Bar, value=None, time=512, desc=0), Event(type=Position, value=0, time=512, desc=512), Event(type=Program, value=0, time=512, desc=513), Event(type=Pitch, value=76, time=512, desc=513), Event(type=Velocity, value=71, time=512, desc=71), Event(type=Duration, value=0.1.8, time=512, desc=1 ticks), Event(type=Position, value=2, time=514, desc=514), Event(type=Program, value=0, time=514, desc=515), Event(type=Pitch, value=77, time=514, desc=515), Event(type=Velocity, value=71, time=514, desc=71), Event(type=Duration, value=0.1.8, time=514, desc=1 ticks), Event(type=Program, value=0, time=514, desc=515), Event(type=Pitch, value=56, time=514, desc=515), Event(type=Velocity, value=67, time=514, desc=67), Event(type=Duration, value=0.1.8, time=514, desc=1 ticks), Event(type=Position, value=4, time=516, desc=516), Event(type=Program, value=0, time=516, desc=517), Event(type=Pitch, value=74, time=516, desc=517), Event(type=Velocity, value=59, time=516, desc=59), Event(type=Duration, value=0.1.8, time=516, desc=1 ticks), Event(type=Position, value=5, time=517, desc=517), Event(type=Program, value=0, time=517, desc=518), Event(type=Pitch, value=76, time=517, desc=518), Event(type=Velocity, value=79, time=517, desc=79), Event(type=Duration, value=0.1.8, time=517, desc=1 ticks), Event(type=Program, value=0, time=517, desc=518), Event(type=Pitch, value=55, time=517, desc=518), Event(type=Velocity, value=79, time=517, desc=79), Event(type=Duration, value=0.1.8, time=517, desc=1 ticks), Event(type=Position, value=6, time=518, desc=518), Event(type=Program, value=0, time=518, desc=520), Event(type=Pitch, value=77, time=518, desc=520), Event(type=Velocity, value=71, time=518, desc=71), Event(type=Duration, value=0.2.8, time=518, desc=2 ticks), Event(type=Position, value=7, time=519, desc=519), Event(type=Program, value=0, time=519, desc=520), Event(type=Pitch, value=76, time=519, desc=520), Event(type=Velocity, value=71, time=519, desc=71), Event(type=Duration, value=0.1.8, time=519, desc=1 ticks), Event(type=Position, value=8, time=520, desc=520), Event(type=Program, value=0, time=520, desc=521), Event(type=Pitch, value=60, time=520, desc=521), Event(type=Velocity, value=67, time=520, desc=67), Event(type=Duration, value=0.1.8, time=520, desc=1 ticks), Event(type=Program, value=0, time=520, desc=521), Event(type=Pitch, value=62, time=520, desc=521), Event(type=Velocity, value=79, time=520, desc=79), Event(type=Duration, value=0.1.8, time=520, desc=1 ticks), Event(type=Position, value=9, time=521, desc=521), Event(type=Program, value=0, time=521, desc=522), Event(type=Pitch, value=74, time=521, desc=522), Event(type=Velocity, value=71, time=521, desc=71), Event(type=Duration, value=0.1.8, time=521, desc=1 ticks), Event(type=Position, value=10, time=522, desc=522), Event(type=Program, value=0, time=522, desc=523), Event(type=Pitch, value=72, time=522, desc=523), Event(type=Velocity, value=71, time=522, desc=71), Event(type=Duration, value=0.1.8, time=522, desc=1 ticks), Event(type=Position, value=11, time=523, desc=523), Event(type=Program, value=0, time=523, desc=524), Event(type=Pitch, value=64, time=523, desc=524), Event(type=Velocity, value=71, time=523, desc=71), Event(type=Duration, value=0.1.8, time=523, desc=1 ticks), Event(type=Program, value=0, time=523, desc=524), Event(type=Pitch, value=60, time=523, desc=524), Event(type=Velocity, value=51, time=523, desc=51), Event(type=Duration, value=0.1.8, time=523, desc=1 ticks), Event(type=Position, value=12, time=524, desc=524), Event(type=Program, value=0, time=524, desc=525), Event(type=Pitch, value=74, time=524, desc=525), Event(type=Velocity, value=59, time=524, desc=59), Event(type=Duration, value=0.1.8, time=524, desc=1 ticks), Event(type=Position, value=13, time=525, desc=525), Event(type=Program, value=0, time=525, desc=526), Event(type=Pitch, value=76, time=525, desc=526), Event(type=Velocity, value=75, time=525, desc=75), Event(type=Duration, value=0.1.8, time=525, desc=1 ticks), Event(type=Program, value=0, time=525, desc=526), Event(type=Pitch, value=54, time=525, desc=526), Event(type=Velocity, value=75, time=525, desc=75), Event(type=Duration, value=0.1.8, time=525, desc=1 ticks), Event(type=Position, value=15, time=527, desc=527), Event(type=Program, value=0, time=527, desc=528), Event(type=Pitch, value=72, time=527, desc=528), Event(type=Velocity, value=71, time=527, desc=71), Event(type=Duration, value=0.1.8, time=527, desc=1 ticks), Event(type=Position, value=16, time=528, desc=528), Event(type=Program, value=0, time=528, desc=529), Event(type=Pitch, value=53, time=528, desc=529), Event(type=Velocity, value=79, time=528, desc=79), Event(type=Duration, value=0.1.8, time=528, desc=1 ticks), Event(type=Program, value=0, time=528, desc=529), Event(type=Pitch, value=74, time=528, desc=529), Event(type=Velocity, value=71, time=528, desc=71), Event(type=Duration, value=0.1.8, time=528, desc=1 ticks), Event(type=Position, value=17, time=529, desc=529), Event(type=Program, value=0, time=529, desc=531), Event(type=Pitch, value=76, time=529, desc=531), Event(type=Velocity, value=75, time=529, desc=75), Event(type=Duration, value=0.2.8, time=529, desc=2 ticks), Event(type=Position, value=19, time=531, desc=531), Event(type=Program, value=0, time=531, desc=532), Event(type=Pitch, value=74, time=531, desc=532), Event(type=Velocity, value=59, time=531, desc=59), Event(type=Duration, value=0.1.8, time=531, desc=1 ticks), Event(type=Program, value=0, time=531, desc=532), Event(type=Pitch, value=55, time=531, desc=532), Event(type=Velocity, value=67, time=531, desc=67), Event(type=Duration, value=0.1.8, time=531, desc=1 ticks), Event(type=Program, value=0, time=531, desc=532), Event(type=Pitch, value=59, time=531, desc=532), Event(type=Velocity, value=67, time=531, desc=67), Event(type=Duration, value=0.1.8, time=531, desc=1 ticks), Event(type=Position, value=20, time=532, desc=532), Event(type=Program, value=0, time=532, desc=533), Event(type=Pitch, value=71, time=532, desc=533), Event(type=Velocity, value=75, time=532, desc=75), Event(type=Duration, value=0.1.8, time=532, desc=1 ticks), Event(type=Position, value=22, time=534, desc=534), Event(type=Program, value=0, time=534, desc=535), Event(type=Pitch, value=67, time=534, desc=535), Event(type=Velocity, value=71, time=534, desc=71), Event(type=Duration, value=0.1.8, time=534, desc=1 ticks), Event(type=Program, value=0, time=534, desc=535), Event(type=Pitch, value=52, time=534, desc=535), Event(type=Velocity, value=75, time=534, desc=75), Event(type=Duration, value=0.1.8, time=534, desc=1 ticks), Event(type=Position, value=23, time=535, desc=535), Event(type=Program, value=0, time=535, desc=536), Event(type=Pitch, value=71, time=535, desc=536), Event(type=Velocity, value=75, time=535, desc=75), Event(type=Duration, value=0.1.8, time=535, desc=1 ticks), Event(type=Position, value=24, time=536, desc=536), Event(type=Program, value=0, time=536, desc=537), Event(type=Pitch, value=74, time=536, desc=537), Event(type=Velocity, value=67, time=536, desc=67), Event(type=Duration, value=0.1.8, time=536, desc=1 ticks), Event(type=Position, value=25, time=537, desc=537), Event(type=Program, value=0, time=537, desc=538), Event(type=Pitch, value=50, time=537, desc=538), Event(type=Velocity, value=55, time=537, desc=55), Event(type=Duration, value=0.1.8, time=537, desc=1 ticks), Event(type=Program, value=0, time=537, desc=538), Event(type=Pitch, value=48, time=537, desc=538), Event(type=Velocity, value=39, time=537, desc=39), Event(type=Duration, value=0.1.8, time=537, desc=1 ticks), Event(type=Position, value=26, time=538, desc=538), Event(type=Program, value=0, time=538, desc=539), Event(type=Pitch, value=79, time=538, desc=539), Event(type=Velocity, value=71, time=538, desc=71), Event(type=Duration, value=0.1.8, time=538, desc=1 ticks), Event(type=Position, value=28, time=540, desc=540), Event(type=Program, value=0, time=540, desc=541), Event(type=Pitch, value=48, time=540, desc=541), Event(type=Velocity, value=75, time=540, desc=75), Event(type=Duration, value=0.1.8, time=540, desc=1 ticks), Event(type=Program, value=0, time=540, desc=541), Event(type=Pitch, value=76, time=540, desc=541), Event(type=Velocity, value=79, time=540, desc=79), Event(type=Duration, value=0.1.8, time=540, desc=1 ticks), Event(type=Position, value=29, time=541, desc=541), Event(type=Program, value=0, time=541, desc=543), Event(type=Pitch, value=77, time=541, desc=543), Event(type=Velocity, value=71, time=541, desc=71), Event(type=Duration, value=0.2.8, time=541, desc=2 ticks), Event(type=Position, value=30, time=542, desc=542), Event(type=Program, value=0, time=542, desc=543), Event(type=Pitch, value=76, time=542, desc=543), Event(type=Velocity, value=59, time=542, desc=59), Event(type=Duration, value=0.1.8, time=542, desc=1 ticks), Event(type=Position, value=31, time=543, desc=543), Event(type=Program, value=0, time=543, desc=544), Event(type=Pitch, value=64, time=543, desc=544), Event(type=Velocity, value=71, time=543, desc=71), Event(type=Duration, value=0.1.8, time=543, desc=1 ticks), Event(type=Program, value=0, time=543, desc=544), Event(type=Pitch, value=67, time=543, desc=544), Event(type=Velocity, value=71, time=543, desc=71), Event(type=Duration, value=0.1.8, time=543, desc=1 ticks), Event(type=Bar, value=None, time=544, desc=0), Event(type=Position, value=0, time=544, desc=544), Event(type=Program, value=0, time=544, desc=545), Event(type=Pitch, value=74, time=544, desc=545), Event(type=Velocity, value=67, time=544, desc=67), Event(type=Duration, value=0.1.8, time=544, desc=1 ticks), Event(type=Position, value=1, time=545, desc=545), Event(type=Program, value=0, time=545, desc=546), Event(type=Pitch, value=72, time=545, desc=546), Event(type=Velocity, value=75, time=545, desc=75), Event(type=Duration, value=0.1.8, time=545, desc=1 ticks), Event(type=Program, value=0, time=545, desc=546), Event(type=Pitch, value=58, time=545, desc=546), Event(type=Velocity, value=79, time=545, desc=79), Event(type=Duration, value=0.1.8, time=545, desc=1 ticks), Event(type=Position, value=2, time=546, desc=546), Event(type=Program, value=0, time=546, desc=547), Event(type=Pitch, value=74, time=546, desc=547), Event(type=Velocity, value=75, time=546, desc=75), Event(type=Duration, value=0.1.8, time=546, desc=1 ticks), Event(type=Position, value=4, time=548, desc=548), Event(type=Program, value=0, time=548, desc=549), Event(type=Pitch, value=76, time=548, desc=549), Event(type=Velocity, value=71, time=548, desc=71), Event(type=Duration, value=0.1.8, time=548, desc=1 ticks), Event(type=Program, value=0, time=548, desc=549), Event(type=Pitch, value=64, time=548, desc=549), Event(type=Velocity, value=75, time=548, desc=75), Event(type=Duration, value=0.1.8, time=548, desc=1 ticks), Event(type=Program, value=0, time=548, desc=549), Event(type=Pitch, value=67, time=548, desc=549), Event(type=Velocity, value=67, time=548, desc=67), Event(type=Duration, value=0.1.8, time=548, desc=1 ticks), Event(type=Position, value=6, time=550, desc=550), Event(type=Program, value=0, time=550, desc=551), Event(type=Pitch, value=72, time=550, desc=551), Event(type=Velocity, value=75, time=550, desc=75), Event(type=Duration, value=0.1.8, time=550, desc=1 ticks), Event(type=Position, value=7, time=551, desc=551), Event(type=Program, value=0, time=551, desc=552), Event(type=Pitch, value=77, time=551, desc=552), Event(type=Velocity, value=79, time=551, desc=79), Event(type=Duration, value=0.1.8, time=551, desc=1 ticks), Event(type=Program, value=0, time=551, desc=552), Event(type=Pitch, value=57, time=551, desc=552), Event(type=Velocity, value=87, time=551, desc=87), Event(type=Duration, value=0.1.8, time=551, desc=1 ticks), Event(type=Position, value=8, time=552, desc=552), Event(type=Program, value=0, time=552, desc=554), Event(type=Pitch, value=79, time=552, desc=554), Event(type=Velocity, value=75, time=552, desc=75), Event(type=Duration, value=0.2.8, time=552, desc=2 ticks), Event(type=Position, value=10, time=554, desc=554), Event(type=Program, value=0, time=554, desc=556), Event(type=Pitch, value=77, time=554, desc=556), Event(type=Velocity, value=67, time=554, desc=67), Event(type=Duration, value=0.2.8, time=554, desc=2 ticks), Event(type=Program, value=0, time=554, desc=555), Event(type=Pitch, value=60, time=554, desc=555), Event(type=Velocity, value=67, time=554, desc=67), Event(type=Duration, value=0.1.8, time=554, desc=1 ticks), Event(type=Program, value=0, time=554, desc=555), Event(type=Pitch, value=65, time=554, desc=555), Event(type=Velocity, value=59, time=554, desc=59), Event(type=Duration, value=0.1.8, time=554, desc=1 ticks), Event(type=Position, value=11, time=555, desc=555), Event(type=Program, value=0, time=555, desc=556), Event(type=Pitch, value=76, time=555, desc=556), Event(type=Velocity, value=71, time=555, desc=71), Event(type=Duration, value=0.1.8, time=555, desc=1 ticks), Event(type=Position, value=12, time=556, desc=556), Event(type=Program, value=0, time=556, desc=557), Event(type=Pitch, value=74, time=556, desc=557), Event(type=Velocity, value=75, time=556, desc=75), Event(type=Duration, value=0.1.8, time=556, desc=1 ticks), Event(type=Position, value=13, time=557, desc=557), Event(type=Program, value=0, time=557, desc=558), Event(type=Pitch, value=56, time=557, desc=558), Event(type=Velocity, value=71, time=557, desc=71), Event(type=Duration, value=0.1.8, time=557, desc=1 ticks), Event(type=Position, value=14, time=558, desc=558), Event(type=Program, value=0, time=558, desc=559), Event(type=Pitch, value=76, time=558, desc=559), Event(type=Velocity, value=79, time=558, desc=79), Event(type=Duration, value=0.1.8, time=558, desc=1 ticks), Event(type=Position, value=15, time=559, desc=559), Event(type=Program, value=0, time=559, desc=560), Event(type=Pitch, value=77, time=559, desc=560), Event(type=Velocity, value=75, time=559, desc=75), Event(type=Duration, value=0.1.8, time=559, desc=1 ticks), Event(type=Position, value=16, time=560, desc=560), Event(type=Program, value=0, time=560, desc=561), Event(type=Pitch, value=59, time=560, desc=561), Event(type=Velocity, value=59, time=560, desc=59), Event(type=Duration, value=0.1.8, time=560, desc=1 ticks), Event(type=Program, value=0, time=560, desc=561), Event(type=Pitch, value=65, time=560, desc=561), Event(type=Velocity, value=59, time=560, desc=59), Event(type=Duration, value=0.1.8, time=560, desc=1 ticks), Event(type=Position, value=17, time=561, desc=561), Event(type=Program, value=0, time=561, desc=562), Event(type=Pitch, value=74, time=561, desc=562), Event(type=Velocity, value=71, time=561, desc=71), Event(type=Duration, value=0.1.8, time=561, desc=1 ticks), Event(type=Position, value=18, time=562, desc=562), Event(type=Program, value=0, time=562, desc=563), Event(type=Pitch, value=54, time=562, desc=563), Event(type=Velocity, value=79, time=562, desc=79), Event(type=Duration, value=0.1.8, time=562, desc=1 ticks), Event(type=Program, value=0, time=562, desc=563), Event(type=Pitch, value=76, time=562, desc=563), Event(type=Velocity, value=75, time=562, desc=75), Event(type=Duration, value=0.1.8, time=562, desc=1 ticks), Event(type=Position, value=20, time=564, desc=564), Event(type=Program, value=0, time=564, desc=566), Event(type=Pitch, value=77, time=564, desc=566), Event(type=Velocity, value=71, time=564, desc=71), Event(type=Duration, value=0.2.8, time=564, desc=2 ticks), Event(type=Position, value=21, time=565, desc=565), Event(type=Program, value=0, time=565, desc=566), Event(type=Pitch, value=76, time=565, desc=566), Event(type=Velocity, value=67, time=565, desc=67), Event(type=Duration, value=0.1.8, time=565, desc=1 ticks), Event(type=Program, value=0, time=565, desc=566), Event(type=Pitch, value=60, time=565, desc=566), Event(type=Velocity, value=67, time=565, desc=67), Event(type=Duration, value=0.1.8, time=565, desc=1 ticks), Event(type=Program, value=0, time=565, desc=566), Event(type=Pitch, value=64, time=565, desc=566), Event(type=Velocity, value=71, time=565, desc=71), Event(type=Duration, value=0.1.8, time=565, desc=1 ticks), Event(type=Position, value=23, time=567, desc=567), Event(type=Program, value=0, time=567, desc=568), Event(type=Pitch, value=72, time=567, desc=568), Event(type=Velocity, value=71, time=567, desc=71), Event(type=Duration, value=0.1.8, time=567, desc=1 ticks), Event(type=Position, value=24, time=568, desc=568), Event(type=Program, value=0, time=568, desc=569), Event(type=Pitch, value=71, time=568, desc=569), Event(type=Velocity, value=75, time=568, desc=75), Event(type=Duration, value=0.1.8, time=568, desc=1 ticks), Event(type=Program, value=0, time=568, desc=569), Event(type=Pitch, value=55, time=568, desc=569), Event(type=Velocity, value=75, time=568, desc=75), Event(type=Duration, value=0.1.8, time=568, desc=1 ticks), Event(type=Position, value=25, time=569, desc=569), Event(type=Program, value=0, time=569, desc=570), Event(type=Pitch, value=72, time=569, desc=570), Event(type=Velocity, value=71, time=569, desc=71), Event(type=Duration, value=0.1.8, time=569, desc=1 ticks), Event(type=Position, value=27, time=571, desc=571), Event(type=Program, value=0, time=571, desc=572), Event(type=Pitch, value=74, time=571, desc=572), Event(type=Velocity, value=71, time=571, desc=71), Event(type=Duration, value=0.1.8, time=571, desc=1 ticks), Event(type=Program, value=0, time=571, desc=572), Event(type=Pitch, value=62, time=571, desc=572), Event(type=Velocity, value=75, time=571, desc=75), Event(type=Duration, value=0.1.8, time=571, desc=1 ticks), Event(type=Program, value=0, time=571, desc=572), Event(type=Pitch, value=65, time=571, desc=572), Event(type=Velocity, value=79, time=571, desc=79), Event(type=Duration, value=0.1.8, time=571, desc=1 ticks), Event(type=Position, value=28, time=572, desc=572), Event(type=Program, value=0, time=572, desc=573), Event(type=Pitch, value=71, time=572, desc=573), Event(type=Velocity, value=67, time=572, desc=67), Event(type=Duration, value=0.1.8, time=572, desc=1 ticks), Event(type=Position, value=30, time=574, desc=574), Event(type=Program, value=0, time=574, desc=575), Event(type=Pitch, value=72, time=574, desc=575), Event(type=Velocity, value=71, time=574, desc=71), Event(type=Duration, value=0.1.8, time=574, desc=1 ticks), Event(type=Program, value=0, time=574, desc=575), Event(type=Pitch, value=64, time=574, desc=575), Event(type=Velocity, value=87, time=574, desc=87), Event(type=Duration, value=0.1.8, time=574, desc=1 ticks), Event(type=Program, value=0, time=574, desc=575), Event(type=Pitch, value=60, time=574, desc=575), Event(type=Velocity, value=71, time=574, desc=71), Event(type=Duration, value=0.1.8, time=574, desc=1 ticks), Event(type=Position, value=31, time=575, desc=575), Event(type=Program, value=0, time=575, desc=577), Event(type=Pitch, value=74, time=575, desc=577), Event(type=Velocity, value=67, time=575, desc=67), Event(type=Duration, value=0.2.8, time=575, desc=2 ticks), Event(type=Bar, value=None, time=576, desc=0), Event(type=Position, value=0, time=576, desc=576), Event(type=Program, value=0, time=576, desc=578), Event(type=Pitch, value=72, time=576, desc=578), Event(type=Velocity, value=75, time=576, desc=75), Event(type=Duration, value=0.2.8, time=576, desc=2 ticks), Event(type=Position, value=1, time=577, desc=577), Event(type=Program, value=0, time=577, desc=578), Event(type=Pitch, value=62, time=577, desc=578), Event(type=Velocity, value=71, time=577, desc=71), Event(type=Duration, value=0.1.8, time=577, desc=1 ticks), Event(type=Program, value=0, time=577, desc=578), Event(type=Pitch, value=55, time=577, desc=578), Event(type=Velocity, value=71, time=577, desc=71), Event(type=Duration, value=0.1.8, time=577, desc=1 ticks), Event(type=Position, value=2, time=578, desc=578), Event(type=Program, value=0, time=578, desc=579), Event(type=Pitch, value=71, time=578, desc=579), Event(type=Velocity, value=75, time=578, desc=75), Event(type=Duration, value=0.1.8, time=578, desc=1 ticks), Event(type=Position, value=4, time=580, desc=580), Event(type=Program, value=0, time=580, desc=581), Event(type=Pitch, value=72, time=580, desc=581), Event(type=Velocity, value=79, time=580, desc=79), Event(type=Duration, value=0.1.8, time=580, desc=1 ticks), Event(type=Program, value=0, time=580, desc=581), Event(type=Pitch, value=48, time=580, desc=581), Event(type=Velocity, value=79, time=580, desc=79), Event(type=Duration, value=0.1.8, time=580, desc=1 ticks), Event(type=Position, value=5, time=581, desc=581), Event(type=Program, value=0, time=581, desc=583), Event(type=Pitch, value=76, time=581, desc=583), Event(type=Velocity, value=91, time=581, desc=91), Event(type=Duration, value=0.2.8, time=581, desc=2 ticks), Event(type=Position, value=7, time=583, desc=583), Event(type=Program, value=0, time=583, desc=584), Event(type=Pitch, value=79, time=583, desc=584), Event(type=Velocity, value=87, time=583, desc=87), Event(type=Duration, value=0.1.8, time=583, desc=1 ticks), Event(type=Position, value=8, time=584, desc=584), Event(type=Program, value=0, time=584, desc=585), Event(type=Pitch, value=84, time=584, desc=585), Event(type=Velocity, value=79, time=584, desc=79), Event(type=Duration, value=0.1.8, time=584, desc=1 ticks), Event(type=Position, value=10, time=586, desc=586), Event(type=Program, value=0, time=586, desc=600), Event(type=Pitch, value=44, time=586, desc=600), Event(type=Velocity, value=95, time=586, desc=95), Event(type=Duration, value=1.6.8, time=586, desc=14 ticks), Event(type=Program, value=0, time=586, desc=600), Event(type=Pitch, value=87, time=586, desc=600), Event(type=Velocity, value=99, time=586, desc=99), Event(type=Duration, value=1.6.8, time=586, desc=14 ticks), Event(type=Program, value=0, time=586, desc=600), Event(type=Pitch, value=32, time=586, desc=600), Event(type=Velocity, value=75, time=586, desc=75), Event(type=Duration, value=1.6.8, time=586, desc=14 ticks), Event(type=Program, value=0, time=586, desc=594), Event(type=Pitch, value=80, time=586, desc=594), Event(type=Velocity, value=95, time=586, desc=95), Event(type=Duration, value=1.0.8, time=586, desc=8 ticks), Event(type=Position, value=12, time=588, desc=588), Event(type=Program, value=0, time=588, desc=593), Event(type=Pitch, value=84, time=588, desc=593), Event(type=Velocity, value=79, time=588, desc=79), Event(type=Duration, value=0.5.8, time=588, desc=5 ticks), Event(type=Program, value=0, time=588, desc=593), Event(type=Pitch, value=75, time=588, desc=593), Event(type=Velocity, value=79, time=588, desc=79), Event(type=Duration, value=0.5.8, time=588, desc=5 ticks), Event(type=Position, value=13, time=589, desc=589), Event(type=Program, value=0, time=589, desc=597), Event(type=Pitch, value=63, time=589, desc=597), Event(type=Velocity, value=79, time=589, desc=79), Event(type=Duration, value=1.0.8, time=589, desc=8 ticks), Event(type=Program, value=0, time=589, desc=594), Event(type=Pitch, value=72, time=589, desc=594), Event(type=Velocity, value=79, time=589, desc=79), Event(type=Duration, value=0.5.8, time=589, desc=5 ticks), Event(type=Position, value=15, time=591, desc=591), Event(type=Program, value=0, time=591, desc=596), Event(type=Pitch, value=68, time=591, desc=596), Event(type=Velocity, value=75, time=591, desc=75), Event(type=Duration, value=0.5.8, time=591, desc=5 ticks), Event(type=Program, value=0, time=591, desc=596), Event(type=Pitch, value=60, time=591, desc=596), Event(type=Velocity, value=71, time=591, desc=71), Event(type=Duration, value=0.5.8, time=591, desc=5 ticks), Event(type=Position, value=17, time=593, desc=593), Event(type=Program, value=0, time=593, desc=600), Event(type=Pitch, value=84, time=593, desc=600), Event(type=Velocity, value=95, time=593, desc=95), Event(type=Duration, value=0.7.8, time=593, desc=7 ticks), Event(type=Program, value=0, time=593, desc=600), Event(type=Pitch, value=75, time=593, desc=600), Event(type=Velocity, value=99, time=593, desc=99), Event(type=Duration, value=0.7.8, time=593, desc=7 ticks), Event(type=Position, value=18, time=594, desc=594), Event(type=Program, value=0, time=594, desc=600), Event(type=Pitch, value=80, time=594, desc=600), Event(type=Velocity, value=87, time=594, desc=87), Event(type=Duration, value=0.6.8, time=594, desc=6 ticks), Event(type=Program, value=0, time=594, desc=599), Event(type=Pitch, value=72, time=594, desc=599), Event(type=Velocity, value=79, time=594, desc=79), Event(type=Duration, value=0.5.8, time=594, desc=5 ticks), Event(type=Position, value=20, time=596, desc=596), Event(type=Program, value=0, time=596, desc=600), Event(type=Pitch, value=68, time=596, desc=600), Event(type=Velocity, value=75, time=596, desc=75), Event(type=Duration, value=0.4.8, time=596, desc=4 ticks), Event(type=Program, value=0, time=596, desc=599), Event(type=Pitch, value=60, time=596, desc=599), Event(type=Velocity, value=59, time=596, desc=59), Event(type=Duration, value=0.3.8, time=596, desc=3 ticks), Event(type=Position, value=21, time=597, desc=597), Event(type=Program, value=0, time=597, desc=600), Event(type=Pitch, value=56, time=597, desc=600), Event(type=Velocity, value=71, time=597, desc=71), Event(type=Duration, value=0.3.8, time=597, desc=3 ticks), Event(type=Program, value=0, time=597, desc=600), Event(type=Pitch, value=63, time=597, desc=600), Event(type=Velocity, value=59, time=597, desc=59), Event(type=Duration, value=0.3.8, time=597, desc=3 ticks), Event(type=Position, value=23, time=599, desc=599), Event(type=Program, value=0, time=599, desc=612), Event(type=Pitch, value=85, time=599, desc=612), Event(type=Velocity, value=95, time=599, desc=95), Event(type=Duration, value=1.5.8, time=599, desc=13 ticks), Event(type=Program, value=0, time=599, desc=606), Event(type=Pitch, value=76, time=599, desc=606), Event(type=Velocity, value=91, time=599, desc=91), Event(type=Duration, value=0.7.8, time=599, desc=7 ticks), Event(type=Position, value=24, time=600, desc=600), Event(type=Program, value=0, time=600, desc=605), Event(type=Pitch, value=82, time=600, desc=605), Event(type=Velocity, value=87, time=600, desc=87), Event(type=Duration, value=0.5.8, time=600, desc=5 ticks), Event(type=Position, value=25, time=601, desc=601), Event(type=Program, value=0, time=601, desc=606), Event(type=Pitch, value=73, time=601, desc=606), Event(type=Velocity, value=87, time=601, desc=87), Event(type=Duration, value=0.5.8, time=601, desc=5 ticks), Event(type=Position, value=26, time=602, desc=602), Event(type=Program, value=0, time=602, desc=609), Event(type=Pitch, value=64, time=602, desc=609), Event(type=Velocity, value=79, time=602, desc=79), Event(type=Duration, value=0.7.8, time=602, desc=7 ticks), Event(type=Position, value=27, time=603, desc=603), Event(type=Program, value=0, time=603, desc=ProgramChord), Event(type=Chord, value=min, time=603, desc=(np.int64(0), np.int64(3), np.int64(7))), Event(type=Program, value=0, time=603, desc=612), Event(type=Pitch, value=58, time=603, desc=612), Event(type=Velocity, value=71, time=603, desc=71), Event(type=Duration, value=1.1.8, time=603, desc=9 ticks), Event(type=Program, value=0, time=603, desc=608), Event(type=Pitch, value=61, time=603, desc=608), Event(type=Velocity, value=67, time=603, desc=67), Event(type=Duration, value=0.5.8, time=603, desc=5 ticks), Event(type=Position, value=28, time=604, desc=604), Event(type=Program, value=0, time=604, desc=612), Event(type=Pitch, value=65, time=604, desc=612), Event(type=Velocity, value=71, time=604, desc=71), Event(type=Duration, value=1.0.8, time=604, desc=8 ticks), Event(type=Position, value=29, time=605, desc=605), Event(type=Program, value=0, time=605, desc=612), Event(type=Pitch, value=82, time=605, desc=612), Event(type=Velocity, value=95, time=605, desc=95), Event(type=Duration, value=0.7.8, time=605, desc=7 ticks), Event(type=Program, value=0, time=605, desc=612), Event(type=Pitch, value=73, time=605, desc=612), Event(type=Velocity, value=95, time=605, desc=95), Event(type=Duration, value=0.7.8, time=605, desc=7 ticks), Event(type=Position, value=30, time=606, desc=606), Event(type=Program, value=0, time=606, desc=612), Event(type=Pitch, value=70, time=606, desc=612), Event(type=Velocity, value=91, time=606, desc=91), Event(type=Duration, value=0.6.8, time=606, desc=6 ticks), Event(type=Position, value=31, time=607, desc=607), Event(type=Program, value=0, time=607, desc=612), Event(type=Pitch, value=76, time=607, desc=612), Event(type=Velocity, value=71, time=607, desc=71), Event(type=Duration, value=0.5.8, time=607, desc=5 ticks), Event(type=Bar, value=None, time=608, desc=0), Event(type=Position, value=0, time=608, desc=608), Event(type=Program, value=0, time=608, desc=612), Event(type=Pitch, value=67, time=608, desc=612), Event(type=Velocity, value=79, time=608, desc=79), Event(type=Duration, value=0.4.8, time=608, desc=4 ticks), Event(type=Program, value=0, time=608, desc=612), Event(type=Pitch, value=61, time=608, desc=612), Event(type=Velocity, value=59, time=608, desc=59), Event(type=Duration, value=0.4.8, time=608, desc=4 ticks), Event(type=Position, value=1, time=609, desc=609), Event(type=Program, value=0, time=609, desc=612), Event(type=Pitch, value=64, time=609, desc=612), Event(type=Velocity, value=71, time=609, desc=71), Event(type=Duration, value=0.3.8, time=609, desc=3 ticks), Event(type=Position, value=4, time=612, desc=612), Event(type=Program, value=0, time=612, desc=619), Event(type=Pitch, value=75, time=612, desc=619), Event(type=Velocity, value=99, time=612, desc=99), Event(type=Duration, value=0.7.8, time=612, desc=7 ticks), Event(type=Program, value=0, time=612, desc=625), Event(type=Pitch, value=84, time=612, desc=625), Event(type=Velocity, value=99, time=612, desc=99), Event(type=Duration, value=1.5.8, time=612, desc=13 ticks), Event(type=Program, value=0, time=612, desc=617), Event(type=Pitch, value=72, time=612, desc=617), Event(type=Velocity, value=91, time=612, desc=91), Event(type=Duration, value=0.5.8, time=612, desc=5 ticks), Event(type=Position, value=5, time=613, desc=613), Event(type=Program, value=0, time=613, desc=618), Event(type=Pitch, value=80, time=613, desc=618), Event(type=Velocity, value=95, time=613, desc=95), Event(type=Duration, value=0.5.8, time=613, desc=5 ticks), Event(type=Position, value=7, time=615, desc=615), Event(type=Program, value=0, time=615, desc=622), Event(type=Pitch, value=60, time=615, desc=622), Event(type=Velocity, value=67, time=615, desc=67), Event(type=Duration, value=0.7.8, time=615, desc=7 ticks), Event(type=Program, value=0, time=615, desc=619), Event(type=Pitch, value=68, time=615, desc=619), Event(type=Velocity, value=79, time=615, desc=79), Event(type=Duration, value=0.4.8, time=615, desc=4 ticks), Event(type=Program, value=0, time=615, desc=620), Event(type=Pitch, value=56, time=615, desc=620), Event(type=Velocity, value=67, time=615, desc=67), Event(type=Duration, value=0.5.8, time=615, desc=5 ticks), Event(type=Program, value=0, time=615, desc=620), Event(type=Pitch, value=63, time=615, desc=620), Event(type=Velocity, value=71, time=615, desc=71), Event(type=Duration, value=0.5.8, time=615, desc=5 ticks), Event(type=Position, value=10, time=618, desc=618), Event(type=Program, value=0, time=618, desc=625), Event(type=Pitch, value=80, time=618, desc=625), Event(type=Velocity, value=95, time=618, desc=95), Event(type=Duration, value=0.7.8, time=618, desc=7 ticks), Event(type=Program, value=0, time=618, desc=625), Event(type=Pitch, value=72, time=618, desc=625), Event(type=Velocity, value=95, time=618, desc=95), Event(type=Duration, value=0.7.8, time=618, desc=7 ticks), Event(type=Position, value=11, time=619, desc=619), Event(type=Program, value=0, time=619, desc=625), Event(type=Pitch, value=68, time=619, desc=625), Event(type=Velocity, value=91, time=619, desc=91), Event(type=Duration, value=0.6.8, time=619, desc=6 ticks), Event(type=Program, value=0, time=619, desc=625), Event(type=Pitch, value=75, time=619, desc=625), Event(type=Velocity, value=87, time=619, desc=87), Event(type=Duration, value=0.6.8, time=619, desc=6 ticks), Event(type=Position, value=13, time=621, desc=621), Event(type=Program, value=0, time=621, desc=625), Event(type=Pitch, value=63, time=621, desc=625), Event(type=Velocity, value=75, time=621, desc=75), Event(type=Duration, value=0.4.8, time=621, desc=4 ticks), Event(type=Program, value=0, time=621, desc=625), Event(type=Pitch, value=56, time=621, desc=625), Event(type=Velocity, value=71, time=621, desc=71), Event(type=Duration, value=0.4.8, time=621, desc=4 ticks), Event(type=Position, value=14, time=622, desc=622), Event(type=Program, value=0, time=622, desc=625), Event(type=Pitch, value=60, time=622, desc=625), Event(type=Velocity, value=67, time=622, desc=67), Event(type=Duration, value=0.3.8, time=622, desc=3 ticks), Event(type=Program, value=0, time=622, desc=625), Event(type=Pitch, value=51, time=622, desc=625), Event(type=Velocity, value=67, time=622, desc=67), Event(type=Duration, value=0.3.8, time=622, desc=3 ticks), Event(type=Position, value=16, time=624, desc=624), Event(type=Program, value=0, time=624, desc=631), Event(type=Pitch, value=73, time=624, desc=631), Event(type=Velocity, value=99, time=624, desc=99), Event(type=Duration, value=0.7.8, time=624, desc=7 ticks), Event(type=Program, value=0, time=624, desc=636), Event(type=Pitch, value=82, time=624, desc=636), Event(type=Velocity, value=107, time=624, desc=107), Event(type=Duration, value=1.4.8, time=624, desc=12 ticks), Event(type=Position, value=17, time=625, desc=625), Event(type=Program, value=0, time=625, desc=630), Event(type=Pitch, value=75, time=625, desc=630), Event(type=Velocity, value=87, time=625, desc=87), Event(type=Duration, value=0.5.8, time=625, desc=5 ticks), Event(type=Program, value=0, time=625, desc=630), Event(type=Pitch, value=70, time=625, desc=630), Event(type=Velocity, value=87, time=625, desc=87), Event(type=Duration, value=0.5.8, time=625, desc=5 ticks), Event(type=Position, value=19, time=627, desc=627), Event(type=Program, value=0, time=627, desc=634), Event(type=Pitch, value=58, time=627, desc=634), Event(type=Velocity, value=91, time=627, desc=91), Event(type=Duration, value=0.7.8, time=627, desc=7 ticks), Event(type=Position, value=20, time=628, desc=628), Event(type=Program, value=0, time=628, desc=636), Event(type=Pitch, value=55, time=628, desc=636), Event(type=Velocity, value=79, time=628, desc=79), Event(type=Duration, value=1.0.8, time=628, desc=8 ticks), Event(type=Program, value=0, time=628, desc=636), Event(type=Pitch, value=60, time=628, desc=636), Event(type=Velocity, value=75, time=628, desc=75), Event(type=Duration, value=1.0.8, time=628, desc=8 ticks), Event(type=Program, value=0, time=628, desc=636), Event(type=Pitch, value=65, time=628, desc=636), Event(type=Velocity, value=59, time=628, desc=59), Event(type=Duration, value=1.0.8, time=628, desc=8 ticks), Event(type=Position, value=22, time=630, desc=630), Event(type=Program, value=0, time=630, desc=636), Event(type=Pitch, value=75, time=630, desc=636), Event(type=Velocity, value=95, time=630, desc=95), Event(type=Duration, value=0.6.8, time=630, desc=6 ticks), Event(type=Program, value=0, time=630, desc=636), Event(type=Pitch, value=70, time=630, desc=636), Event(type=Velocity, value=95, time=630, desc=95), Event(type=Duration, value=0.6.8, time=630, desc=6 ticks), Event(type=Position, value=23, time=631, desc=631), Event(type=Program, value=0, time=631, desc=634), Event(type=Pitch, value=63, time=631, desc=634), Event(type=Velocity, value=95, time=631, desc=95), Event(type=Duration, value=0.3.8, time=631, desc=3 ticks), Event(type=Program, value=0, time=631, desc=636), Event(type=Pitch, value=73, time=631, desc=636), Event(type=Velocity, value=87, time=631, desc=87), Event(type=Duration, value=0.5.8, time=631, desc=5 ticks), Event(type=Position, value=25, time=633, desc=633), Event(type=Program, value=0, time=633, desc=636), Event(type=Pitch, value=61, time=633, desc=636), Event(type=Velocity, value=87, time=633, desc=87), Event(type=Duration, value=0.3.8, time=633, desc=3 ticks), Event(type=Program, value=0, time=633, desc=635), Event(type=Pitch, value=63, time=633, desc=635), Event(type=Velocity, value=75, time=633, desc=75), Event(type=Duration, value=0.2.8, time=633, desc=2 ticks), Event(type=Position, value=26, time=634, desc=634), Event(type=Program, value=0, time=634, desc=636), Event(type=Pitch, value=67, time=634, desc=636), Event(type=Velocity, value=39, time=634, desc=39), Event(type=Duration, value=0.2.8, time=634, desc=2 ticks), Event(type=Program, value=0, time=634, desc=636), Event(type=Pitch, value=58, time=634, desc=636), Event(type=Velocity, value=51, time=634, desc=51), Event(type=Duration, value=0.2.8, time=634, desc=2 ticks), Event(type=Position, value=28, time=636, desc=636), Event(type=Program, value=0, time=636, desc=649), Event(type=Pitch, value=87, time=636, desc=649), Event(type=Velocity, value=99, time=636, desc=99), Event(type=Duration, value=1.5.8, time=636, desc=13 ticks), Event(type=Program, value=0, time=636, desc=643), Event(type=Pitch, value=80, time=636, desc=643), Event(type=Velocity, value=95, time=636, desc=95), Event(type=Duration, value=0.7.8, time=636, desc=7 ticks), Event(type=Program, value=0, time=636, desc=648), Event(type=Pitch, value=44, time=636, desc=648), Event(type=Velocity, value=87, time=636, desc=87), Event(type=Duration, value=1.4.8, time=636, desc=12 ticks), Event(type=Program, value=0, time=636, desc=646), Event(type=Pitch, value=56, time=636, desc=646), Event(type=Velocity, value=91, time=636, desc=91), Event(type=Duration, value=1.2.8, time=636, desc=10 ticks), Event(type=Position, value=29, time=637, desc=637), Event(type=Program, value=0, time=637, desc=642), Event(type=Pitch, value=84, time=637, desc=642), Event(type=Velocity, value=79, time=637, desc=79), Event(type=Duration, value=0.5.8, time=637, desc=5 ticks), Event(type=Program, value=0, time=637, desc=642), Event(type=Pitch, value=75, time=637, desc=642), Event(type=Velocity, value=91, time=637, desc=91), Event(type=Duration, value=0.5.8, time=637, desc=5 ticks), Event(type=Bar, value=None, time=640, desc=0), Event(type=Position, value=0, time=640, desc=640), Event(type=Program, value=0, time=640, desc=647), Event(type=Pitch, value=63, time=640, desc=647), Event(type=Velocity, value=91, time=640, desc=91), Event(type=Duration, value=0.7.8, time=640, desc=7 ticks), Event(type=Program, value=0, time=640, desc=644), Event(type=Pitch, value=72, time=640, desc=644), Event(type=Velocity, value=87, time=640, desc=87), Event(type=Duration, value=0.4.8, time=640, desc=4 ticks), Event(type=Position, value=1, time=641, desc=641), Event(type=Program, value=0, time=641, desc=646), Event(type=Pitch, value=68, time=641, desc=646), Event(type=Velocity, value=75, time=641, desc=75), Event(type=Duration, value=0.5.8, time=641, desc=5 ticks), Event(type=Program, value=0, time=641, desc=645), Event(type=Pitch, value=60, time=641, desc=645), Event(type=Velocity, value=79, time=641, desc=79), Event(type=Duration, value=0.4.8, time=641, desc=4 ticks), Event(type=Position, value=2, time=642, desc=642), Event(type=Program, value=0, time=642, desc=648), Event(type=Pitch, value=84, time=642, desc=648), Event(type=Velocity, value=95, time=642, desc=95), Event(type=Duration, value=0.6.8, time=642, desc=6 ticks), Event(type=Program, value=0, time=642, desc=648), Event(type=Pitch, value=75, time=642, desc=648), Event(type=Velocity, value=99, time=642, desc=99), Event(type=Duration, value=0.6.8, time=642, desc=6 ticks), Event(type=Position, value=3, time=643, desc=643), Event(type=Program, value=0, time=643, desc=648), Event(type=Pitch, value=80, time=643, desc=648), Event(type=Velocity, value=87, time=643, desc=87), Event(type=Duration, value=0.5.8, time=643, desc=5 ticks), Event(type=Position, value=4, time=644, desc=644), Event(type=Program, value=0, time=644, desc=649), Event(type=Pitch, value=72, time=644, desc=649), Event(type=Velocity, value=79, time=644, desc=79), Event(type=Duration, value=0.5.8, time=644, desc=5 ticks), Event(type=Position, value=5, time=645, desc=645), Event(type=Program, value=0, time=645, desc=649), Event(type=Pitch, value=60, time=645, desc=649), Event(type=Velocity, value=75, time=645, desc=75), Event(type=Duration, value=0.4.8, time=645, desc=4 ticks), Event(type=Program, value=0, time=645, desc=648), Event(type=Pitch, value=68, time=645, desc=648), Event(type=Velocity, value=79, time=645, desc=79), Event(type=Duration, value=0.3.8, time=645, desc=3 ticks), Event(type=Position, value=6, time=646, desc=646), Event(type=Program, value=0, time=646, desc=648), Event(type=Pitch, value=56, time=646, desc=648), Event(type=Velocity, value=75, time=646, desc=75), Event(type=Duration, value=0.2.8, time=646, desc=2 ticks), Event(type=Position, value=7, time=647, desc=647), Event(type=Program, value=0, time=647, desc=649), Event(type=Pitch, value=63, time=647, desc=649), Event(type=Velocity, value=67, time=647, desc=67), Event(type=Duration, value=0.2.8, time=647, desc=2 ticks), Event(type=Position, value=8, time=648, desc=648), Event(type=Program, value=0, time=648, desc=661), Event(type=Pitch, value=85, time=648, desc=661), Event(type=Velocity, value=95, time=648, desc=95), Event(type=Duration, value=1.5.8, time=648, desc=13 ticks), Event(type=Program, value=0, time=648, desc=655), Event(type=Pitch, value=76, time=648, desc=655), Event(type=Velocity, value=95, time=648, desc=95), Event(type=Duration, value=0.7.8, time=648, desc=7 ticks), Event(type=Position, value=9, time=649, desc=649), Event(type=Program, value=0, time=649, desc=654), Event(type=Pitch, value=82, time=649, desc=654), Event(type=Velocity, value=91, time=649, desc=91), Event(type=Duration, value=0.5.8, time=649, desc=5 ticks), Event(type=Position, value=10, time=650, desc=650), Event(type=Program, value=0, time=650, desc=655), Event(type=Pitch, value=73, time=650, desc=655), Event(type=Velocity, value=91, time=650, desc=91), Event(type=Duration, value=0.5.8, time=650, desc=5 ticks), Event(type=Position, value=11, time=651, desc=651), Event(type=Program, value=0, time=651, desc=658), Event(type=Pitch, value=64, time=651, desc=658), Event(type=Velocity, value=79, time=651, desc=79), Event(type=Duration, value=0.7.8, time=651, desc=7 ticks), Event(type=Position, value=12, time=652, desc=652), Event(type=Program, value=0, time=652, desc=662), Event(type=Pitch, value=58, time=652, desc=662), Event(type=Velocity, value=79, time=652, desc=79), Event(type=Duration, value=1.2.8, time=652, desc=10 ticks), Event(type=Program, value=0, time=652, desc=657), Event(type=Pitch, value=61, time=652, desc=657), Event(type=Velocity, value=75, time=652, desc=75), Event(type=Duration, value=0.5.8, time=652, desc=5 ticks), Event(type=Position, value=13, time=653, desc=653), Event(type=Program, value=0, time=653, desc=661), Event(type=Pitch, value=65, time=653, desc=661), Event(type=Velocity, value=75, time=653, desc=75), Event(type=Duration, value=1.0.8, time=653, desc=8 ticks), Event(type=Position, value=15, time=655, desc=655), Event(type=Program, value=0, time=655, desc=662), Event(type=Pitch, value=82, time=655, desc=662), Event(type=Velocity, value=99, time=655, desc=99), Event(type=Duration, value=0.7.8, time=655, desc=7 ticks), Event(type=Program, value=0, time=655, desc=662), Event(type=Pitch, value=73, time=655, desc=662), Event(type=Velocity, value=95, time=655, desc=95), Event(type=Duration, value=0.7.8, time=655, desc=7 ticks), Event(type=Position, value=16, time=656, desc=656), Event(type=Program, value=0, time=656, desc=662), Event(type=Pitch, value=76, time=656, desc=662), Event(type=Velocity, value=79, time=656, desc=79), Event(type=Duration, value=0.6.8, time=656, desc=6 ticks), Event(type=Program, value=0, time=656, desc=662), Event(type=Pitch, value=70, time=656, desc=662), Event(type=Velocity, value=87, time=656, desc=87), Event(type=Duration, value=0.6.8, time=656, desc=6 ticks), Event(type=Position, value=17, time=657, desc=657), Event(type=Program, value=0, time=657, desc=661), Event(type=Pitch, value=67, time=657, desc=661), Event(type=Velocity, value=75, time=657, desc=75), Event(type=Duration, value=0.4.8, time=657, desc=4 ticks), Event(type=Position, value=18, time=658, desc=658), Event(type=Program, value=0, time=658, desc=662), Event(type=Pitch, value=61, time=658, desc=662), Event(type=Velocity, value=67, time=658, desc=67), Event(type=Duration, value=0.4.8, time=658, desc=4 ticks), Event(type=Program, value=0, time=658, desc=661), Event(type=Pitch, value=64, time=658, desc=661), Event(type=Velocity, value=71, time=658, desc=71), Event(type=Duration, value=0.3.8, time=658, desc=3 ticks), Event(type=Position, value=21, time=661, desc=661), Event(type=Program, value=0, time=661, desc=686), Event(type=Pitch, value=84, time=661, desc=686), Event(type=Velocity, value=95, time=661, desc=95), Event(type=Duration, value=3.1.8, time=661, desc=25 ticks), Event(type=Program, value=0, time=661, desc=668), Event(type=Pitch, value=75, time=661, desc=668), Event(type=Velocity, value=95, time=661, desc=95), Event(type=Duration, value=0.7.8, time=661, desc=7 ticks), Event(type=Position, value=22, time=662, desc=662), Event(type=Program, value=0, time=662, desc=673), Event(type=Pitch, value=80, time=662, desc=673), Event(type=Velocity, value=91, time=662, desc=91), Event(type=Duration, value=1.3.8, time=662, desc=11 ticks), Event(type=Program, value=0, time=662, desc=685), Event(type=Pitch, value=72, time=662, desc=685), Event(type=Velocity, value=91, time=662, desc=91), Event(type=Duration, value=2.7.8, time=662, desc=23 ticks), Event(type=Position, value=24, time=664, desc=664), Event(type=Program, value=0, time=664, desc=686), Event(type=Pitch, value=60, time=664, desc=686), Event(type=Velocity, value=67, time=664, desc=67), Event(type=Duration, value=2.6.8, time=664, desc=22 ticks), Event(type=Program, value=0, time=664, desc=674), Event(type=Pitch, value=68, time=664, desc=674), Event(type=Velocity, value=71, time=664, desc=71), Event(type=Duration, value=1.2.8, time=664, desc=10 ticks), Event(type=Position, value=25, time=665, desc=665), Event(type=Program, value=0, time=665, desc=677), Event(type=Pitch, value=56, time=665, desc=677), Event(type=Velocity, value=67, time=665, desc=67), Event(type=Duration, value=1.4.8, time=665, desc=12 ticks), Event(type=Program, value=0, time=665, desc=670), Event(type=Pitch, value=63, time=665, desc=670), Event(type=Velocity, value=71, time=665, desc=71), Event(type=Duration, value=0.5.8, time=665, desc=5 ticks), Event(type=Position, value=27, time=667, desc=667), Event(type=Program, value=0, time=667, desc=685), Event(type=Pitch, value=73, time=667, desc=685), Event(type=Velocity, value=91, time=667, desc=91), Event(type=Duration, value=2.2.8, time=667, desc=18 ticks), Event(type=Program, value=0, time=667, desc=682), Event(type=Pitch, value=82, time=667, desc=682), Event(type=Velocity, value=99, time=667, desc=99), Event(type=Duration, value=1.7.8, time=667, desc=15 ticks), Event(type=Position, value=28, time=668, desc=668), Event(type=Program, value=0, time=668, desc=686), Event(type=Pitch, value=75, time=668, desc=686), Event(type=Velocity, value=87, time=668, desc=87), Event(type=Duration, value=2.2.8, time=668, desc=18 ticks), Event(type=Program, value=0, time=668, desc=683), Event(type=Pitch, value=70, time=668, desc=683), Event(type=Velocity, value=87, time=668, desc=87), Event(type=Duration, value=1.7.8, time=668, desc=15 ticks), Event(type=Position, value=30, time=670, desc=670), Event(type=Program, value=0, time=670, desc=686), Event(type=Pitch, value=55, time=670, desc=686), Event(type=Velocity, value=47, time=670, desc=47), Event(type=Duration, value=2.0.8, time=670, desc=16 ticks), Event(type=Program, value=0, time=670, desc=686), Event(type=Pitch, value=63, time=670, desc=686), Event(type=Velocity, value=71, time=670, desc=71), Event(type=Duration, value=2.0.8, time=670, desc=16 ticks), Event(type=Position, value=31, time=671, desc=671), Event(type=Program, value=0, time=671, desc=686), Event(type=Pitch, value=58, time=671, desc=686), Event(type=Velocity, value=67, time=671, desc=67), Event(type=Duration, value=1.7.8, time=671, desc=15 ticks), Event(type=Program, value=0, time=671, desc=683), Event(type=Pitch, value=51, time=671, desc=683), Event(type=Velocity, value=67, time=671, desc=67), Event(type=Duration, value=1.4.8, time=671, desc=12 ticks), Event(type=Bar, value=None, time=672, desc=0), Event(type=Position, value=1, time=673, desc=673), Event(type=Program, value=0, time=673, desc=679), Event(type=Pitch, value=80, time=673, desc=679), Event(type=Velocity, value=91, time=673, desc=91), Event(type=Duration, value=0.6.8, time=673, desc=6 ticks), Event(type=Position, value=3, time=675, desc=675), Event(type=Program, value=0, time=675, desc=681), Event(type=Pitch, value=68, time=675, desc=681), Event(type=Velocity, value=79, time=675, desc=79), Event(type=Duration, value=0.6.8, time=675, desc=6 ticks), Event(type=Position, value=4, time=676, desc=676), Event(type=Program, value=0, time=676, desc=685), Event(type=Pitch, value=56, time=676, desc=685), Event(type=Velocity, value=71, time=676, desc=71), Event(type=Duration, value=1.1.8, time=676, desc=9 ticks), Event(type=Program, value=0, time=676, desc=685), Event(type=Pitch, value=79, time=676, desc=685), Event(type=Velocity, value=87, time=676, desc=87), Event(type=Duration, value=1.1.8, time=676, desc=9 ticks), Event(type=Position, value=6, time=678, desc=678), Event(type=Program, value=0, time=678, desc=686), Event(type=Pitch, value=44, time=678, desc=686), Event(type=Velocity, value=79, time=678, desc=79), Event(type=Duration, value=1.0.8, time=678, desc=8 ticks), Event(type=Program, value=0, time=678, desc=686), Event(type=Pitch, value=67, time=678, desc=686), Event(type=Velocity, value=79, time=678, desc=79), Event(type=Duration, value=1.0.8, time=678, desc=8 ticks), Event(type=Position, value=8, time=680, desc=680), Event(type=Program, value=0, time=680, desc=686), Event(type=Pitch, value=80, time=680, desc=686), Event(type=Velocity, value=91, time=680, desc=91), Event(type=Duration, value=0.6.8, time=680, desc=6 ticks), Event(type=Position, value=9, time=681, desc=681), Event(type=Program, value=0, time=681, desc=686), Event(type=Pitch, value=68, time=681, desc=686), Event(type=Velocity, value=79, time=681, desc=79), Event(type=Duration, value=0.5.8, time=681, desc=5 ticks), Event(type=Position, value=10, time=682, desc=682), Event(type=Program, value=0, time=682, desc=685), Event(type=Pitch, value=82, time=682, desc=685), Event(type=Velocity, value=87, time=682, desc=87), Event(type=Duration, value=0.3.8, time=682, desc=3 ticks), Event(type=Position, value=11, time=683, desc=683), Event(type=Program, value=0, time=683, desc=686), Event(type=Pitch, value=51, time=683, desc=686), Event(type=Velocity, value=71, time=683, desc=71), Event(type=Duration, value=0.3.8, time=683, desc=3 ticks), Event(type=Program, value=0, time=683, desc=685), Event(type=Pitch, value=70, time=683, desc=685), Event(type=Velocity, value=87, time=683, desc=87), Event(type=Duration, value=0.2.8, time=683, desc=2 ticks), Event(type=Position, value=12, time=684, desc=684), Event(type=Program, value=0, time=684, desc=686), Event(type=Pitch, value=39, time=684, desc=686), Event(type=Velocity, value=67, time=684, desc=67), Event(type=Duration, value=0.2.8, time=684, desc=2 ticks)], 'are_ids_encoded': True, '_ticks_bars': [0, 32, 64, 96, 128, 160, 192, 224, 256, 288, 320, 352, 384, 416, 448, 480, 512, 544, 576, 608, 640, 672], '_ticks_beats': [0, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96, 104, 112, 120, 128, 136, 144, 152, 160, 168, 176, 184, 192, 200, 208, 216, 224, 232, 240, 248, 256, 264, 272, 280, 288, 296, 304, 312, 320, 328, 336, 344, 352, 360, 368, 376, 384, 392, 400, 408, 416, 424, 432, 440, 448, 456, 464, 472, 480, 488, 496, 504, 512, 520, 528, 536, 544, 552, 560, 568, 576, 584, 592, 600, 608, 616, 624, 632, 640, 648, 656, 664, 672, 680], '_ids_decoded': []}



In [ ]:
['Bar_None', 'Position_0', 'Tempo_121.29', 'Position_13', 'Program_0', 'Pitch_43', 'Velocity_79', 'Duration_0.7.8', 'Program_0', 'Pitch_36', 'Velocity_79', 'Duration_0.7.8', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_79', 'Velocity_99', 'Duration_0.5.8', 'Program_0', 'Pitch_67', 'Velocity_99', 'Duration_0.7.8', 'Program_0', 'Pitch_76', 'Velocity_87', 'Duration_0.6.8', 'Position_18', 'Program_0', 'Pitch_79', 'Velocity_91', 'Duration_1.0.8', 'Position_20', 'Program_0', 'Pitch_76', 'Velocity_91', 'Duration_0.5.8', 'Program_0', 'Pitch_58', 'Velocity_75', 'Duration_0.7.8', 'Program_0', 'Pitch_61', 'Velocity_79', 'Duration_0.7.8', 'Program_0', 'Pitch_73', 'Velocity_79', 'Duration_0.7.8', 'Program_0', 'Pitch_67', 'Velocity_91', 'Duration_0.7.8', 'Position_23', 'Program_0', 'Pitch_70', 'Velocity_51', 'Duration_0.4.8', 'Position_24', 'Program_0', 'Pitch_76', 'Velocity_75', 'Duration_0.2.8', 'Position_26', 'Program_0', 'Pitch_57', 'Velocity_79', 'Duration_0.7.8', 'Program_0', 'Pitch_77', 'Velocity_91', 'Duration_0.5.8', 'Program_0', 'Pitch_65', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_60', 'Velocity_75', 'Duration_0.6.8', 'Position_31', 'Program_0', 'Pitch_77', 'Velocity_79', 'Duration_0.2.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_56', 'Velocity_71', 'Duration_0.7.8', 'Program_0', 'Pitch_71', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_74', 'Velocity_91', 'Duration_0.5.8', 'Program_0', 'Pitch_65', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_59', 'Velocity_75', 'Duration_0.6.8', 'Position_5', 'Program_0', 'Pitch_74', 'Velocity_79', 'Duration_0.2.8', 'Position_6', 'Program_0', 'Pitch_76', 'Velocity_91', 'Duration_0.5.8', 'Program_0', 'Pitch_58', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_64', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_55', 'Velocity_75', 'Duration_0.7.8', 'Program_0', 'Pitch_70', 'Velocity_91', 'Duration_0.7.8', 'Position_11', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_0.2.8', 'Position_12', 'Program_0', 'Pitch_57', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_72', 'Velocity_95', 'Duration_0.4.8', 'Program_0', 'Pitch_54', 'Velocity_75', 'Duration_0.6.8', 'Program_0', 'Pitch_63', 'Velocity_87', 'Duration_0.6.8', 'Position_13', 'Program_0', 'Pitch_69', 'Velocity_75', 'Duration_0.6.8', 'Program_0', 'Pitch_55', 'Velocity_47', 'Duration_0.5.8', 'Position_17', 'Program_0', 'Pitch_72', 'Velocity_79', 'Duration_0.2.8', 'Position_18', 'Program_0', 'Pitch_68', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_56', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_74', 'Velocity_91', 'Duration_0.5.8', 'Program_0', 'Pitch_62', 'Velocity_95', 'Duration_0.6.8', 'Position_19', 'Program_0', 'Pitch_53', 'Velocity_75', 'Duration_0.6.8', 'Position_23', 'Program_0', 'Pitch_74', 'Velocity_79', 'Duration_0.2.8', 'Position_24', 'Program_0', 'Pitch_67', 'Velocity_87', 'Duration_0.5.8', 'Position_25', 'Program_0', 'Pitch_62', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_55', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_59', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_65', 'Velocity_79', 'Duration_0.6.8', 'Position_29', 'Program_0', 'Pitch_67', 'Velocity_79', 'Duration_0.2.8', 'Position_31', 'Program_0', 'Pitch_43', 'Velocity_95', 'Duration_0.7.8', 'Program_0', 'Pitch_36', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_79', 'Velocity_99', 'Duration_0.5.8', 'Program_0', 'Pitch_67', 'Velocity_99', 'Duration_0.6.8', 'Program_0', 'Pitch_76', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_72', 'Velocity_95', 'Duration_0.6.8', 'Bar_None', 'Position_4', 'Program_0', 'Pitch_79', 'Velocity_91', 'Duration_1.0.8', 'Position_5', 'Program_0', 'Pitch_58', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_76', 'Velocity_95', 'Duration_0.4.8', 'Program_0', 'Pitch_67', 'Velocity_95', 'Duration_0.7.8', 'Program_0', 'Pitch_73', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_61', 'Velocity_91', 'Duration_0.6.8', 'Position_10', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_0.2.8', 'Position_11', 'Program_0', 'Pitch_77', 'Velocity_91', 'Duration_0.5.8', 'Position_12', 'Program_0', 'Pitch_60', 'Velocity_79', 'Duration_0.7.8', 'Program_0', 'Pitch_57', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_65', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.7.8', 'Position_16', 'Program_0', 'Pitch_77', 'Velocity_79', 'Duration_0.2.8', 'Position_17', 'Program_0', 'Pitch_56', 'Velocity_75', 'Duration_0.7.8', 'Program_0', 'Pitch_59', 'Velocity_87', 'Duration_0.7.8', 'Position_18', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.4.8', 'Program_0', 'Pitch_71', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_65', 'Velocity_91', 'Duration_0.7.8', 'Position_22', 'Program_0', 'Pitch_74', 'Velocity_79', 'Duration_0.2.8', 'Position_24', 'Program_0', 'Pitch_64', 'Velocity_95', 'Duration_0.7.8', 'Program_0', 'Pitch_76', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_55', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_60', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_67', 'Velocity_91', 'Duration_0.6.8', 'Position_28', 'Program_0', 'Pitch_72', 'Velocity_79', 'Duration_0.2.8', 'Position_30', 'Program_0', 'Pitch_67', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_62', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_55', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_71', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_65', 'Velocity_91', 'Duration_0.6.8', 'Bar_None', 'Position_2', 'Program_0', 'Pitch_74', 'Velocity_75', 'Duration_0.2.8', 'Position_3', 'Program_0', 'Pitch_64', 'Velocity_91', 'Duration_0.1.8', 'Position_4', 'Program_0', 'Pitch_60', 'Velocity_91', 'Duration_0.1.8', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.1.8', 'Position_7', 'Program_0', 'Pitch_55', 'Velocity_87', 'Duration_0.1.8', 'Program_0', 'Pitch_71', 'Velocity_87', 'Duration_0.1.8', 'Program_0', 'Pitch_65', 'Velocity_79', 'Duration_0.1.8', 'Position_10', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_87', 'Duration_0.1.8', 'Program_0', 'Pitch_48', 'Velocity_59', 'Duration_0.1.8', 'Position_12', 'Program_0', 'Pitch_43', 'Velocity_67', 'Duration_0.1.8', 'Position_16', 'Program_0', 'Pitch_79', 'Velocity_99', 'Duration_0.5.8', 'Program_0', 'Pitch_36', 'Velocity_87', 'Duration_1.5.8', 'Program_0', 'Pitch_43', 'Velocity_87', 'Duration_1.5.8', 'Position_17', 'Program_0', 'Pitch_67', 'Velocity_87', 'Duration_0.6.8', 'Position_19', 'Program_0', 'Pitch_72', 'Velocity_79', 'Duration_1.2.8', 'Position_21', 'Program_0', 'Pitch_79', 'Velocity_87', 'Duration_1.0.8', 'Position_22', 'Program_0', 'Pitch_58', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_61', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_76', 'Velocity_91', 'Duration_0.5.8', 'Position_23', 'Program_0', 'Pitch_67', 'Velocity_79', 'Duration_0.5.8', 'Position_25', 'Program_0', 'Pitch_73', 'Velocity_75', 'Duration_0.4.8', 'Position_27', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_0.2.8', 'Position_28', 'Program_0', 'Pitch_57', 'Velocity_87', 'Duration_1.4.8', 'Program_0', 'Pitch_60', 'Velocity_87', 'Duration_1.4.8', 'Program_0', 'Pitch_77', 'Velocity_91', 'Duration_0.5.8', 'Position_30', 'Program_0', 'Pitch_65', 'Velocity_75', 'Duration_0.6.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_72', 'Velocity_75', 'Duration_1.1.8', 'Position_1', 'Program_0', 'Pitch_77', 'Velocity_79', 'Duration_1.0.8', 'Position_2', 'Program_0', 'Pitch_59', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_56', 'Velocity_75', 'Duration_0.7.8', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.5.8', 'Position_4', 'Program_0', 'Pitch_65', 'Velocity_79', 'Duration_0.5.8', 'Position_5', 'Program_0', 'Pitch_71', 'Velocity_79', 'Duration_0.4.8', 'Position_7', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.2.8', 'Position_8', 'Program_0', 'Pitch_58', 'Velocity_95', 'Duration_0.7.8', 'Program_0', 'Pitch_55', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_76', 'Velocity_91', 'Duration_0.5.8', 'Position_10', 'Program_0', 'Pitch_64', 'Velocity_79', 'Duration_0.5.8', 'Position_11', 'Program_0', 'Pitch_70', 'Velocity_79', 'Duration_0.3.8', 'Position_13', 'Program_0', 'Pitch_76', 'Velocity_87', 'Duration_0.2.8', 'Position_14', 'Program_0', 'Pitch_54', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_72', 'Velocity_95', 'Duration_0.5.8', 'Program_0', 'Pitch_57', 'Velocity_95', 'Duration_0.6.8', 'Position_16', 'Program_0', 'Pitch_63', 'Velocity_79', 'Duration_0.5.8', 'Position_18', 'Program_0', 'Pitch_69', 'Velocity_75', 'Duration_0.3.8', 'Position_19', 'Program_0', 'Pitch_72', 'Velocity_79', 'Duration_0.2.8', 'Position_20', 'Program_0', 'Pitch_56', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.4.8', 'Position_21', 'Program_0', 'Pitch_53', 'Velocity_71', 'Duration_0.6.8', 'Program_0', 'Pitch_62', 'Velocity_87', 'Duration_0.5.8', 'Position_23', 'Program_0', 'Pitch_68', 'Velocity_79', 'Duration_0.4.8', 'Position_25', 'Program_0', 'Pitch_74', 'Velocity_79', 'Duration_0.2.8', 'Position_26', 'Program_0', 'Pitch_55', 'Velocity_71', 'Duration_0.7.8', 'Program_0', 'Pitch_67', 'Velocity_91', 'Duration_0.4.8', 'Position_27', 'Program_0', 'Pitch_59', 'Velocity_87', 'Duration_0.6.8', 'Position_28', 'Program_0', 'Pitch_62', 'Velocity_91', 'Duration_0.5.8', 'Position_29', 'Program_0', 'Pitch_65', 'Velocity_87', 'Duration_0.4.8', 'Position_30', 'Program_0', 'Pitch_67', 'Velocity_87', 'Duration_0.3.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_43', 'Velocity_99', 'Duration_0.1.8', 'Program_0', 'Pitch_36', 'Velocity_91', 'Duration_0.1.8', 'Position_1', 'Program_0', 'Pitch_79', 'Velocity_99', 'Duration_0.5.8', 'Position_2', 'Program_0', 'Pitch_67', 'Velocity_87', 'Duration_0.5.8', 'Position_4', 'Program_0', 'Pitch_72', 'Velocity_87', 'Duration_0.4.8', 'Position_5', 'Program_0', 'Pitch_79', 'Velocity_87', 'Duration_1.0.8', 'Position_7', 'Program_0', 'Pitch_58', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_76', 'Velocity_95', 'Duration_0.5.8', 'Program_0', 'Pitch_61', 'Velocity_95', 'Duration_0.7.8', 'Position_8', 'Program_0', 'Pitch_67', 'Velocity_75', 'Duration_0.5.8', 'Position_10', 'Program_0', 'Pitch_73', 'Velocity_79', 'Duration_0.4.8', 'Position_12', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_0.2.8', 'Position_13', 'Program_0', 'Pitch_60', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_57', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_77', 'Velocity_95', 'Duration_0.4.8', 'Position_14', 'Program_0', 'Pitch_65', 'Velocity_75', 'Duration_0.5.8', 'Position_16', 'Program_0', 'Pitch_72', 'Velocity_75', 'Duration_0.4.8', 'Position_17', 'Program_0', 'Pitch_77', 'Velocity_87', 'Duration_0.2.8', 'Position_19', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.5.8', 'Program_0', 'Pitch_59', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_56', 'Velocity_75', 'Duration_0.6.8', 'Position_20', 'Program_0', 'Pitch_65', 'Velocity_75', 'Duration_0.5.8', 'Position_21', 'Program_0', 'Pitch_71', 'Velocity_79', 'Duration_0.4.8', 'Position_24', 'Program_0', 'Pitch_74', 'Velocity_79', 'Duration_0.2.8', 'Position_25', 'Program_0', 'Pitch_55', 'Velocity_75', 'Duration_0.6.8', 'Program_0', 'Pitch_60', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_76', 'Velocity_87', 'Duration_0.6.8', 'Position_26', 'Program_0', 'Pitch_64', 'Velocity_75', 'Duration_0.5.8', 'Position_27', 'Program_0', 'Pitch_67', 'Velocity_87', 'Duration_0.4.8', 'Position_29', 'Program_0', 'Pitch_72', 'Velocity_87', 'Duration_0.2.8', 'Position_30', 'Program_0', 'Pitch_71', 'Velocity_87', 'Duration_0.6.8', 'Position_31', 'Program_0', 'Pitch_55', 'Velocity_75', 'Duration_0.6.8', 'Program_0', 'Pitch_62', 'Velocity_91', 'Duration_0.6.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_65', 'Velocity_79', 'Duration_0.5.8', 'Position_1', 'Program_0', 'Pitch_67', 'Velocity_87', 'Duration_0.4.8', 'Position_3', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.2.8', 'Position_4', 'Program_0', 'Pitch_72', 'Velocity_87', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_75', 'Duration_0.1.8', 'Position_5', 'Program_0', 'Pitch_64', 'Velocity_71', 'Duration_0.1.8', 'Position_7', 'Program_0', 'Pitch_55', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_65', 'Velocity_79', 'Duration_0.1.8', 'Position_8', 'Program_0', 'Pitch_71', 'Velocity_71', 'Duration_0.2.8', 'Position_10', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.2.8', 'Program_0', 'Pitch_64', 'Velocity_91', 'Duration_0.1.8', 'Program_0', 'Pitch_48', 'Velocity_79', 'Duration_0.1.8', 'Position_13', 'Program_0', 'Pitch_43', 'Velocity_67', 'Duration_0.1.8', 'Position_16', 'Program_0', 'Pitch_64', 'Velocity_99', 'Duration_0.7.8', 'Program_0', 'Pitch_76', 'Velocity_95', 'Duration_0.5.8', 'Program_0', 'Pitch_67', 'Velocity_99', 'Duration_0.7.8', 'Program_0', 'Pitch_36', 'Velocity_79', 'Duration_0.7.8', 'Program_0', 'Pitch_43', 'Velocity_79', 'Duration_0.7.8', 'Position_21', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_1.0.8', 'Position_23', 'Program_0', 'Pitch_64', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_67', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_60', 'Velocity_71', 'Duration_0.6.8', 'Program_0', 'Pitch_58', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_72', 'Velocity_87', 'Duration_0.4.8', 'Position_27', 'Program_0', 'Pitch_72', 'Velocity_87', 'Duration_0.2.8', 'Position_29', 'Program_0', 'Pitch_60', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_57', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_77', 'Velocity_95', 'Duration_0.5.8', 'Program_0', 'Pitch_65', 'Velocity_95', 'Duration_0.6.8', 'Bar_None', 'Position_1', 'Program_0', 'Pitch_77', 'Velocity_79', 'Duration_0.2.8', 'Position_3', 'Program_0', 'Pitch_71', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_56', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_59', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.5.8', 'Program_0', 'Pitch_65', 'Velocity_91', 'Duration_0.6.8', 'Position_8', 'Program_0', 'Pitch_74', 'Velocity_79', 'Duration_0.2.8', 'Position_9', 'Program_0', 'Pitch_36', 'Velocity_91', 'Duration_0.1.8', 'Program_0', 'Pitch_43', 'Velocity_95', 'Duration_0.1.8', 'Program_0', 'Pitch_76', 'Velocity_95', 'Duration_0.5.8', 'Program_0', 'Pitch_64', 'Velocity_99', 'Duration_0.7.8', 'Program_0', 'Pitch_67', 'Velocity_95', 'Duration_0.6.8', 'Position_13', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_0.2.8', 'Position_15', 'Program_0', 'Pitch_67', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_72', 'Velocity_95', 'Duration_0.4.8', 'Program_0', 'Pitch_63', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_57', 'Velocity_95', 'Duration_0.6.8', 'Position_19', 'Program_0', 'Pitch_72', 'Velocity_79', 'Duration_0.2.8', 'Position_21', 'Program_0', 'Pitch_62', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_66', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_58', 'Velocity_95', 'Duration_0.7.8', 'Program_0', 'Pitch_55', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.5.8', 'Position_26', 'Program_0', 'Pitch_74', 'Velocity_79', 'Duration_0.2.8', 'Position_27', 'Program_0', 'Pitch_67', 'Velocity_91', 'Duration_0.4.8', 'Program_0', 'Pitch_65', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_55', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_59', 'Velocity_91', 'Duration_0.1.8', 'Program_0', 'Pitch_62', 'Velocity_91', 'Duration_0.6.8', 'Position_31', 'Program_0', 'Pitch_67', 'Velocity_91', 'Duration_0.1.8', 'Bar_None', 'Position_1', 'Program_0', 'Pitch_76', 'Velocity_95', 'Duration_0.4.8', 'Program_0', 'Pitch_67', 'Velocity_95', 'Duration_0.6.8', 'Program_0', 'Pitch_43', 'Velocity_95', 'Duration_0.7.8', 'Program_0', 'Pitch_64', 'Velocity_99', 'Duration_0.6.8', 'Program_0', 'Pitch_36', 'Velocity_91', 'Duration_0.6.8', 'Position_6', 'Program_0', 'Pitch_76', 'Velocity_87', 'Duration_1.0.8', 'Position_7', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.4.8', 'Program_0', 'Pitch_67', 'Velocity_95', 'Duration_0.6.8', 'Position_8', 'Program_0', 'Pitch_60', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_58', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_64', 'Velocity_91', 'Duration_0.6.8', 'Position_11', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_1.0.8', 'Position_13', 'Program_0', 'Pitch_60', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_57', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_77', 'Velocity_95', 'Duration_0.4.8', 'Program_0', 'Pitch_65', 'Velocity_95', 'Duration_0.6.8', 'Position_18', 'Program_0', 'Pitch_77', 'Velocity_87', 'Duration_0.2.8', 'Position_19', 'Program_0', 'Pitch_71', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_59', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_56', 'Velocity_75', 'Duration_0.6.8', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.4.8', 'Program_0', 'Pitch_65', 'Velocity_91', 'Duration_0.6.8', 'Position_24', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.2.8', 'Position_25', 'Program_0', 'Pitch_55', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_87', 'Duration_0.1.8', 'Program_0', 'Pitch_76', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_67', 'Velocity_95', 'Duration_0.6.8', 'Program_0', 'Pitch_64', 'Velocity_91', 'Duration_0.6.8', 'Position_29', 'Program_0', 'Pitch_72', 'Velocity_79', 'Duration_0.7.8', 'Position_31', 'Program_0', 'Pitch_43', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_36', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_71', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_67', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_65', 'Velocity_95', 'Duration_0.6.8', 'Bar_None', 'Position_3', 'Program_0', 'Pitch_74', 'Velocity_79', 'Duration_0.2.8', 'Position_5', 'Program_0', 'Pitch_64', 'Velocity_95', 'Duration_0.1.8', 'Program_0', 'Pitch_72', 'Velocity_95', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_91', 'Duration_0.1.8', 'Position_8', 'Program_0', 'Pitch_55', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_71', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_65', 'Velocity_87', 'Duration_0.1.8', 'Position_11', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.1.8', 'Program_0', 'Pitch_48', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_87', 'Duration_0.1.8', 'Position_14', 'Program_0', 'Pitch_43', 'Velocity_67', 'Duration_0.1.8', 'Position_17', 'Program_0', 'Pitch_76', 'Velocity_95', 'Duration_0.5.8', 'Program_0', 'Pitch_43', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_36', 'Velocity_87', 'Duration_0.7.8', 'Position_18', 'Program_0', 'Pitch_64', 'Velocity_79', 'Duration_0.5.8', 'Position_20', 'Program_0', 'Pitch_67', 'Velocity_75', 'Duration_0.4.8', 'Position_22', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_1.0.8', 'Position_24', 'Program_0', 'Pitch_58', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_72', 'Velocity_95', 'Duration_0.4.8', 'Program_0', 'Pitch_60', 'Velocity_75', 'Duration_0.6.8', 'Position_25', 'Program_0', 'Pitch_64', 'Velocity_79', 'Duration_0.5.8', 'Position_26', 'Program_0', 'Pitch_67', 'Velocity_79', 'Duration_0.4.8', 'Position_28', 'Program_0', 'Pitch_72', 'Velocity_87', 'Duration_0.2.8', 'Position_29', 'Program_0', 'Pitch_57', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_77', 'Velocity_95', 'Duration_0.5.8', 'Position_30', 'Program_0', 'Pitch_60', 'Velocity_87', 'Duration_0.6.8', 'Position_31', 'Program_0', 'Pitch_65', 'Velocity_79', 'Duration_0.4.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_72', 'Velocity_75', 'Duration_0.3.8', 'Position_2', 'Program_0', 'Pitch_77', 'Velocity_87', 'Duration_1.0.8', 'Position_4', 'Program_0', 'Pitch_56', 'Velocity_75', 'Duration_0.6.8', 'Program_0', 'Pitch_59', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.4.8', 'Position_5', 'Program_0', 'Pitch_65', 'Velocity_87', 'Duration_0.5.8', 'Position_6', 'Program_0', 'Pitch_71', 'Velocity_79', 'Duration_0.4.8', 'Position_8', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.2.8', 'Position_10', 'Program_0', 'Pitch_76', 'Velocity_91', 'Duration_0.5.8', 'Program_0', 'Pitch_43', 'Velocity_95', 'Duration_0.7.8', 'Program_0', 'Pitch_36', 'Velocity_91', 'Duration_0.7.8', 'Position_11', 'Program_0', 'Pitch_64', 'Velocity_79', 'Duration_0.5.8', 'Position_13', 'Program_0', 'Pitch_67', 'Velocity_71', 'Duration_0.3.8', 'Position_15', 'Program_0', 'Pitch_76', 'Velocity_75', 'Duration_0.2.8', 'Position_16', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.4.8', 'Program_0', 'Pitch_57', 'Velocity_95', 'Duration_0.6.8', 'Position_17', 'Program_0', 'Pitch_63', 'Velocity_75', 'Duration_0.5.8', 'Position_19', 'Program_0', 'Pitch_67', 'Velocity_79', 'Duration_0.4.8', 'Position_20', 'Program_0', 'Pitch_72', 'Velocity_79', 'Duration_0.2.8', 'Position_22', 'Program_0', 'Pitch_74', 'Velocity_91', 'Duration_0.5.8', 'Program_0', 'Pitch_58', 'Velocity_91', 'Duration_1.4.8', 'Program_0', 'Pitch_55', 'Velocity_79', 'Duration_0.6.8', 'Position_23', 'Program_0', 'Pitch_62', 'Velocity_79', 'Duration_0.6.8', 'Position_24', 'Program_0', 'Pitch_66', 'Velocity_87', 'Duration_1.2.8', 'Position_26', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_1.0.8', 'Position_28', 'Program_0', 'Pitch_67', 'Velocity_91', 'Duration_0.4.8', 'Program_0', 'Pitch_55', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_59', 'Velocity_91', 'Duration_0.6.8', 'Position_29', 'Program_0', 'Pitch_62', 'Velocity_87', 'Duration_0.5.8', 'Position_31', 'Program_0', 'Pitch_65', 'Velocity_87', 'Duration_0.3.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_67', 'Velocity_79', 'Duration_0.2.8', 'Position_2', 'Program_0', 'Pitch_36', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_43', 'Velocity_99', 'Duration_0.7.8', 'Program_0', 'Pitch_76', 'Velocity_95', 'Duration_0.5.8', 'Position_3', 'Program_0', 'Pitch_64', 'Velocity_79', 'Duration_0.5.8', 'Position_5', 'Program_0', 'Pitch_67', 'Velocity_79', 'Duration_0.4.8', 'Position_6', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_1.0.8', 'Position_8', 'Program_0', 'Pitch_60', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.4.8', 'Program_0', 'Pitch_58', 'Velocity_91', 'Duration_0.6.8', 'Position_9', 'Program_0', 'Pitch_64', 'Velocity_79', 'Duration_0.5.8', 'Position_10', 'Program_0', 'Pitch_67', 'Velocity_87', 'Duration_0.4.8', 'Position_12', 'Program_0', 'Pitch_72', 'Velocity_87', 'Duration_0.2.8', 'Position_13', 'Program_0', 'Pitch_77', 'Velocity_95', 'Duration_0.5.8', 'Position_14', 'Program_0', 'Pitch_57', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_60', 'Velocity_91', 'Duration_0.6.8', 'Position_15', 'Program_0', 'Pitch_65', 'Velocity_87', 'Duration_0.5.8', 'Position_16', 'Program_0', 'Pitch_72', 'Velocity_87', 'Duration_0.3.8', 'Position_18', 'Program_0', 'Pitch_77', 'Velocity_87', 'Duration_1.5.8', 'Position_20', 'Program_0', 'Pitch_74', 'Velocity_87', 'Duration_0.4.8', 'Program_0', 'Pitch_56', 'Velocity_79', 'Duration_1.4.8', 'Program_0', 'Pitch_59', 'Velocity_87', 'Duration_1.4.8', 'Position_21', 'Program_0', 'Pitch_65', 'Velocity_79', 'Duration_1.3.8', 'Position_22', 'Program_0', 'Pitch_71', 'Velocity_79', 'Duration_1.1.8', 'Position_24', 'Program_0', 'Pitch_74', 'Velocity_91', 'Duration_1.0.8', 'Position_26', 'Program_0', 'Pitch_55', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_76', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_60', 'Velocity_87', 'Duration_0.6.8', 'Position_27', 'Program_0', 'Pitch_64', 'Velocity_87', 'Duration_0.5.8', 'Position_28', 'Program_0', 'Pitch_67', 'Velocity_87', 'Duration_0.3.8', 'Position_30', 'Program_0', 'Pitch_72', 'Velocity_87', 'Duration_0.2.8', 'Position_31', 'Program_0', 'Pitch_71', 'Velocity_87', 'Duration_0.7.8', 'Program_0', 'Pitch_43', 'Velocity_91', 'Duration_0.7.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_36', 'Velocity_79', 'Duration_0.6.8', 'Position_1', 'Program_0', 'Pitch_65', 'Velocity_87', 'Duration_0.5.8', 'Position_2', 'Program_0', 'Pitch_67', 'Velocity_79', 'Duration_0.4.8', 'Position_4', 'Program_0', 'Pitch_74', 'Velocity_75', 'Duration_0.2.8', 'Position_5', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.2.8', 'Program_0', 'Pitch_36', 'Velocity_87', 'Duration_0.1.8', 'Position_6', 'Program_0', 'Pitch_64', 'Velocity_71', 'Duration_0.1.8', 'Position_8', 'Program_0', 'Pitch_65', 'Velocity_87', 'Duration_0.1.8', 'Program_0', 'Pitch_43', 'Velocity_75', 'Duration_0.1.8', 'Position_10', 'Program_0', 'Pitch_71', 'Velocity_75', 'Duration_0.2.8', 'Position_11', 'Program_0', 'Pitch_72', 'Velocity_87', 'Duration_0.2.8', 'Program_0', 'Pitch_64', 'Velocity_91', 'Duration_0.1.8', 'Program_0', 'Pitch_48', 'Velocity_79', 'Duration_0.1.8', 'Position_14', 'Program_0', 'Pitch_55', 'Velocity_79', 'Duration_0.1.8', 'Position_17', 'Program_0', 'Pitch_79', 'Velocity_91', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_79', 'Duration_0.1.8', 'Position_19', 'Program_0', 'Pitch_81', 'Velocity_75', 'Duration_0.2.8', 'Position_20', 'Program_0', 'Pitch_67', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_72', 'Velocity_75', 'Duration_0.1.8', 'Position_22', 'Program_0', 'Pitch_77', 'Velocity_79', 'Duration_0.1.8', 'Position_23', 'Program_0', 'Pitch_76', 'Velocity_75', 'Duration_0.1.8', 'Position_24', 'Program_0', 'Pitch_67', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_73', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_71', 'Duration_0.1.8', 'Position_25', 'Program_0', 'Pitch_77', 'Velocity_71', 'Duration_0.1.8', 'Position_26', 'Program_0', 'Pitch_79', 'Velocity_79', 'Duration_0.1.8', 'Position_27', 'Program_0', 'Pitch_58', 'Velocity_75', 'Duration_0.1.8', 'Position_28', 'Program_0', 'Pitch_76', 'Velocity_71', 'Duration_0.2.8', 'Position_29', 'Program_0', 'Pitch_77', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_57', 'Velocity_87', 'Duration_0.1.8', 'Position_30', 'Program_0', 'Pitch_79', 'Velocity_75', 'Duration_0.2.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_69', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_59', 'Duration_0.1.8', 'Program_0', 'Pitch_77', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_65', 'Velocity_51', 'Duration_0.1.8', 'Position_1', 'Program_0', 'Pitch_76', 'Velocity_75', 'Duration_0.1.8', 'Position_3', 'Program_0', 'Pitch_74', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_71', 'Velocity_87', 'Duration_0.1.8', 'Program_0', 'Pitch_65', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_62', 'Velocity_67', 'Duration_0.1.8', 'Position_4', 'Program_0', 'Pitch_76', 'Velocity_75', 'Duration_0.1.8', 'Position_6', 'Program_0', 'Pitch_77', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_56', 'Velocity_67', 'Duration_0.1.8', 'Position_8', 'Program_0', 'Pitch_74', 'Velocity_59', 'Duration_0.1.8', 'Position_9', 'Program_0', 'Pitch_76', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_55', 'Velocity_79', 'Duration_0.1.8', 'Position_10', 'Program_0', 'Pitch_77', 'Velocity_71', 'Duration_0.2.8', 'Position_12', 'Program_0', 'Pitch_76', 'Velocity_75', 'Duration_0.2.8', 'Program_0', 'Pitch_67', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_51', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_51', 'Duration_0.1.8', 'Position_13', 'Program_0', 'Pitch_74', 'Velocity_71', 'Duration_0.1.8', 'Position_15', 'Program_0', 'Pitch_72', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_55', 'Duration_0.1.8', 'Program_0', 'Pitch_69', 'Velocity_75', 'Duration_0.1.8', 'Position_16', 'Program_0', 'Pitch_74', 'Velocity_71', 'Duration_0.1.8', 'Position_17', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_0.1.8', 'Position_18', 'Program_0', 'Pitch_54', 'Velocity_75', 'Duration_0.1.8', 'Position_19', 'Program_0', 'Pitch_72', 'Velocity_71', 'Duration_0.1.8', 'Position_20', 'Program_0', 'Pitch_74', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_53', 'Velocity_79', 'Duration_0.1.8', 'Position_21', 'Program_0', 'Pitch_76', 'Velocity_67', 'Duration_0.2.8', 'Position_23', 'Program_0', 'Pitch_74', 'Velocity_55', 'Duration_0.1.8', 'Program_0', 'Pitch_59', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_62', 'Velocity_75', 'Duration_0.1.8', 'Position_25', 'Program_0', 'Pitch_71', 'Velocity_67', 'Duration_0.1.8', 'Position_26', 'Program_0', 'Pitch_67', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_52', 'Velocity_79', 'Duration_0.1.8', 'Position_28', 'Program_0', 'Pitch_69', 'Velocity_67', 'Duration_0.1.8', 'Position_29', 'Program_0', 'Pitch_71', 'Velocity_59', 'Duration_0.1.8', 'Program_0', 'Pitch_50', 'Velocity_71', 'Duration_0.1.8', 'Position_31', 'Program_0', 'Pitch_74', 'Velocity_67', 'Duration_0.1.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_48', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_79', 'Velocity_87', 'Duration_0.1.8', 'Position_2', 'Program_0', 'Pitch_81', 'Velocity_79', 'Duration_0.1.8', 'Position_3', 'Program_0', 'Pitch_79', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_67', 'Velocity_67', 'Duration_0.1.8', 'Position_4', 'Program_0', 'Pitch_77', 'Velocity_75', 'Duration_0.1.8', 'Position_6', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_58', 'Velocity_71', 'Duration_0.1.8', 'Position_7', 'Program_0', 'Pitch_77', 'Velocity_71', 'Duration_0.1.8', 'Position_9', 'Program_0', 'Pitch_79', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_67', 'Velocity_59', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_67', 'Duration_0.1.8', 'Position_10', 'Program_0', 'Pitch_76', 'Velocity_67', 'Duration_0.1.8', 'Position_12', 'Program_0', 'Pitch_77', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_57', 'Velocity_79', 'Duration_0.1.8', 'Position_13', 'Program_0', 'Pitch_79', 'Velocity_67', 'Duration_0.2.8', 'Position_14', 'Program_0', 'Pitch_77', 'Velocity_71', 'Duration_0.1.8', 'Position_15', 'Program_0', 'Pitch_65', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_55', 'Duration_0.1.8', 'Position_16', 'Program_0', 'Pitch_76', 'Velocity_71', 'Duration_0.1.8', 'Position_17', 'Program_0', 'Pitch_74', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_56', 'Velocity_71', 'Duration_0.1.8', 'Position_19', 'Program_0', 'Pitch_76', 'Velocity_71', 'Duration_0.1.8', 'Position_20', 'Program_0', 'Pitch_77', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_65', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_59', 'Velocity_59', 'Duration_0.1.8', 'Position_22', 'Program_0', 'Pitch_74', 'Velocity_67', 'Duration_0.1.8', 'Position_23', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_54', 'Velocity_79', 'Duration_0.1.8', 'Position_24', 'Program_0', 'Pitch_77', 'Velocity_71', 'Duration_0.2.8', 'Position_26', 'Program_0', 'Pitch_76', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_75', 'Duration_0.1.8', 'Position_27', 'Program_0', 'Pitch_72', 'Velocity_75', 'Duration_0.1.8', 'Position_29', 'Program_0', 'Pitch_55', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_71', 'Velocity_75', 'Duration_0.1.8', 'Position_30', 'Program_0', 'Pitch_72', 'Velocity_75', 'Duration_0.1.8', 'Position_31', 'Program_0', 'Pitch_74', 'Velocity_75', 'Duration_0.1.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_62', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_65', 'Velocity_75', 'Duration_0.1.8', 'Position_1', 'Program_0', 'Pitch_71', 'Velocity_75', 'Duration_0.1.8', 'Position_2', 'Program_0', 'Pitch_72', 'Velocity_87', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_87', 'Duration_0.1.8', 'Position_3', 'Program_0', 'Pitch_60', 'Velocity_71', 'Duration_0.1.8', 'Position_4', 'Program_0', 'Pitch_74', 'Velocity_75', 'Duration_0.1.8', 'Position_5', 'Program_0', 'Pitch_72', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_55', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_62', 'Velocity_79', 'Duration_0.1.8', 'Position_6', 'Program_0', 'Pitch_71', 'Velocity_71', 'Duration_0.1.8', 'Position_8', 'Program_0', 'Pitch_72', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_48', 'Velocity_75', 'Duration_0.1.8', 'Position_11', 'Program_0', 'Pitch_55', 'Velocity_75', 'Duration_0.1.8', 'Position_14', 'Program_0', 'Pitch_76', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_75', 'Duration_0.1.8', 'Position_15', 'Program_0', 'Pitch_77', 'Velocity_71', 'Duration_0.1.8', 'Position_17', 'Program_0', 'Pitch_66', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_59', 'Duration_0.1.8', 'Position_18', 'Program_0', 'Pitch_74', 'Velocity_71', 'Duration_0.1.8', 'Position_19', 'Program_0', 'Pitch_72', 'Velocity_75', 'Duration_0.1.8', 'Position_20', 'Program_0', 'Pitch_64', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_67', 'Velocity_71', 'Duration_0.1.8', 'Position_21', 'Program_0', 'Pitch_74', 'Velocity_55', 'Duration_0.1.8', 'Position_23', 'Program_0', 'Pitch_76', 'Velocity_51', 'Duration_0.1.8', 'Program_0', 'Pitch_58', 'Velocity_55', 'Duration_0.1.8', 'Position_24', 'Program_0', 'Pitch_72', 'Velocity_59', 'Duration_0.1.8', 'Position_25', 'Program_0', 'Pitch_77', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_57', 'Velocity_87', 'Duration_0.1.8', 'Position_27', 'Program_0', 'Pitch_79', 'Velocity_71', 'Duration_0.2.8', 'Position_28', 'Program_0', 'Pitch_77', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_62', 'Velocity_75', 'Duration_0.1.8', 'Position_30', 'Program_0', 'Pitch_76', 'Velocity_71', 'Duration_0.1.8', 'Position_31', 'Program_0', 'Pitch_74', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_62', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_65', 'Velocity_71', 'Duration_0.1.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_76', 'Velocity_71', 'Duration_0.1.8', 'Position_2', 'Program_0', 'Pitch_77', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_56', 'Velocity_67', 'Duration_0.1.8', 'Position_4', 'Program_0', 'Pitch_74', 'Velocity_59', 'Duration_0.1.8', 'Position_5', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_55', 'Velocity_79', 'Duration_0.1.8', 'Position_6', 'Program_0', 'Pitch_77', 'Velocity_71', 'Duration_0.2.8', 'Position_7', 'Program_0', 'Pitch_76', 'Velocity_71', 'Duration_0.1.8', 'Position_8', 'Program_0', 'Pitch_60', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_62', 'Velocity_79', 'Duration_0.1.8', 'Position_9', 'Program_0', 'Pitch_74', 'Velocity_71', 'Duration_0.1.8', 'Position_10', 'Program_0', 'Pitch_72', 'Velocity_71', 'Duration_0.1.8', 'Position_11', 'Program_0', 'Pitch_64', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_51', 'Duration_0.1.8', 'Position_12', 'Program_0', 'Pitch_74', 'Velocity_59', 'Duration_0.1.8', 'Position_13', 'Program_0', 'Pitch_76', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_54', 'Velocity_75', 'Duration_0.1.8', 'Position_15', 'Program_0', 'Pitch_72', 'Velocity_71', 'Duration_0.1.8', 'Position_16', 'Program_0', 'Pitch_53', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_74', 'Velocity_71', 'Duration_0.1.8', 'Position_17', 'Program_0', 'Pitch_76', 'Velocity_75', 'Duration_0.2.8', 'Position_19', 'Program_0', 'Pitch_74', 'Velocity_59', 'Duration_0.1.8', 'Program_0', 'Pitch_55', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_59', 'Velocity_67', 'Duration_0.1.8', 'Position_20', 'Program_0', 'Pitch_71', 'Velocity_75', 'Duration_0.1.8', 'Position_22', 'Program_0', 'Pitch_67', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_52', 'Velocity_75', 'Duration_0.1.8', 'Position_23', 'Program_0', 'Pitch_71', 'Velocity_75', 'Duration_0.1.8', 'Position_24', 'Program_0', 'Pitch_74', 'Velocity_67', 'Duration_0.1.8', 'Position_25', 'Program_0', 'Pitch_50', 'Velocity_55', 'Duration_0.1.8', 'Program_0', 'Pitch_48', 'Velocity_39', 'Duration_0.1.8', 'Position_26', 'Program_0', 'Pitch_79', 'Velocity_71', 'Duration_0.1.8', 'Position_28', 'Program_0', 'Pitch_48', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_0.1.8', 'Position_29', 'Program_0', 'Pitch_77', 'Velocity_71', 'Duration_0.2.8', 'Position_30', 'Program_0', 'Pitch_76', 'Velocity_59', 'Duration_0.1.8', 'Position_31', 'Program_0', 'Pitch_64', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_67', 'Velocity_71', 'Duration_0.1.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_74', 'Velocity_67', 'Duration_0.1.8', 'Position_1', 'Program_0', 'Pitch_72', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_58', 'Velocity_79', 'Duration_0.1.8', 'Position_2', 'Program_0', 'Pitch_74', 'Velocity_75', 'Duration_0.1.8', 'Position_4', 'Program_0', 'Pitch_76', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_67', 'Velocity_67', 'Duration_0.1.8', 'Position_6', 'Program_0', 'Pitch_72', 'Velocity_75', 'Duration_0.1.8', 'Position_7', 'Program_0', 'Pitch_77', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_57', 'Velocity_87', 'Duration_0.1.8', 'Position_8', 'Program_0', 'Pitch_79', 'Velocity_75', 'Duration_0.2.8', 'Position_10', 'Program_0', 'Pitch_77', 'Velocity_67', 'Duration_0.2.8', 'Program_0', 'Pitch_60', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_65', 'Velocity_59', 'Duration_0.1.8', 'Position_11', 'Program_0', 'Pitch_76', 'Velocity_71', 'Duration_0.1.8', 'Position_12', 'Program_0', 'Pitch_74', 'Velocity_75', 'Duration_0.1.8', 'Position_13', 'Program_0', 'Pitch_56', 'Velocity_71', 'Duration_0.1.8', 'Position_14', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_0.1.8', 'Position_15', 'Program_0', 'Pitch_77', 'Velocity_75', 'Duration_0.1.8', 'Position_16', 'Program_0', 'Pitch_59', 'Velocity_59', 'Duration_0.1.8', 'Program_0', 'Pitch_65', 'Velocity_59', 'Duration_0.1.8', 'Position_17', 'Program_0', 'Pitch_74', 'Velocity_71', 'Duration_0.1.8', 'Position_18', 'Program_0', 'Pitch_54', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_76', 'Velocity_75', 'Duration_0.1.8', 'Position_20', 'Program_0', 'Pitch_77', 'Velocity_71', 'Duration_0.2.8', 'Position_21', 'Program_0', 'Pitch_76', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_67', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_71', 'Duration_0.1.8', 'Position_23', 'Program_0', 'Pitch_72', 'Velocity_71', 'Duration_0.1.8', 'Position_24', 'Program_0', 'Pitch_71', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_55', 'Velocity_75', 'Duration_0.1.8', 'Position_25', 'Program_0', 'Pitch_72', 'Velocity_71', 'Duration_0.1.8', 'Position_27', 'Program_0', 'Pitch_74', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_62', 'Velocity_75', 'Duration_0.1.8', 'Program_0', 'Pitch_65', 'Velocity_79', 'Duration_0.1.8', 'Position_28', 'Program_0', 'Pitch_71', 'Velocity_67', 'Duration_0.1.8', 'Position_30', 'Program_0', 'Pitch_72', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_64', 'Velocity_87', 'Duration_0.1.8', 'Program_0', 'Pitch_60', 'Velocity_71', 'Duration_0.1.8', 'Position_31', 'Program_0', 'Pitch_74', 'Velocity_67', 'Duration_0.2.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_72', 'Velocity_75', 'Duration_0.2.8', 'Position_1', 'Program_0', 'Pitch_62', 'Velocity_71', 'Duration_0.1.8', 'Program_0', 'Pitch_55', 'Velocity_71', 'Duration_0.1.8', 'Position_2', 'Program_0', 'Pitch_71', 'Velocity_75', 'Duration_0.1.8', 'Position_4', 'Program_0', 'Pitch_72', 'Velocity_79', 'Duration_0.1.8', 'Program_0', 'Pitch_48', 'Velocity_79', 'Duration_0.1.8', 'Position_5', 'Program_0', 'Pitch_76', 'Velocity_91', 'Duration_0.2.8', 'Position_7', 'Program_0', 'Pitch_79', 'Velocity_87', 'Duration_0.1.8', 'Position_8', 'Program_0', 'Pitch_84', 'Velocity_79', 'Duration_0.1.8', 'Position_10', 'Program_0', 'Pitch_44', 'Velocity_95', 'Duration_1.6.8', 'Program_0', 'Pitch_87', 'Velocity_99', 'Duration_1.6.8', 'Program_0', 'Pitch_32', 'Velocity_75', 'Duration_1.6.8', 'Program_0', 'Pitch_80', 'Velocity_95', 'Duration_1.0.8', 'Position_12', 'Program_0', 'Pitch_84', 'Velocity_79', 'Duration_0.5.8', 'Program_0', 'Pitch_75', 'Velocity_79', 'Duration_0.5.8', 'Position_13', 'Program_0', 'Pitch_63', 'Velocity_79', 'Duration_1.0.8', 'Program_0', 'Pitch_72', 'Velocity_79', 'Duration_0.5.8', 'Position_15', 'Program_0', 'Pitch_68', 'Velocity_75', 'Duration_0.5.8', 'Program_0', 'Pitch_60', 'Velocity_71', 'Duration_0.5.8', 'Position_17', 'Program_0', 'Pitch_84', 'Velocity_95', 'Duration_0.7.8', 'Program_0', 'Pitch_75', 'Velocity_99', 'Duration_0.7.8', 'Position_18', 'Program_0', 'Pitch_80', 'Velocity_87', 'Duration_0.6.8', 'Program_0', 'Pitch_72', 'Velocity_79', 'Duration_0.5.8', 'Position_20', 'Program_0', 'Pitch_68', 'Velocity_75', 'Duration_0.4.8', 'Program_0', 'Pitch_60', 'Velocity_59', 'Duration_0.3.8', 'Position_21', 'Program_0', 'Pitch_56', 'Velocity_71', 'Duration_0.3.8', 'Program_0', 'Pitch_63', 'Velocity_59', 'Duration_0.3.8', 'Position_23', 'Program_0', 'Pitch_85', 'Velocity_95', 'Duration_1.5.8', 'Program_0', 'Pitch_76', 'Velocity_91', 'Duration_0.7.8', 'Position_24', 'Program_0', 'Pitch_82', 'Velocity_87', 'Duration_0.5.8', 'Position_25', 'Program_0', 'Pitch_73', 'Velocity_87', 'Duration_0.5.8', 'Position_26', 'Program_0', 'Pitch_64', 'Velocity_79', 'Duration_0.7.8', 'Position_27', 'Program_0', 'Chord_min', 'Program_0', 'Pitch_58', 'Velocity_71', 'Duration_1.1.8', 'Program_0', 'Pitch_61', 'Velocity_67', 'Duration_0.5.8', 'Position_28', 'Program_0', 'Pitch_65', 'Velocity_71', 'Duration_1.0.8', 'Position_29', 'Program_0', 'Pitch_82', 'Velocity_95', 'Duration_0.7.8', 'Program_0', 'Pitch_73', 'Velocity_95', 'Duration_0.7.8', 'Position_30', 'Program_0', 'Pitch_70', 'Velocity_91', 'Duration_0.6.8', 'Position_31', 'Program_0', 'Pitch_76', 'Velocity_71', 'Duration_0.5.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_67', 'Velocity_79', 'Duration_0.4.8', 'Program_0', 'Pitch_61', 'Velocity_59', 'Duration_0.4.8', 'Position_1', 'Program_0', 'Pitch_64', 'Velocity_71', 'Duration_0.3.8', 'Position_4', 'Program_0', 'Pitch_75', 'Velocity_99', 'Duration_0.7.8', 'Program_0', 'Pitch_84', 'Velocity_99', 'Duration_1.5.8', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_0.5.8', 'Position_5', 'Program_0', 'Pitch_80', 'Velocity_95', 'Duration_0.5.8', 'Position_7', 'Program_0', 'Pitch_60', 'Velocity_67', 'Duration_0.7.8', 'Program_0', 'Pitch_68', 'Velocity_79', 'Duration_0.4.8', 'Program_0', 'Pitch_56', 'Velocity_67', 'Duration_0.5.8', 'Program_0', 'Pitch_63', 'Velocity_71', 'Duration_0.5.8', 'Position_10', 'Program_0', 'Pitch_80', 'Velocity_95', 'Duration_0.7.8', 'Program_0', 'Pitch_72', 'Velocity_95', 'Duration_0.7.8', 'Position_11', 'Program_0', 'Pitch_68', 'Velocity_91', 'Duration_0.6.8', 'Program_0', 'Pitch_75', 'Velocity_87', 'Duration_0.6.8', 'Position_13', 'Program_0', 'Pitch_63', 'Velocity_75', 'Duration_0.4.8', 'Program_0', 'Pitch_56', 'Velocity_71', 'Duration_0.4.8', 'Position_14', 'Program_0', 'Pitch_60', 'Velocity_67', 'Duration_0.3.8', 'Program_0', 'Pitch_51', 'Velocity_67', 'Duration_0.3.8', 'Position_16', 'Program_0', 'Pitch_73', 'Velocity_99', 'Duration_0.7.8', 'Program_0', 'Pitch_82', 'Velocity_107', 'Duration_1.4.8', 'Position_17', 'Program_0', 'Pitch_75', 'Velocity_87', 'Duration_0.5.8', 'Program_0', 'Pitch_70', 'Velocity_87', 'Duration_0.5.8', 'Position_19', 'Program_0', 'Pitch_58', 'Velocity_91', 'Duration_0.7.8', 'Position_20', 'Program_0', 'Pitch_55', 'Velocity_79', 'Duration_1.0.8', 'Program_0', 'Pitch_60', 'Velocity_75', 'Duration_1.0.8', 'Program_0', 'Pitch_65', 'Velocity_59', 'Duration_1.0.8', 'Position_22', 'Program_0', 'Pitch_75', 'Velocity_95', 'Duration_0.6.8', 'Program_0', 'Pitch_70', 'Velocity_95', 'Duration_0.6.8', 'Position_23', 'Program_0', 'Pitch_63', 'Velocity_95', 'Duration_0.3.8', 'Program_0', 'Pitch_73', 'Velocity_87', 'Duration_0.5.8', 'Position_25', 'Program_0', 'Pitch_61', 'Velocity_87', 'Duration_0.3.8', 'Program_0', 'Pitch_63', 'Velocity_75', 'Duration_0.2.8', 'Position_26', 'Program_0', 'Pitch_67', 'Velocity_39', 'Duration_0.2.8', 'Program_0', 'Pitch_58', 'Velocity_51', 'Duration_0.2.8', 'Position_28', 'Program_0', 'Pitch_87', 'Velocity_99', 'Duration_1.5.8', 'Program_0', 'Pitch_80', 'Velocity_95', 'Duration_0.7.8', 'Program_0', 'Pitch_44', 'Velocity_87', 'Duration_1.4.8', 'Program_0', 'Pitch_56', 'Velocity_91', 'Duration_1.2.8', 'Position_29', 'Program_0', 'Pitch_84', 'Velocity_79', 'Duration_0.5.8', 'Program_0', 'Pitch_75', 'Velocity_91', 'Duration_0.5.8', 'Bar_None', 'Position_0', 'Program_0', 'Pitch_63', 'Velocity_91', 'Duration_0.7.8', 'Program_0', 'Pitch_72', 'Velocity_87', 'Duration_0.4.8', 'Position_1', 'Program_0', 'Pitch_68', 'Velocity_75', 'Duration_0.5.8', 'Program_0', 'Pitch_60', 'Velocity_79', 'Duration_0.4.8', 'Position_2', 'Program_0', 'Pitch_84', 'Velocity_95', 'Duration_0.6.8', 'Program_0', 'Pitch_75', 'Velocity_99', 'Duration_0.6.8', 'Position_3', 'Program_0', 'Pitch_80', 'Velocity_87', 'Duration_0.5.8', 'Position_4', 'Program_0', 'Pitch_72', 'Velocity_79', 'Duration_0.5.8', 'Position_5', 'Program_0', 'Pitch_60', 'Velocity_75', 'Duration_0.4.8', 'Program_0', 'Pitch_68', 'Velocity_79', 'Duration_0.3.8', 'Position_6', 'Program_0', 'Pitch_56', 'Velocity_75', 'Duration_0.2.8', 'Position_7', 'Program_0', 'Pitch_63', 'Velocity_67', 'Duration_0.2.8', 'Position_8', 'Program_0', 'Pitch_85', 'Velocity_95', 'Duration_1.5.8', 'Program_0', 'Pitch_76', 'Velocity_95', 'Duration_0.7.8', 'Position_9', 'Program_0', 'Pitch_82', 'Velocity_91', 'Duration_0.5.8', 'Position_10', 'Program_0', 'Pitch_73', 'Velocity_91', 'Duration_0.5.8', 'Position_11', 'Program_0', 'Pitch_64', 'Velocity_79', 'Duration_0.7.8', 'Position_12', 'Program_0', 'Pitch_58', 'Velocity_79', 'Duration_1.2.8', 'Program_0', 'Pitch_61', 'Velocity_75', 'Duration_0.5.8', 'Position_13', 'Program_0', 'Pitch_65', 'Velocity_75', 'Duration_1.0.8', 'Position_15', 'Program_0', 'Pitch_82', 'Velocity_99', 'Duration_0.7.8', 'Program_0', 'Pitch_73', 'Velocity_95', 'Duration_0.7.8', 'Position_16', 'Program_0', 'Pitch_76', 'Velocity_79', 'Duration_0.6.8', 'Program_0', 'Pitch_70', 'Velocity_87', 'Duration_0.6.8', 'Position_17', 'Program_0', 'Pitch_67', 'Velocity_75', 'Duration_0.4.8', 'Position_18', 'Program_0', 'Pitch_61', 'Velocity_67', 'Duration_0.4.8', 'Program_0', 'Pitch_64', 'Velocity_71', 'Duration_0.3.8', 'Position_21', 'Program_0', 'Pitch_84', 'Velocity_95', 'Duration_3.1.8', 'Program_0', 'Pitch_75', 'Velocity_95', 'Duration_0.7.8', 'Position_22', 'Program_0', 'Pitch_80', 'Velocity_91', 'Duration_1.3.8', 'Program_0', 'Pitch_72', 'Velocity_91', 'Duration_2.7.8', 'Position_24', 'Program_0', 'Pitch_60', 'Velocity_67', 'Duration_2.6.8', 'Program_0', 'Pitch_68', 'Velocity_71', 'Duration_1.2.8', 'Position_25', 'Program_0', 'Pitch_56', 'Velocity_67', 'Duration_1.4.8', 'Program_0', 'Pitch_63', 'Velocity_71', 'Duration_0.5.8', 'Position_27', 'Program_0', 'Pitch_73', 'Velocity_91', 'Duration_2.2.8', 'Program_0', 'Pitch_82', 'Velocity_99', 'Duration_1.7.8', 'Position_28', 'Program_0', 'Pitch_75', 'Velocity_87', 'Duration_2.2.8', 'Program_0', 'Pitch_70', 'Velocity_87', 'Duration_1.7.8', 'Position_30', 'Program_0', 'Pitch_55', 'Velocity_47', 'Duration_2.0.8', 'Program_0', 'Pitch_63', 'Velocity_71', 'Duration_2.0.8', 'Position_31', 'Program_0', 'Pitch_58', 'Velocity_67', 'Duration_1.7.8', 'Program_0', 'Pitch_51', 'Velocity_67', 'Duration_1.4.8', 'Bar_None', 'Position_1', 'Program_0', 'Pitch_80', 'Velocity_91', 'Duration_0.6.8', 'Position_3', 'Program_0', 'Pitch_68', 'Velocity_79', 'Duration_0.6.8', 'Position_4', 'Program_0', 'Pitch_56', 'Velocity_71', 'Duration_1.1.8', 'Program_0', 'Pitch_79', 'Velocity_87', 'Duration_1.1.8', 'Position_6', 'Program_0', 'Pitch_44', 'Velocity_79', 'Duration_1.0.8', 'Program_0', 'Pitch_67', 'Velocity_79', 'Duration_1.0.8', 'Position_8', 'Program_0', 'Pitch_80', 'Velocity_91', 'Duration_0.6.8', 'Position_9', 'Program_0', 'Pitch_68', 'Velocity_79', 'Duration_0.5.8', 'Position_10', 'Program_0', 'Pitch_82', 'Velocity_87', 'Duration_0.3.8', 'Position_11', 'Program_0', 'Pitch_51', 'Velocity_71', 'Duration_0.3.8', 'Program_0', 'Pitch_70', 'Velocity_87', 'Duration_0.2.8', 'Position_12', 'Program_0', 'Pitch_39', 'Velocity_67', 'Duration_0.2.8']
